# Overview

This notebook provides the reproducible computational workflow for evaluating seven deep learning models for dermoscopic skin-lesion classification: CNN, ResNet-50, DenseNet-121, EfficientNet-B3, ConvNeXt-Tiny, MobileNetV3-Large, and Vision Transformer (ViT). Fixed archived HAM10000-trained model checkpoints are evaluated using architecture-specific preprocessing and a canonical seven-class output ordering.

For the retrospective harmonized HAM10000 comparison, all reported quantitative analyses use the same standardized 2,003-image evaluation cohort and harmonized prediction outputs. An equal-weight probability ensemble is constructed by averaging the predicted probabilities of all seven models, with each model assigned a weight of 1/7 and no validation-based optimization of ensemble weights.

On the standardized HAM10000 evaluation cohort, ViT achieved the highest point estimates for accuracy, weighted F1-score, macro ROC-AUC, and micro ROC-AUC, while the equal-weight ensemble remained competitive. External cross-dataset performance is evaluated on the harmonized ISIC 2019 dataset, where ConvNeXt-Tiny achieved the highest point estimates across the reported aggregate performance metrics. Because the ISIC 2019 challenge set is assembled from HAM10000, BCN_20000 and MSK, 10,011 of its 25,331 images (39.5%) are HAM10000 images seen during training by every checkpoint; external results are therefore also reported on the 15,320 unseen BCN_20000 and MSK images, which give the genuinely external estimate. On that remainder ConvNeXt-Tiny retains the highest accuracy and weighted F1, and the ensemble attains the highest macro and weighted ROC-AUC.

The notebook also includes calibration analysis with post-hoc temperature scaling (the ensemble is under-confident, ECE 11.54%, reduced to 1.26% by one out-of-fold temperature), computational-cost measurement, statistical comparisons, confusion matrices, ROC analyses, and Grad-CAM attribution analyses. Grad-CAM maps are generated directly from the fixed model checkpoints using their corresponding preprocessing procedures, while the quantitative analyses use the harmonized prediction outputs.

**Keywords:** Skin lesion classification, deep learning, HAM10000, ISIC 2019, external evaluation, ensemble learning, Vision Transformer, Grad-CAM, calibration.

## 2.1 Environment Setup

###  Install Required Packages (Grad-CAM + EfficientNet)

In [ ]:
# Install Grad-CAM
!pip install -q git+https://github.com/jacobgil/pytorch-grad-cam.git

# Optional: efficientnet_pytorch if needed
!pip install -q efficientnet_pytorch

### Re-import Grad-CAM Modules After Installation

In [ ]:
# Re-import Grad-CAM components after installation
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

print("Grad-CAM successfull|y re-imported.")

### Import Libraries and Define Device

In [ ]:
# Core
import os, gc, json, copy, time, glob, random, warnings
from pathlib import Path
import zipfile
import numpy as np
import pandas as pd

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter
from matplotlib.patches import Patch

# Imaging
from PIL import Image
import cv2                            # optional, for overlays if available
from skimage.transform import resize  # only if need to resize CAM masks

# Progress
from tqdm import tqdm

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

# TorchVision
import torchvision
from torchvision import transforms
from torchvision import models
from torchvision.models import (
    efficientnet_b3, EfficientNet_B3_Weights,
    resnet50, densenet121, DenseNet121_Weights, mobilenet_v3_large, MobileNet_V3_Large_Weights,
    convnext_tiny, ConvNeXt_Tiny_Weights  # already imported before; moved here to consolidate
)
from torchvision.datasets.folder import default_loader

# Hugging Face Transformers (for ViT only)
from transformers import ViTConfig, ViTForImageClassification, ViTImageProcessor

# timm (preinstalled on Kaggle) for ViT and many CNNs
import timm

# EfficientNet from efficientnet_pytorch (required for loading EfficientNet models not from timm)
from efficientnet_pytorch import EfficientNet

# SciPy / Stats
from scipy.optimize import linear_sum_assignment
from scipy.stats import binomtest, chi2

# Scikit-learn
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, roc_curve,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)
from sklearn.metrics import auc as sk_auc  # avoid conflict with built-in auc

# McNemar's test
from statsmodels.stats.contingency_tables import mcnemar

# Optional: Grad-CAM (skip gracefully if not available)
HAS_GRADCAM = True
try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam import GradCAM as PTGradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
except Exception:
    HAS_GRADCAM = False
    print("Grad-CAM not available; CAM visualizations will be skipped.")

from torchvision.transforms.functional import to_pil_image
import shutil

# Both timm and HuggingFace transformers available for model loading (e.g., ViT)

# Config / Reproducibility
warnings.filterwarnings("ignore")

def seed_all(seed: int = 123):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_all(123)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
sns.set_theme(context="notebook", style="whitegrid")
plt.rcParams["figure.dpi"] = 120

### Global Label Mapping & Class Names

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv")
label_map = {name:i for i,name in enumerate(sorted(df['dx'].unique()))}
class_names = [k for k,_ in sorted(label_map.items(), key=lambda x:x[1])]
df['label'] = df['dx'].map(label_map).astype(int)

### Globals & Paths (Once)

In [ ]:
# Canonical class list must already exist (we have it from 'Global Label Mapping & Class Names')
assert 'class_names' in globals() and isinstance(class_names, (list, tuple)) and len(class_names) > 0, \
    "Define class_names first (canonical order)."
NUM_C = len(class_names)

# Image folders and resolver (used throughout the ensemble cells)
IMG_DIRS = [
    "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1",
    "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_2",
]

def find_image_path_from_id(image_id: str):
    fname = f"{image_id}.jpg"
    for d in IMG_DIRS:
        p = os.path.join(d, fname)
        if os.path.exists(p):
            return p
    return None

### Define ROC-AUC Plotting Utility (Global)

In [ ]:
def plot_multiclass_roc(all_labels, all_probs, class_names, title, outfile):
    """
    all_labels: 1D array-like of true integer labels (0..C-1)
    all_probs:  2D array-like of predicted probabilities, shape [N, C]
    class_names: list of class names in the same index order
    """
    y_true = label_binarize(all_labels, classes=list(range(len(class_names))))
    y_prob = np.asarray(all_probs)

    plt.figure(figsize=(10, 8))
    for i, cname in enumerate(class_names):
        fpr, tpr, _ = roc_curve(y_true[:, i], y_prob[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"{cname} (AUC = {roc_auc:.2f})")

    plt.plot([0, 1], [0, 1], 'k--', label='Random')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(title)
    plt.legend(loc='lower right')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(outfile, dpi=300)
    plt.show()

    macro_auc = roc_auc_score(y_true, y_prob, average='macro')
    micro_auc = roc_auc_score(y_true, y_prob, average='micro')
    print(f"Macro AUC: {macro_auc:.4f}")
    print(f"Micro AUC: {micro_auc:.4f}")

## 2.2 Dataset Preparation

### Custom Dataset Class for HAM10000

In [ ]:
class HAM10000Dataset(Dataset):
    def __init__(self, dataframe, image_dirs, transform=None):
        self.data = dataframe
        self.image_dirs = image_dirs
        self.transform = transform
        self.loader = default_loader

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        filename = row['image_id'] + ".jpg"

        # Search for image in both directories
        for d in self.image_dirs:
            img_path = os.path.join(d, filename)
            if os.path.exists(img_path):
                break
        else:
            raise FileNotFoundError(f"Image {filename} not found in provided directories.")

        image = self.loader(img_path).convert("RGB")
        label = torch.tensor(int(row['label']), dtype=torch.long)

        if self.transform:
            image = self.transform(image)

        return image, label

### Load and Prepare HAM10000 Test Dataset

In [ ]:
# Load and Prepare Canonical HAM10000 Test Dataset

# HAM10000 paths
META_PATH = (
    "/kaggle/input/datasets/kmader/"
    "skin-cancer-mnist-ham10000/HAM10000_metadata.csv"
)

image_dirs = [
    "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1",
    "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_2",
]

assert os.path.exists(META_PATH), f"Metadata not found: {META_PATH}"

for d in image_dirs:
    assert os.path.exists(d), f"Image directory not found: {d}"

# Reconstruct canonical 2,003-image test split if necessary
canonical_csv = "/kaggle/working/canonical_ham10000_test_2003.csv"

if "canonical_test_df" not in globals():

    if os.path.exists(canonical_csv):
        canonical_test_df = pd.read_csv(canonical_csv)
        print("Loaded canonical test split from:", canonical_csv)

    else:
        print("Canonical split not found in current session.")
        print("Reconstructing fixed stratified 20% HAM10000 test split...")

        df = pd.read_csv(META_PATH)

        # Canonical alphabetical class order
        canonical_classes = sorted(df["dx"].dropna().unique().tolist())
        label_map = {
            name: i
            for i, name in enumerate(canonical_classes)
        }

        df["label"] = df["dx"].map(label_map).astype(int)

        # Resolve image paths
        def find_image_path(image_id):
            fname = f"{image_id}.jpg"

            for d in image_dirs:
                candidate = os.path.join(d, fname)
                if os.path.exists(candidate):
                    return candidate

            return None

        df["path"] = df["image_id"].apply(find_image_path)

        df = df[
            df["path"].notna()
        ].reset_index(drop=True)

        # Fixed canonical stratified 20% test split
        sss = StratifiedShuffleSplit(
            n_splits=1,
            test_size=0.20,
            random_state=42
        )

        _, test_idx = next(
            sss.split(
                df["image_id"],
                df["label"]
            )
        )

        canonical_test_df = (
            df.iloc[test_idx]
            .reset_index(drop=True)
        )

        # Save for reuse during this Kaggle session
        canonical_test_df.to_csv(
            canonical_csv,
            index=False
        )

        print("Canonical split reconstructed and saved.")

# Safety checks
assert len(canonical_test_df) == 2003, \
    f"Expected 2003 canonical test images, found {len(canonical_test_df)}"

assert canonical_test_df["image_id"].nunique() == 2003, \
    "Duplicate image IDs detected in canonical test split."

test_df = canonical_test_df.copy()

# ResNet-50 and ViT were trained with the original metadata-order label mapping;
# probabilities are reordered deterministically to the canonical alphabetical order.

# Dataset class
if "HAM10000Dataset" not in globals():

    class HAM10000Dataset(Dataset):
        def __init__(self, dataframe, image_dirs, transform=None):
            self.df = dataframe.reset_index(drop=True)
            self.image_dirs = image_dirs
            self.transform = transform

        def __len__(self):
            return len(self.df)

        def __getitem__(self, idx):
            row = self.df.iloc[idx]

            image_id = str(row["image_id"])
            label = int(row["label"])

            image_path = None

            if (
                "path" in self.df.columns
                and isinstance(row.get("path"), str)
                and os.path.exists(row["path"])
            ):
                image_path = row["path"]

            if image_path is None:
                for d in self.image_dirs:
                    candidate = os.path.join(
                        d,
                        f"{image_id}.jpg"
                    )

                    if os.path.exists(candidate):
                        image_path = candidate
                        break

            if image_path is None:
                raise FileNotFoundError(
                    f"HAM10000 image not found: {image_id}.jpg"
                )

            image = Image.open(image_path).convert("RGB")

            if self.transform is not None:
                image = self.transform(image)

            return image, torch.tensor(
                label,
                dtype=torch.long
            )

# Evaluation transform
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.5] * 3,
        [0.5] * 3
    ),
])

# Canonical test dataset and loader
test_dataset = HAM10000Dataset(
    dataframe=test_df,
    image_dirs=image_dirs,
    transform=val_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

# Audit and Fix Training-Time Class Order

# Canonical revised-evaluation order
CANONICAL_CLASSES = [
    "akiec",
    "bcc",
    "bkl",
    "df",
    "mel",
    "nv",
    "vasc",
]

# Original training-time order used by ResNet-50 and ViT
RESNET_VIT_TRAIN_CLASSES = [
    "bkl",
    "nv",
    "df",
    "mel",
    "vasc",
    "bcc",
    "akiec",
]

# Deterministic probability-column mapping:
# ResNet/ViT training order -> canonical order
RESNET_VIT_TO_CANONICAL = [
    RESNET_VIT_TRAIN_CLASSES.index(c)
    for c in CANONICAL_CLASSES
]

assert RESNET_VIT_TO_CANONICAL == [6, 5, 0, 2, 3, 1, 4]

class_names = CANONICAL_CLASSES

print("Canonical classes:")
print(class_names)

print("\nResNet/ViT training classes:")
print(RESNET_VIT_TRAIN_CLASSES)

print("\nResNet/ViT probability reorder:")
print(RESNET_VIT_TO_CANONICAL)

# Final verification
print("\nCanonical HAM10000 evaluation set")
print("---------------------------------")
print("Test images:", len(test_dataset))
print("Unique image IDs:", test_df["image_id"].nunique())
print("Test batches:", len(test_loader))
print("Class distribution:")
print(test_df["label"].value_counts().sort_index())

assert len(test_dataset) == 2003
assert test_df["image_id"].nunique() == 2003

print("\nCanonical test loader prepared successfully.")

### Canonical Preprocessing

In [ ]:
# Canonical Evaluation Preprocessing

# CNN, DenseNet-121, MobileNetV3
preprocess_224_05 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.5, 0.5, 0.5],
        [0.5, 0.5, 0.5]
    )
])

# ResNet-50
preprocess_resnet = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

# EfficientNet-B3
preprocess_effnet = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.5, 0.5, 0.5],
        [0.5, 0.5, 0.5]
    )
])

# ConvNeXt-Tiny
# IMPORTANT: original ConvNeXt training used ImageNet normalization.
preprocess_convnext = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

# ViT
vit_processor = ViTImageProcessor.from_pretrained(
    "google/vit-base-patch16-224"
)

print("Canonical preprocessing pipelines defined.")

### Utility: Visualize Per Class Gradcam (PTGradCAM)

In [ ]:
def visualize_per_class_gradcam(model, dataset, class_names, target_layer):
    model.eval()
    seen_classes = set()

    # Convert layer name to actual layer
    target_module = dict(model.named_modules())[target_layer]
    cam = PTGradCAM(model=model, target_layers=[target_module])

    for i in range(len(dataset)):
        image, label = dataset[i]
        if label in seen_classes:
            continue

        img_in = image.unsqueeze(0).to(device)
        with torch.no_grad():
            pred = model(img_in).argmax(1).item()

        if pred == label:
            img_np = image.permute(1, 2, 0).cpu().numpy()
            img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)

            grayscale_cam = cam(img_in, targets=[ClassifierOutputTarget(label)])[0]
            overlay = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)

            plt.figure(figsize=(6, 3))
            plt.suptitle(f"Class: {class_names[label]}")
            plt.subplot(1, 2, 1)
            plt.imshow(img_np)
            plt.title("Original")
            plt.axis('off')

            plt.subplot(1, 2, 2)
            plt.imshow(overlay)
            plt.title("Grad-CAM")
            plt.axis('off')
            plt.show()

            seen_classes.add(label)
            if len(seen_classes) == len(class_names):
                break

    try:
        cam.activations_and_grads.release()
    except:
        pass
    del cam

## 2.3 Model Architectures

## CNN (Baseline Model)

In [ ]:
class CNNModel(nn.Module):
    def __init__(self, num_classes=7):
        super(CNNModel, self).__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 4
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        # 224x224 -> after 4 pools -> 14x14 feature map with 256 channels
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 14 * 14, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

### CNN Training Function with Early Stopping (50 Epochs)

In [ ]:
# CNN Training Function with Early Stopping
def train_model(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=50, patience=7):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    epochs_no_improve = 0

    model.to(device)

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print("-" * 30)

        # Train
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        for images, labels in tqdm(train_loader, desc="Training", leave=False):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)
            train_correct += (outputs.argmax(1) == labels).sum().item()
            train_total += labels.size(0)

        train_loss /= train_total
        train_acc = train_correct / train_total

        # Validate
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc="Validating", leave=False):
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                val_correct += (outputs.argmax(1) == labels).sum().item()
                val_total += labels.size(0)

        val_loss /= val_total
        val_acc = val_correct / val_total

        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
        print(f"Val   Loss: {val_loss:.4f}, Val   Acc: {val_acc:.4f}")

        # Early stopping + best checkpoint
        if val_acc > best_acc:
            best_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(best_model_wts, "/kaggle/working/cnn_ham10000_best_model.pth")
            print("Best CNN model saved.")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"\nEarly stopping triggered at epoch {epoch+1}")
                break

    return model, best_model_wts

### CNN Training: 50 Epochs with Early Stopping

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNNModel(num_classes=7).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=1e-4)

trained_model, best_model_wts = train_model(
    model, train_loader, val_loader, criterion, optimizer,
    device, num_epochs=50, patience=7
)

# Save best CNN weights
model.load_state_dict(best_model_wts)
torch.save(model.state_dict(), "/kaggle/working/cnn_ham10000_best_model.pth")
print("CNN model saved to /kaggle/working/cnn_ham10000_best_model.pth")

### Reload the Model

In [ ]:
cnn_model = CNNModel(num_classes=7)
cnn_model.load_state_dict(torch.load("/kaggle/input/datasets/nhsaykat/kaggleinputham10000-model-weights/cnn_ham10000_best_model.pth", map_location=device))
cnn_model = cnn_model.to(device)
cnn_model.eval()

### Evaluate on Test Set — Metrics & Confusion Matrix

In [ ]:
all_preds = []
all_labels = []
all_probs = []

cnn_model.eval()
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images, labels = images.to(device), labels.to(device)
        outputs = cnn_model(images)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(probs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

# Convert to NumPy arrays
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

# Accuracy, F1 score
acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average="weighted")

# ROC AUC (one-vs-rest)
try:
    auc_score = roc_auc_score(all_labels, all_probs, multi_class="ovr")
except:
    auc_score = "N/A (AUC needs >1 class in test set)"

# Print metrics
print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds, target_names=class_names))
print(f"Test Accuracy: {acc:.4f}")
print(f"Weighted F1 Score: {f1:.4f}")
print(f"ROC AUC Score: {auc_score}")

# Confusion Matrix Plot
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrix.png", dpi=300)
plt.show()

print("Confusion matrix saved to 'confusion_matrix.png'.")

### Plot ROC Curves for Multiclass Classification (Per Class)

In [ ]:
def plot_multiclass_roc(all_labels, all_probs, class_names, title, outfile):

    y_true = label_binarize(all_labels, classes=list(range(len(class_names))))
    y_prob = all_probs

    plt.figure(figsize=(12, 10))

    for i, cname in enumerate(class_names):
        fpr, tpr, _ = roc_curve(y_true[:, i], y_prob[:, i])
        roc_auc = sk_auc(fpr, tpr)  # <<< fix: use sk_auc instead of auc
        plt.plot(fpr, tpr, label=f"{cname} (AUC = {roc_auc:.2f})")

    plt.plot([0, 1], [0, 1], "k--", label="Random")
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(outfile, dpi=300)
    plt.show()

### ROC‑AUC Curves

In [ ]:
plot_multiclass_roc(
    all_labels=all_labels,
    all_probs=all_probs,
    class_names=class_names,
    title="ROC Curves — CNN",
    outfile="/kaggle/working/roc_curves_cnn.png"
)

### Grad-CAM Generation for Test Images — CNN (Plot + Save, with Fallback)

In [ ]:
# Sanity checks
assert 'cnn_model' in globals(), "cnn_model not found. Reload the trained CNN first."
assert 'test_loader' in globals(), "test_loader not found."
assert 'class_names' in globals(), "class_names not found."
assert 'device' in globals(), "device not set."

# Helpers
def inv_norm_05(img_t):
    """Invert Normalize([0.5]*3,[0.5]*3) -> RGB [0,1]. img_t: CHW tensor."""
    x = img_t.detach().cpu().clone()
    for c in range(3):
        x[c] = x[c] * 0.5 + 0.5
    return x.clamp(0, 1).permute(1, 2, 0).numpy()

def find_last_conv_module(net: nn.Module):
    last = None
    for m in net.modules():
        if isinstance(m, nn.Conv2d):
            last = m
    if last is None:
        raise RuntimeError("No Conv2d layer found in the CNN.")
    return last

# Setup Grad‑CAM
cnn_model.eval()
target_layer = find_last_conv_module(cnn_model)
cam = PTGradCAM(model=cnn_model, target_layers=[target_layer])

got = {c: False for c in class_names}
cam_results = {}

# Pass 1: prefer correctly predicted examples
for images, labels in test_loader:
    images, labels = images.to(device), labels.to(device)
    with torch.no_grad():
        preds = cnn_model(images).argmax(1)

    for i in range(images.size(0)):
        t_lbl = labels[i].item()
        p_lbl = preds[i].item()
        cname = class_names[t_lbl]
        if got[cname] or p_lbl != t_lbl:
            continue

        it = images[i].unsqueeze(0).requires_grad_(True)
        cnn_model.zero_grad()
        gcam = cam(input_tensor=it, targets=[ClassifierOutputTarget(p_lbl)])[0]

        rgb01 = inv_norm_05(images[i])
        cam_img = show_cam_on_image(rgb01, gcam, use_rgb=True)

        cam_results[cname] = (rgb01, cam_img)  # store original (denorm) + overlay
        got[cname] = True

    if all(got.values()):
        break

# Pass 2: fallback for missing classes (use first available, even if misclassified)
if not all(got.values()):
    missing = [c for c, v in got.items() if not v]
    print(f"[Info] No correct predictions for classes: {missing}. Falling back to first available sample.")
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        for i in range(images.size(0)):
            t_lbl = labels[i].item()
            cname = class_names[t_lbl]
            if got[cname]:
                continue

            it = images[i].unsqueeze(0).requires_grad_(True)
            cnn_model.zero_grad()
            # drive CAM toward the true label for visualization
            gcam = cam(input_tensor=it, targets=[ClassifierOutputTarget(t_lbl)])[0]

            rgb01 = inv_norm_05(images[i])
            cam_img = show_cam_on_image(rgb01, gcam, use_rgb=True)

            cam_results[cname] = (rgb01, cam_img)
            got[cname] = True

        if all(got.values()):
            break

# Plot & save
out_dir = Path("/kaggle/working/gradcam_cnn_per_class")
out_dir.mkdir(parents=True, exist_ok=True)

if not cam_results:
    print("[Warning] No Grad‑CAM images generated for CNN. Check model/loader.")
else:
    for cname, (orig_rgb01, cam_img) in cam_results.items():
        plt.figure(figsize=(8, 4))
        plt.subplot(1, 2, 1); plt.imshow(orig_rgb01); plt.title(f"Original — {cname}"); plt.axis('off')
        plt.subplot(1, 2, 2); plt.imshow(cam_img);    plt.title("Grad‑CAM");          plt.axis('off')
        plt.tight_layout()
        save_path = out_dir / f"gradcam_{cname}.png"
        plt.savefig(save_path, dpi=300, bbox_inches="tight", pad_inches=0.05)
        plt.show()
    print(f"Saved CNN Grad‑CAM images to: {out_dir.resolve()}")

# Cleanup hooks
try:
    cam.activations_and_grads.release()
except Exception:
    pass
del cam

## ResNet-50

### Custom Dataset Loader

In [ ]:
class SkinLesionDataset(Dataset):
    def __init__(self, df, img_dirs, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dirs = img_dirs
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = self.df.iloc[idx]['image_id']
        img_name = img_id + ".jpg"

        for img_dir in self.img_dirs:
            img_path = os.path.join(img_dir, img_name)
            if os.path.exists(img_path):
                image = Image.open(img_path).convert("RGB")
                break
        else:
            raise FileNotFoundError(f"{img_name} not found in {self.img_dirs}")

        # Ensure numeric label -> torch.long tensor
        label = int(self.df.iloc[idx]['label'])
        label = torch.tensor(label, dtype=torch.long)

        if self.transform:
            image = self.transform(image)

        return image, label

### Data Preparation

In [ ]:
# Metadata
df = pd.read_csv("/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv")

# Label encoding
label_dict = {label: idx for idx, label in enumerate(df['dx'].unique())}
df['label'] = df['dx'].map(label_dict)
df['image_id'] = df['image_id'].astype(str)

# Train/Val/Test split
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.1, stratify=train_df['label'], random_state=42)

# Reset index to prevent DataLoader bugs
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

### Transforms

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

### Dataset and DataLoader Setup

In [ ]:
img_dirs = [
    "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1",
    "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_2"
]

train_dataset = SkinLesionDataset(train_df, img_dirs, transform=train_transform)
val_dataset   = SkinLesionDataset(val_df, img_dirs, transform=test_transform)
test_dataset  = SkinLesionDataset(test_df, img_dirs, transform=test_transform)

### Handle Class Imbalance

In [ ]:
class_counts = train_df['label'].value_counts().sort_index().values
weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
sample_weights = weights[train_df['label'].values]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

### DataLoader Setup

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

### Define ResNet50 Architecture

In [ ]:
# Define ResNet50 model
def get_resnet50_model(num_classes=7):
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)  # load pretrained ResNet50
    for param in model.parameters():
        param.requires_grad = False  # freeze all layers

    # Replace the fully connected layer
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    return model

### Prepare Model, Loss, Optimizer

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

resnet_model = get_resnet50_model(num_classes=7)
resnet_model = resnet_model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet_model.fc.parameters(), lr=1e-4)

### Define Training Function (with Early Stopping)

In [ ]:
def train_model_resnet50(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=50, patience=5):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    epochs_no_improve = 0

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print("-" * 50)

        # Training
        model.train()
        running_loss, correct_train, total_train = 0.0, 0, 0
        for images, labels in tqdm(train_loader, desc="Training", leave=False):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss  += loss.item() * images.size(0)
            correct_train += (outputs.argmax(1) == labels).sum().item()
            total_train   += labels.size(0)

        train_loss = running_loss / total_train
        train_acc  = correct_train / total_train

        # Validation
        model.eval()
        val_running, correct_val, total_val = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc="Validation", leave=False):
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss    = criterion(outputs, labels)

                val_running += loss.item() * images.size(0)
                correct_val += (outputs.argmax(1) == labels).sum().item()
                total_val   += labels.size(0)

        val_loss = val_running / total_val
        val_acc  = correct_val / total_val

        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

        # Early stopping logic
        if val_acc > best_acc:
            best_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            print("Best ResNet50 model updated.")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("Early stopping triggered.")
                break

    return model, best_model_wts

### Train the Model

In [ ]:
resnet_model, best_model_wts = train_model_resnet50(
    model=resnet_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    num_epochs=50,
    patience=5
)

# Save best model weights right after training finishes
resnet_model.load_state_dict(best_model_wts)
torch.save(resnet_model.state_dict(), "/kaggle/working/resnet50_ham10000.pth")
print("ResNet50 model saved to /kaggle/working/resnet50_ham10000.pth")

### Reload the Model

In [ ]:
RESNET_PATH = "/kaggle/input/datasets/nhsaykat/resnet50-ham10000-model-weights/resnet50_ham10000.pth"

resnet_model = get_resnet50_model(num_classes=7)

resnet_model.load_state_dict(
    torch.load(
        RESNET_PATH,
        map_location=device
    )
)

resnet_model = resnet_model.to(device)
resnet_model.eval()

print("ResNet-50 loaded successfully from:", RESNET_PATH)

### Evaluate on Test Set — Metrics & Confusion Matrix

In [ ]:
resnet_model.eval()

all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images, labels = images.to(device), labels.to(device)
        outputs = resnet_model(images)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(probs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

# Convert to NumPy arrays
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

# Metrics
acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average="weighted")

# ROC AUC (one-vs-rest)
try:
    auc_score = roc_auc_score(all_labels, all_probs, multi_class="ovr")
except ValueError:
    auc_score = "N/A (AUC needs >1 class in test set)"

# Report
print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds, target_names=class_names))
print(f"Test Accuracy: {acc:.4f}")
print(f"Weighted F1 Score: {f1:.4f}")
print(f"ROC AUC Score: {auc_score}")

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix — ResNet-50")
plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrix_resnet50.png", dpi=300)
plt.show()

print("Confusion matrix saved to 'confusion_matrix_resnet50.png'.")

### ROC‑AUC Curves

In [ ]:
plot_multiclass_roc(
    all_labels=all_labels,
    all_probs=all_probs,
    class_names=class_names,
    title="ROC Curves — ResNet-50",
    outfile="/kaggle/working/roc_curves_resnet50.png"
)

### Grad‑CAM Generation for Test Images — ResNet‑50 (Plot + Save, with Fallback)

In [ ]:
# Sanity checks
assert 'resnet_model' in globals(), "resnet_model not found. Reload/train it first."
assert 'test_loader'   in globals(), "test_loader not found."
assert 'class_names'   in globals(), "class_names not found."
assert 'device'        in globals(), "device not set (cuda/cpu)."

# Helpers (invert ImageNet normalization for correct colors)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def inv_imagenet_norm(img_t):
    """Invert Normalize(IMAGENET_MEAN, IMAGENET_STD) -> RGB [0,1]. img_t: CHW tensor."""
    x = img_t.detach().cpu().clone()
    for c, m, s in zip(range(3), IMAGENET_MEAN, IMAGENET_STD):
        x[c] = x[c] * s + m
    return x.clamp(0, 1).permute(1, 2, 0).numpy()

def find_last_conv_module(net: nn.Module):
    last = None
    for m in net.modules():
        if isinstance(m, nn.Conv2d):
            last = m
    if last is None:
        raise RuntimeError("No Conv2d layer found in the model.")
    return last

# Setup Grad‑CAM target layer (best for ResNet‑50: last bottleneck conv)
resnet_model.eval()
target_layer = getattr(resnet_model.layer4[-1], "conv3", None)
if target_layer is None:  # fallback in case of different variant
    target_layer = find_last_conv_module(resnet_model)

cam = PTGradCAM(model=resnet_model, target_layers=[target_layer])

# Collect one image per class (prefer correctly predicted), else fallback
got = {c: False for c in class_names}
cam_results = {}

# Pass 1 — prefer correct examples
for images, labels in test_loader:
    images, labels = images.to(device), labels.to(device)
    with torch.no_grad():
        preds = resnet_model(images).argmax(1)

    for i in range(images.size(0)):
        t_lbl = labels[i].item()
        p_lbl = preds[i].item()
        cname = class_names[t_lbl]
        if got[cname] or p_lbl != t_lbl:
            continue

        it = images[i].unsqueeze(0).requires_grad_(True)
        resnet_model.zero_grad()
        gcam = cam(input_tensor=it, targets=[ClassifierOutputTarget(p_lbl)])[0]

        rgb01 = inv_imagenet_norm(images[i])
        cam_img = show_cam_on_image(rgb01, gcam, use_rgb=True)

        cam_results[cname] = (rgb01, cam_img)
        got[cname] = True
    if all(got.values()):
        break

# Pass 2 — fallback: take first available even if misclassified
if not all(got.values()):
    missing = [c for c, v in got.items() if not v]
    print(f"[Info] No correct predictions for classes: {missing}. Falling back to first available sample.")
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        for i in range(images.size(0)):
            t_lbl = labels[i].item()
            cname = class_names[t_lbl]
            if got[cname]:
                continue

            it = images[i].unsqueeze(0).requires_grad_(True)
            resnet_model.zero_grad()
            # drive CAM toward the TRUE label for visualization
            gcam = cam(input_tensor=it, targets=[ClassifierOutputTarget(t_lbl)])[0]

            rgb01 = inv_imagenet_norm(images[i])
            cam_img = show_cam_on_image(rgb01, gcam, use_rgb=True)

            cam_results[cname] = (rgb01, cam_img)
            got[cname] = True
        if all(got.values()):
            break

# Plot & save
out_dir = Path("/kaggle/working/gradcam_resnet50_per_class")
out_dir.mkdir(parents=True, exist_ok=True)

if not cam_results:
    print("[Warning] No Grad‑CAM images generated for ResNet‑50. Check model/loader.")
else:
    for cname, (orig_rgb01, cam_img) in cam_results.items():
        plt.figure(figsize=(8, 4))
        plt.subplot(1, 2, 1); plt.imshow(orig_rgb01); plt.title(f"Original — {cname}"); plt.axis('off')
        plt.subplot(1, 2, 2); plt.imshow(cam_img);   plt.title("Grad‑CAM");          plt.axis('off')
        plt.tight_layout()
        sp = out_dir / f"gradcam_{cname}.png"
        plt.savefig(sp, dpi=300, bbox_inches="tight", pad_inches=0.05)
        plt.show()
    print(f"Saved ResNet‑50 Grad‑CAM images to: {out_dir.resolve()}")

# Cleanup hooks
try:
    cam.activations_and_grads.release()
except Exception:
    pass
del cam

## Vision Transformer (ViT)

### Load ViT Processor

In [ ]:
# Load image processor for ViT
processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")

### Define Custom Dataset Class

In [ ]:
class HAM10000ViTDataset(Dataset):
    def __init__(self, dataframe, image_dirs, processor, transform=None):
        """
        Args:
            dataframe: Pandas DataFrame with columns ['image_id', 'label'].
            image_dirs: List of folders where HAM10000 images are stored.
            processor: HuggingFace ViTImageProcessor.
            transform: Optional torchvision transform (applied before processor).
        """
        self.df = dataframe
        self.image_dirs = image_dirs
        self.processor = processor
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_filename = row["image_id"] + ".jpg"

        # Find the image in one of the directories
        img_path = None
        for folder in self.image_dirs:
            candidate = os.path.join(folder, img_filename)
            if os.path.exists(candidate):
                img_path = candidate
                break

        if img_path is None:
            raise FileNotFoundError(f"Image {img_filename} not found in given folders.")

        # Load image
        image = Image.open(img_path).convert("RGB")

        # Optional torchvision transforms (e.g., augmentation)
        if self.transform:
            image = self.transform(image)

        # Processor: resize, normalize -> pixel_values
        encodings = self.processor(image, return_tensors="pt")

        return {
            "pixel_values": encodings["pixel_values"].squeeze(0),
            "labels": torch.tensor(row["label"], dtype=torch.long)
        }

### Create ViT-Compatible Datasets and Dataloaders

In [ ]:
# Create ViT-Compatible Canonical Test Dataset and DataLoader

image_dirs = [
    "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1",
    "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_2"
]

# Safety checks
assert "canonical_test_df" in globals(), \
    "Run the canonical 2,003-image test-split cell first."

assert len(canonical_test_df) == 2003, \
    f"Expected 2003 images, found {len(canonical_test_df)}"

assert canonical_test_df["image_id"].nunique() == 2003, \
    "Duplicate image IDs detected."

# ViT dataset using ONLY the locked canonical test set
test_dataset_vit = HAM10000ViTDataset(
    dataframe=canonical_test_df,
    image_dirs=image_dirs,
    processor=processor
)

test_loader_vit = DataLoader(
    test_dataset_vit,
    batch_size=16,
    shuffle=False
)

print("ViT canonical test dataset:", len(test_dataset_vit))
print("ViT test batches:", len(test_loader_vit))

### Define & Initialize ViT Model

In [ ]:
VIT_PATH = "/kaggle/input/datasets/nhsaykat/vit-ham10000-model-weights/vit_ham10000_best_model.pth"

vit_model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=len(class_names),
    label2id={name: i for i, name in enumerate(class_names)},
    id2label={i: name for i, name in enumerate(class_names)},
    ignore_mismatched_sizes=True
).to(device)

vit_model.load_state_dict(
    torch.load(
        VIT_PATH,
        map_location=device,
        weights_only=True
    )
)

vit_model.eval()

print("ViT model initialized and fine-tuned weights loaded successfully")
print("Checkpoint:", VIT_PATH)
print("Class order:", class_names)

### Train ViT — Early Stopping + Save Best

In [ ]:
# ViT training (dict-based DataLoader: batch["pixel_values"], batch["labels"])
criterion   = nn.CrossEntropyLoss()
optimizer   = Adam(vit_model.parameters(), lr=2e-5)
num_epochs  = 50
patience    = 5

best_val_acc       = 0.0
epochs_no_improve  = 0
best_model_wts     = deepcopy(vit_model.state_dict())  # from copy import deepcopy
vit_model.to(device)

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-" * 30)

    # Train
    vit_model.train()
    train_loss, train_correct, train_n = 0.0, 0, 0

    for batch in tqdm(train_loader, desc="Training", leave=False):
        inputs = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = vit_model(inputs).logits
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        bs = labels.size(0)
        train_loss   += loss.item() * bs
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_n      += bs

    train_loss /= max(1, train_n)
    train_acc   = train_correct / max(1, train_n)

    # Validate
    vit_model.eval()
    val_loss, val_correct, val_n = 0.0, 0, 0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validating", leave=False):
            inputs = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)

            outputs = vit_model(inputs).logits
            loss    = criterion(outputs, labels)

            bs = labels.size(0)
            val_loss   += loss.item() * bs
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_n      += bs

    val_loss /= max(1, val_n)
    val_acc   = val_correct / max(1, val_n)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

    # Early stopping on Val Acc
    if val_acc > best_val_acc:
        best_val_acc      = val_acc
        best_model_wts    = deepcopy(vit_model.state_dict())
        epochs_no_improve = 0
        print("Best ViT weights updated.")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

# Restore best and save
vit_model.load_state_dict(best_model_wts)
out_path = "/kaggle/working/vit_ham10000_best_model.pth"
torch.save(vit_model.state_dict(), out_path)
print(f"Saved: {out_path} | Best Val Acc: {best_val_acc:.4f}")

### Save Fine-tuned ViT Model & Processor in Hugging Face Format (with Reload Test)

In [ ]:
# Ensure the model has the best weights loaded
vit_model.load_state_dict(
    torch.load("/kaggle/input/datasets/nhsaykat/vit-ham10000-model-weights/vit_ham10000_best_model.pth", map_location="cpu")
)
vit_model.eval()

# Save both model and processor in Hugging Face format
save_dir = "/kaggle/working/vit_ham10000_finetuned"
vit_model.save_pretrained(save_dir)
processor.save_pretrained(save_dir)

print(f"ViT model and processor saved to {save_dir}")

# Example: reload later
vit_model = ViTForImageClassification.from_pretrained(save_dir).to(device)
processor = ViTImageProcessor.from_pretrained(save_dir)
vit_model.eval()

print("Reloaded ViT model and processor successfully.")

### Load Fine-tuned ViT and Evaluate on Test Set

In [ ]:
# Canonical ViT Evaluation — HAM10000 2,003-image test split

assert len(test_df) == 2003
assert test_df["image_id"].nunique() == 2003

assert class_names == [
    "akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"
]

assert RESNET_VIT_TO_CANONICAL == [6, 5, 0, 2, 3, 1, 4]

vit_model.eval()

vit_probs_raw = []
vit_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader_vit, desc="Evaluating ViT"):
        pixel_values = batch["pixel_values"].to(device)
        labels_batch = batch["labels"].cpu().numpy()

        logits = vit_model(pixel_values=pixel_values).logits
        probs_raw = torch.softmax(logits, dim=1).cpu().numpy()

        vit_probs_raw.append(probs_raw)
        vit_labels.append(labels_batch)

vit_probs_raw = np.concatenate(vit_probs_raw, axis=0)
vit_labels = np.concatenate(vit_labels, axis=0)

# Convert ViT training-time output order to canonical class order
vit_probs = vit_probs_raw[:, RESNET_VIT_TO_CANONICAL]

vit_preds = vit_probs.argmax(axis=1)

# Safety checks
assert vit_probs.shape == (2003, 7)
assert vit_labels.shape == (2003,)
assert np.allclose(vit_probs.sum(axis=1), 1.0, atol=1e-5)

# Metrics
vit_acc = accuracy_score(vit_labels, vit_preds)

vit_weighted_f1 = f1_score(
    vit_labels,
    vit_preds,
    average="weighted"
)

vit_macro_auc = roc_auc_score(
    vit_labels,
    vit_probs,
    multi_class="ovr",
    average="macro"
)

vit_micro_auc = roc_auc_score(
    vit_labels,
    vit_probs,
    multi_class="ovr",
    average="micro"
)

print("\nClassification Report:\n")
print(
    classification_report(
        vit_labels,
        vit_preds,
        target_names=class_names,
        digits=4
    )
)

print("\nCanonical HAM10000 ViT Results")
print("--------------------------------")
print(f"N:                    {len(vit_labels)}")
print(f"Accuracy:             {vit_acc:.4f}")
print(f"Weighted F1:          {vit_weighted_f1:.4f}")
print(f"Macro ROC-AUC (OvR):  {vit_macro_auc:.4f}")
print(f"Micro ROC-AUC (OvR):  {vit_micro_auc:.4f}")

### ViT Grad-CAM Wrapper & Reshape Transform

In [ ]:
class ViTWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.model.eval()
    def forward(self, x):
        return self.model(x).logits

def vit_reshape_transform(tensor, height=14, width=14):
    # remove class token, then B x (H*W) x C -> B x C x H x W
    tensor = tensor[:, 1:, :]
    return tensor.reshape(tensor.size(0), height, width, -1).permute(0, 3, 1, 2)

### ROC‑AUC Curves - ViT

In [ ]:
plot_multiclass_roc(
    all_labels=np.array(all_labels),
    all_probs=np.array(all_probs),
    class_names=class_names,
    title="ROC Curves — ViT",
    outfile="/kaggle/working/roc_curves_vit.png"
)

### Grad‑CAM Generation for Test Images — ViT (Plot + Save, with Fallback)

In [ ]:
# Sanity checks / required objects
assert 'vit_model'   in globals(), "vit_model not found. Train or reload ViT first."
assert 'test_df'     in globals(), "test_df not found. Please define the test set split first."
assert 'class_names' in globals(), "class_names not found. Define from label_map."
assert 'device'      in globals(), "device not set (cuda/cpu)."

# Use the same processor used during training
vit_processor = globals().get('vit_processor', None)
if vit_processor is None:
    from transformers import ViTImageProcessor
    vit_processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")

# Minimal wrappers
if 'ViTWrapper' not in globals():
    class ViTWrapper(nn.Module):
        def __init__(self, model):
            super().__init__()
            self.model = model
            self.model.eval()
        def forward(self, x):
            return self.model(x).logits

if 'vit_reshape_transform' not in globals():
    def vit_reshape_transform(tensor, height=14, width=14):
        tensor = tensor[:, 1:, :]
        return tensor.reshape(tensor.size(0), height, width, -1).permute(0, 3, 1, 2)

# Helpers
def _load_rgb01(path):
    return np.asarray(Image.open(path).convert("RGB")).astype(np.float32) / 255.0

def _row_to_path(row):
    image_id = row["image_id"]
    for d in image_dirs:
        candidate = os.path.join(d, image_id + ".jpg")
        if os.path.exists(candidate):
            return candidate
    return None  # Not found in any directory

# Prepare model + target layer
vit_model.eval()
wrapped_vit = ViTWrapper(vit_model)
try:
    target_layers = [vit_model.vit.encoder.layer[-1].output]
except:
    target_layers = [list(vit_model.vit.encoder.layer)[-1].output]

got = {c: False for c in class_names}
cam_results = {}

# Pass 1 — prefer correct predictions
with GradCAM(model=wrapped_vit, target_layers=target_layers, reshape_transform=vit_reshape_transform) as cam:
    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="ViT Grad-CAM (correct pass)"):
        lbl = int(row["label"])
        cname = class_names[lbl]
        if got[cname]:
            continue

        p = _row_to_path(row)
        if p is None or not os.path.exists(p):
            continue

        pil_img = Image.open(p).convert("RGB")
        enc = vit_processor(images=pil_img, return_tensors="pt")
        x = enc["pixel_values"].to(device)

        with torch.no_grad():
            pred = vit_model(x).logits.argmax(1).item()
        if pred != lbl:
            continue

        rgb01 = _load_rgb01(p)
        grayscale_cam = cam(input_tensor=x, targets=[ClassifierOutputTarget(lbl)])[0]

        # Resize original to Grad-CAM mask size
        rgb01_resized = resize(rgb01, (grayscale_cam.shape[0], grayscale_cam.shape[1]), preserve_range=True)

        cam_img = show_cam_on_image(rgb01_resized.astype(np.float32), grayscale_cam, use_rgb=True)
        cam_results[cname] = (rgb01_resized, cam_img)
        got[cname] = True

        if all(got.values()):
            break

# Pass 2 — fallback if some classes missing
if not all(got.values()):
    missing = [c for c, v in got.items() if not v]
    print(f"[Info] No correct predictions for classes: {missing}. Falling back to first available sample.")
    with GradCAM(model=wrapped_vit, target_layers=target_layers, reshape_transform=vit_reshape_transform) as cam:
        for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="ViT Grad-CAM (fallback pass)"):
            lbl = int(row["label"])
            cname = class_names[lbl]
            if got[cname]:
                continue

            p = _row_to_path(row)
            if p is None or not os.path.exists(p):
                continue

            pil_img = Image.open(p).convert("RGB")
            enc = vit_processor(images=pil_img, return_tensors="pt")
            x = enc["pixel_values"].to(device)

            rgb01 = _load_rgb01(p)
            grayscale_cam = cam(input_tensor=x, targets=[ClassifierOutputTarget(lbl)])[0]

            rgb01_resized = resize(rgb01, (grayscale_cam.shape[0], grayscale_cam.shape[1]), preserve_range=True)

            cam_img = show_cam_on_image(rgb01_resized.astype(np.float32), grayscale_cam, use_rgb=True)
            cam_results[cname] = (rgb01_resized, cam_img)
            got[cname] = True

            if all(got.values()):
                break

# Plot & save
out_dir = Path("/kaggle/working/gradcam_vit_per_class")
out_dir.mkdir(parents=True, exist_ok=True)

if not cam_results:
    print("[Warning] No Grad-CAM images generated for ViT. Check model/paths.")
else:
    for cname, (orig_rgb01, cam_img) in cam_results.items():
        plt.figure(figsize=(8, 4))
        plt.subplot(1, 2, 1); plt.imshow(orig_rgb01);  plt.title(f"Original — {cname}"); plt.axis('off')
        plt.subplot(1, 2, 2); plt.imshow(cam_img);     plt.title("Grad-CAM");           plt.axis('off')
        plt.tight_layout()
        save_path = out_dir / f"gradcam_{cname}.png"
        plt.savefig(save_path, dpi=300, bbox_inches="tight", pad_inches=0.05)
        plt.show()
    print(f"Saved ViT Grad-CAM images to: {out_dir.resolve()}")

# Cleanup
try:
    cam.activations_and_grads.release()
except:
    pass
del cam

## DenseNet-121

### Load Metadata, Resolve Image Paths, and Prepare Train/Val/Test

In [ ]:
# Load metadata
df = pd.read_csv("/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv")

# Always (re)create label_map and class_names from the current df
label_map = {name: i for i, name in enumerate(sorted(df["dx"].dropna().unique()))}
class_names = [k for k, _ in sorted(label_map.items(), key=lambda x: x[1])]

# Resolve image paths
image_dir_1 = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1"
image_dir_2 = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_2"

def resolve_path(img_id):
    p1 = os.path.join(image_dir_1, f"{img_id}.jpg")
    p2 = os.path.join(image_dir_2, f"{img_id}.jpg")
    if os.path.exists(p1):
        return p1
    if os.path.exists(p2):
        return p2
    return None  # return None if file not found

# Add paths to dataframe
df["path"] = df["image_id"].apply(resolve_path)
df = df.dropna(subset=["path"]).reset_index(drop=True)

# Map dx to integer labels, drop rows if mapping fails
df["label"] = df["dx"].map(label_map)
df = df.dropna(subset=["label"]).reset_index(drop=True)
df["label"] = df["label"].astype(int)

# Split dataset (80/16/20)
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["label"],
    random_state=42
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.20,
    stratify=train_df["label"],
    random_state=42
)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print("Classes:", class_names)

### Transforms - DenseNet

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

### Custom Dataset Class for HAM10000 Skin Lesion Images

In [ ]:
class SkinLesionDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        label = row["label"]

        if self.transform:
            img = self.transform(img)
        return img, label

### DataLoaders (DenseNet)

In [ ]:
train_dataset = SkinLesionDataset(train_df, transform=train_transform)
val_dataset   = SkinLesionDataset(val_df,   transform=val_transform)
test_dataset  = SkinLesionDataset(test_df,  transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

### Define DenseNet121 Model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Modern API: weights=... instead of pretrained=True
densenet = densenet121(weights=DenseNet121_Weights.DEFAULT)
densenet.classifier = nn.Linear(densenet.classifier.in_features, len(class_names))
densenet = densenet.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = Adam(densenet.parameters(), lr=1e-4)

### Train Model with Early Stopping (FIXED to return best_model_wts)

In [ ]:
def train_model(model, criterion, optimizer, train_loader, val_loader, device, num_epochs=50, patience=5):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    patience_counter = 0

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        model.train()
        running_loss, running_corrects = 0.0, 0

        for inputs, labels in tqdm(train_loader, desc="Training", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            preds = outputs.argmax(1)
            running_loss    += loss.item() * inputs.size(0)
            running_corrects += (preds == labels).sum().item()

        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc  = running_corrects / len(train_loader.dataset)
        print(f"Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.4f}")

        # Validation
        model.eval()
        val_corrects = 0
        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc="Validating", leave=False):
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                preds = outputs.argmax(1)
                val_corrects += (preds == labels).sum().item()

        val_acc = val_corrects / len(val_loader.dataset)
        print(f"Val   Acc: {val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            patience_counter = 0
            print("Best DenseNet121 model updated.")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping!")
                break

    return model, best_model_wts

### Train & Save Best Weights

In [ ]:
densenet, best_model_wts = train_model(
    densenet, criterion, optimizer, train_loader, val_loader,
    device, num_epochs=50, patience=5
)

# Save best weights
densenet.load_state_dict(best_model_wts)
torch.save(densenet.state_dict(), "/kaggle/working/densenet121_ham10000.pth")
print("DenseNet121 model saved to /kaggle/working/densenet121_ham10000.pth")

### Reload the Model — DenseNet121

In [ ]:
densenet = densenet121(weights=None)
densenet.classifier = nn.Linear(densenet.classifier.in_features, len(class_names))
densenet.load_state_dict(torch.load("/kaggle/input/datasets/nhsaykat/densenet121-ham10000-model-weights/densenet121_ham10000.pth", map_location=device))
densenet = densenet.to(device)
densenet.eval()

### Evaluate on Test Set — Accuracy, F1, AUC, Confusion Matrix

In [ ]:
all_preds, all_labels, all_probs = [], [], []

densenet.eval()
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images = images.to(device)
        outputs = densenet(images)
        probs = torch.softmax(outputs, dim=1)
        preds = probs.argmax(1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

acc = accuracy_score(all_labels, all_preds)
f1  = f1_score(all_labels, all_preds, average="weighted")
try:
    auc_score = roc_auc_score(all_labels, all_probs, multi_class="ovr")
except ValueError:
    auc_score = "N/A (AUC needs >1 class in test set)"

print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds, target_names=class_names))
print(f"Test Accuracy: {acc:.4f}")
print(f"Weighted F1 Score: {f1:.4f}")
print(f"ROC AUC Score: {auc_score}")

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted Label"); plt.ylabel("True Label")
plt.title("Confusion Matrix — DenseNet121")
plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrix_densenet121.png", dpi=300)
plt.show()
print("Saved: confusion_matrix_densenet121.png")

### ROC‑AUC Curves - DenseNet121

In [ ]:
plot_multiclass_roc(
    all_labels=np.array(all_labels),
    all_probs=np.array(all_probs),
    class_names=class_names,
    title="ROC Curves — DenseNet121",
    outfile="/kaggle/working/roc_curves_densenet121.png"
)

### Grad‑CAM Generation for Test Images — DenseNet‑121 (Plot + Save, with Fallback)

In [ ]:
# Sanity checks
assert 'densenet' in globals(), "densenet model not found. Train/reload DenseNet‑121 first."
assert 'test_loader' in globals(), "test_loader not found."
assert 'class_names' in globals(), "class_names not found."
assert 'device' in globals(), "device not found (cuda/cpu)."

# Inverse of normalize([0.5]*3, [0.5]*3) for correct colors
def inv_norm_05(img_t):
    x = img_t.detach().cpu().clone()
    for c in range(3):
        x[c] = x[c] * 0.5 + 0.5
    return x.clamp(0,1).permute(1,2,0).numpy()  # HWC, [0,1]

# Find the last conv layer (good default CAM target)
def _find_last_conv(m: nn.Module):
    last = None
    for mod in m.modules():
        if isinstance(mod, nn.Conv2d):
            last = mod
    if last is None:
        raise RuntimeError("No Conv2d layer found in DenseNet.")
    return last

densenet.eval()
target_layer = _find_last_conv(densenet)
cam = PTGradCAM(model=densenet, target_layers=[target_layer])

# Collect one image per class (prefer correct predictions), else fallback
got = {c: False for c in class_names}
cam_results = {}

# Pass 1 — correct predictions
for images, labels in test_loader:
    images, labels = images.to(device), labels.to(device)
    with torch.no_grad():
        preds = densenet(images).argmax(1)

    for i in range(images.size(0)):
        t_lbl = labels[i].item()
        p_lbl = preds[i].item()
        cname = class_names[t_lbl]
        if got[cname] or p_lbl != t_lbl:
            continue

        it = images[i].unsqueeze(0).requires_grad_(True)
        densenet.zero_grad()
        gcam = cam(input_tensor=it, targets=[ClassifierOutputTarget(p_lbl)])[0]

        rgb01 = inv_norm_05(images[i])
        cam_img = show_cam_on_image(rgb01, gcam, use_rgb=True)

        cam_results[cname] = (images[i].cpu(), cam_img)
        got[cname] = True
    if all(got.values()):
        break

# Pass 2 — fallback (first available, even if misclassified)
if not all(got.values()):
    missing = [c for c,v in got.items() if not v]
    print(f"[Info] No correct predictions for classes: {missing}. Falling back to first available sample.")
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        for i in range(images.size(0)):
            t_lbl = labels[i].item()
            cname = class_names[t_lbl]
            if got[cname]:
                continue

            it = images[i].unsqueeze(0).requires_grad_(True)
            densenet.zero_grad()
            gcam = cam(input_tensor=it, targets=[ClassifierOutputTarget(t_lbl)])[0]

            rgb01 = inv_norm_05(images[i])
            cam_img = show_cam_on_image(rgb01, gcam, use_rgb=True)

            cam_results[cname] = (images[i].cpu(), cam_img)
            got[cname] = True
        if all(got.values()):
            break

# Plot & save
out_dir = Path("/kaggle/working/gradcam_densenet121_per_class")
out_dir.mkdir(parents=True, exist_ok=True)

if not cam_results:
    print("[Warning] No Grad‑CAM images generated for DenseNet‑121. Check model/loader.")
else:
    for cname, (img_t, cam_img) in cam_results.items():
        orig = inv_norm_05(img_t)
        plt.figure(figsize=(8,4))
        plt.subplot(1,2,1); plt.imshow(orig);   plt.title(f"Original — {cname}"); plt.axis('off')
        plt.subplot(1,2,2); plt.imshow(cam_img); plt.title("Grad‑CAM");           plt.axis('off')
        plt.tight_layout()
        save_path = out_dir / f"gradcam_{cname}.png"
        plt.savefig(save_path, dpi=300, bbox_inches="tight", pad_inches=0.05)
        plt.show()
    print(f"Saved DenseNet‑121 Grad‑CAM images to: {out_dir.resolve()}")

# Cleanup hooks
try:
    cam.activations_and_grads.release()
except Exception:
    pass
del cam

## EfficientNet-B3

### Add/Ensure Image Paths — EfficientNet‑B3

In [ ]:
# Define image directories for HAM10000
image_dir_1 = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1"
image_dir_2 = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_2"

# Function to resolve full image path
def resolve_path(img_id):
    f1 = os.path.join(image_dir_1, f"{img_id}.jpg")
    f2 = os.path.join(image_dir_2, f"{img_id}.jpg")
    return f1 if os.path.exists(f1) else f2

# Apply to existing 'image_id' column
df["path"] = df["image_id"].apply(resolve_path)

# Check results
print("Sample resolved paths:")
print(df[["image_id", "path"]].head())

### Transforms — EfficientNet‑B3 (300×300)

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

val_transform = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

### Define SkinLesionDataset Class

In [ ]:
class SkinLesionDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = row["label"]
        return image, label

### Validate and Clean Image Paths

In [ ]:
# Check for missing image paths
df["path"] = df["image_id"].apply(resolve_path)  # <- use image_id here
df["file_exists"] = df["path"].apply(lambda p: os.path.exists(p))

# Drop rows where the image file does not exist
df = df[df["file_exists"]].reset_index(drop=True)

# Drop the helper column
df = df.drop(columns=["file_exists"])

### Train/Val/Test Split & DataLoaders — EfficientNet‑B3

In [ ]:
# 70/15/15 or 80/10/10
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=42)
val_df,   test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=42)

train_dataset = SkinLesionDataset(train_df, transform=train_transform)
val_dataset   = SkinLesionDataset(val_df,   transform=val_transform)
test_dataset  = SkinLesionDataset(test_df,  transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

### EfficientNet‑B3 Initialization (TorchVision)

In [ ]:
# Set number of output classes
num_classes = 7  # or use len(class_names) if already defined

# Initialize EfficientNet-B3 without pretrained weights (no download)
effnet = efficientnet_b3(weights=None)

# Replace classifier layer
effnet.classifier[1] = nn.Linear(effnet.classifier[1].in_features, num_classes)

# Move to GPU if available
effnet = effnet.to(device)

# Define loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(effnet.parameters(), lr=1e-4)

### Train Function — EfficientNet‑B3 (Early Stopping)

In [ ]:
def train_model_effnet(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=50, patience=5):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    patience_counter = 0

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        model.train()
        running_loss, running_corrects = 0.0, 0

        for inputs, labels in tqdm(train_loader, desc="Training", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            preds = outputs.argmax(1)
            running_loss += loss.item() * inputs.size(0)
            running_corrects += (preds == labels).sum().item()

        train_loss = running_loss / len(train_loader.dataset)
        train_acc  = running_corrects / len(train_loader.dataset)

        # Validation phase
        model.eval()
        val_corrects = 0
        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc="Validating", leave=False):
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                preds = outputs.argmax(1)
                val_corrects += (preds == labels).sum().item()

        val_acc = val_corrects / len(val_loader.dataset)
        print(f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping!")
                break

    return model, best_model_wts

###  Train & Save the Best EfficientNet‑B3

In [ ]:
# Train EfficientNet-B3 on HAM10000
effnet, best_wts = train_model_effnet(
    effnet, train_loader, val_loader, criterion, optimizer, device,
    num_epochs=50, patience=5
)

# Load best weights
effnet.load_state_dict(best_wts)

# Save to disk
torch.save(effnet.state_dict(), "/kaggle/working/efficientnet_b3_best.pth")
print("EfficientNet‑B3 model saved to /kaggle/working/efficientnet_b3_best.pth")

### Reload the Model — EfficientNet‑B3

In [ ]:
# Define class names again
class_names = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']

# Reload the Model — EfficientNet‑B3
effnet = efficientnet_b3(weights=None)
effnet.classifier[1] = nn.Linear(effnet.classifier[1].in_features, len(class_names))
effnet.load_state_dict(torch.load("/kaggle/input/datasets/nhsaykat/efficientnetb3-ham10000-model-weights/efficientnet_b3_best.pth", map_location=device))
effnet = effnet.to(device)
effnet.eval()

### Evaluate on Test Set — EfficientNet‑B3

In [ ]:
all_preds, all_labels, all_probs = [], [], []

effnet.eval()
with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="Testing"):
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = effnet(inputs)
        probs = torch.softmax(outputs, dim=1)
        preds = probs.argmax(1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

acc = accuracy_score(all_labels, all_preds)
f1  = f1_score(all_labels, all_preds, average="weighted")
try:
    auc_score = roc_auc_score(all_labels, all_probs, multi_class="ovr")
except ValueError:
    auc_score = "N/A (AUC needs >1 class in test set)"

print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds, target_names=class_names))
print(f"Test Accuracy: {acc:.4f}")
print(f"Weighted F1 Score: {f1:.4f}")
print(f"ROC AUC Score: {auc_score}")

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title("Confusion Matrix — EfficientNet‑B3")
plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrix_efficientnetb3.png", dpi=300)
plt.show()

### ROC‑AUC Curves — EfficientNet‑B3

In [ ]:
plot_multiclass_roc(
    all_labels=all_labels,
    all_probs=all_probs,
    class_names=class_names,
    title="ROC Curves — EfficientNet‑B3",
    outfile="/kaggle/working/roc_curves_efficientnetb3.png"
)

### Grad‑CAM Generation for Test Images — EfficientNet‑B3 (Plot + Save, with Fallback)

In [ ]:
# Pick objects / sanity checks
effnet_model = globals().get("effnet", None) or globals().get("effnet_model", None)
assert effnet_model is not None, "EfficientNet-B3 model not found."
test_loader_effnet = globals().get("test_loader_effnet", None) or globals().get("test_loader", None)
assert test_loader_effnet is not None, "Test loader not found."
assert 'class_names' in globals(), "class_names not found."
assert 'device' in globals(), "device not found."

# Helper functions
def inv_norm_05(img_t):
    """Invert Normalize([0.5]*3,[0.5]*3) to RGB [0,1]. img_t: CHW tensor."""
    x = img_t.detach().cpu().clone()
    for c in range(3):
        x[c] = x[c] * 0.5 + 0.5
    return x.clamp(0, 1).permute(1, 2, 0).numpy()

def find_last_conv_module(net: nn.Module):
    last = None
    for m in net.modules():
        if isinstance(m, nn.Conv2d):
            last = m
    if last is None:
        raise RuntimeError("No Conv2d layer found in the model.")
    return last

# Setup Grad-CAM
effnet_model.eval()
target_layer = find_last_conv_module(effnet_model)
cam = PTGradCAM(model=effnet_model, target_layers=[target_layer])

got = {c: False for c in class_names}
cam_results = {}

# Pass 1: prefer correctly predicted examples
for images, labels in test_loader_effnet:
    images, labels = images.to(device), labels.to(device)
    with torch.no_grad():
        preds = effnet_model(images).argmax(1)

    for i in range(images.size(0)):
        t_lbl = labels[i].item()
        p_lbl = preds[i].item()
        cname = class_names[t_lbl]
        if got[cname] or p_lbl != t_lbl:
            continue

        it = images[i].unsqueeze(0).requires_grad_(True)
        effnet_model.zero_grad()
        grayscale_cam = cam(input_tensor=it, targets=[ClassifierOutputTarget(p_lbl)])[0]

        rgb01 = inv_norm_05(images[i])
        cam_img = show_cam_on_image(rgb01, grayscale_cam, use_rgb=True)

        cam_results[cname] = (rgb01, cam_img)
        got[cname] = True

    if all(got.values()):
        break

# Pass 2: fallback for missing classes
if not all(got.values()):
    missing = [c for c, v in got.items() if not v]
    print(f"[Info] No correct predictions for classes: {missing}. Falling back to first available sample.")
    for images, labels in test_loader_effnet:
        images, labels = images.to(device), labels.to(device)
        for i in range(images.size(0)):
            t_lbl = labels[i].item()
            cname = class_names[t_lbl]
            if got[cname]:
                continue

            it = images[i].unsqueeze(0).requires_grad_(True)
            effnet_model.zero_grad()
            grayscale_cam = cam(input_tensor=it, targets=[ClassifierOutputTarget(t_lbl)])[0]

            rgb01 = inv_norm_05(images[i])
            cam_img = show_cam_on_image(rgb01, grayscale_cam, use_rgb=True)

            cam_results[cname] = (rgb01, cam_img)
            got[cname] = True

        if all(got.values()):
            break

# Plot & save
out_dir = Path("/kaggle/working/gradcam_efficientnet_b3_per_class")
out_dir.mkdir(parents=True, exist_ok=True)

if not cam_results:
    print("[Warning] No Grad-CAM images generated for EfficientNet-B3.")
else:
    for cname, (orig_rgb01, cam_img) in cam_results.items():
        plt.figure(figsize=(8, 4))
        plt.subplot(1, 2, 1); plt.imshow(orig_rgb01); plt.title(f"Original — {cname}"); plt.axis('off')
        plt.subplot(1, 2, 2); plt.imshow(cam_img);     plt.title("Grad-CAM");          plt.axis('off')
        plt.tight_layout()
        sp = out_dir / f"gradcam_{cname}.png"
        plt.savefig(sp, dpi=300, bbox_inches="tight", pad_inches=0.05)
        plt.show()
    print(f"Saved EfficientNet-B3 Grad-CAM images to: {out_dir.resolve()}")

# Cleanup hooks
try:
    cam.activations_and_grads.release()
except Exception:
    pass
del cam

## ConvNeXt-Tiny

### Reproducibility & ImageNet Normalization Setup

In [ ]:
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ImageNet stats (ConvNeXt expects these)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

### Dataset, Fixed Label Map, Splits, Transforms, Loaders

In [ ]:
# Dataset class that reads PIL images from df['path'] and returns (tensor, label)
class SkinLesionDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        p = self.df.loc[idx, "path"]
        y = int(self.df.loc[idx, "label"])
        img = Image.open(p).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, y

# Resolve image paths from both folders
img_dir1 = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1"
img_dir2 = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_2"

def resolve_path(img_id):
    p1 = os.path.join(img_dir1, f"{img_id}.jpg")
    p2 = os.path.join(img_dir2, f"{img_id}.jpg")
    return p1 if os.path.exists(p1) else p2

# Load metadata
df = pd.read_csv("/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv")
df["path"] = df["image_id"].apply(resolve_path)

# Use a stable label mapping (alphabetical is fine)
class_names = sorted(df["dx"].unique().tolist())
label_map = {name: i for i, name in enumerate(class_names)}
df["label"] = df["dx"].map(label_map).astype(int)

# Split 70/15/15 stratified
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["label"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"], random_state=42
)

print("Split sizes:", len(train_df), len(val_df), len(test_df))
print("Classes:", class_names)

# Use ImageNet normalization everywhere for ConvNeXt
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Datasets & loaders
train_ds = SkinLesionDataset(train_df, transform=train_tf)
val_ds   = SkinLesionDataset(val_df,   transform=eval_tf)
test_ds  = SkinLesionDataset(test_df,  transform=eval_tf)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

### Model Init (ConvNeXt‑Tiny), Loss, Optimizer, Scheduler

In [ ]:
convnext = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
convnext.classifier[2] = nn.Linear(convnext.classifier[2].in_features, len(class_names))
convnext = convnext.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(convnext.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", patience=2, factor=0.5
)

### Train + Early Stopping (AMP), Save Best

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer,
                device, num_epochs=50, patience=5, scheduler=None, use_amp=True):
    best_wts = deepcopy(model.state_dict())
    best_acc = 0.0
    no_improve = 0

    scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and device.type == "cuda"))

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")

        # Train
        model.train()
        tr_loss, tr_correct, tr_seen = 0.0, 0, 0
        for x, y in tqdm(train_loader, desc="Training", leave=False):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=scaler.is_enabled()):
                logits = model(x)
                loss = criterion(logits, y)

            if scaler.is_enabled():
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()

            preds = logits.argmax(1)
            tr_loss   += loss.item() * x.size(0)
            tr_correct+= (preds == y).sum().item()
            tr_seen   += y.size(0)

        train_loss = tr_loss / tr_seen
        train_acc  = tr_correct / tr_seen

        # Validate
        model.eval()
        val_correct, val_seen = 0, 0
        with torch.no_grad():
            for x, y in tqdm(val_loader, desc="Validating", leave=False):
                x, y = x.to(device), y.to(device)
                logits = model(x)
                val_correct += (logits.argmax(1) == y).sum().item()
                val_seen    += y.size(0)

        val_acc = val_correct / val_seen
        if scheduler is not None:
            scheduler.step(val_acc)

        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val   Acc: {val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            best_wts = deepcopy(model.state_dict())
            no_improve = 0
            print("New best model (by Val Acc).")
        else:
            no_improve += 1
            if no_improve >= patience:
                print("Early stopping triggered.")
                break

    model.load_state_dict(best_wts)
    print(f"Best Val Acc: {best_acc:.4f}")

    # Save best
    save_path = "/kaggle/working/convnext_tiny_ham10000.pth"
    torch.save(model.state_dict(), save_path)
    print(f"Saved best model to {save_path}")
    return model

convnext = train_model(
    convnext, train_loader, val_loader, criterion, optimizer,
    device, num_epochs=50, patience=5, scheduler=scheduler, use_amp=True
)

### ConvNeXt-Tiny Split-Overlap Audit

To verify that the revised canonical evaluation does not inadvertently include samples used during ConvNeXt-Tiny training or validation, the original stratified 70/15/15 split is reconstructed using the same label mapping and random seed (random_state=42) employed during model development.

The image identifiers in the canonical 2,003-image evaluation set are then compared with the reconstructed ConvNeXt-Tiny training, validation, and test partitions. This audit is used to determine whether the canonical evaluation set constitutes an independent held-out test set for the archived ConvNeXt-Tiny checkpoint.

In [ ]:
# ConvNeXt Split-Overlap Audit

# Reconstruct the exact ConvNeXt 70/15/15 split
meta_conv = pd.read_csv(META_PATH)

canonical_classes = sorted(meta_conv["dx"].unique().tolist())
conv_label_map = {
    name: i for i, name in enumerate(canonical_classes)
}

meta_conv["label"] = (
    meta_conv["dx"]
    .map(conv_label_map)
    .astype(int)
)

conv_train_df, conv_temp_df = train_test_split(
    meta_conv,
    test_size=0.30,
    stratify=meta_conv["label"],
    random_state=42
)

conv_val_df, conv_test_df = train_test_split(
    conv_temp_df,
    test_size=0.50,
    stratify=conv_temp_df["label"],
    random_state=42
)

# Image-ID sets
canonical_ids = set(
    canonical_test_df["image_id"].astype(str)
)

conv_train_ids = set(
    conv_train_df["image_id"].astype(str)
)

conv_val_ids = set(
    conv_val_df["image_id"].astype(str)
)

conv_test_ids = set(
    conv_test_df["image_id"].astype(str)
)

# Overlaps
overlap_train = canonical_ids & conv_train_ids
overlap_val   = canonical_ids & conv_val_ids
overlap_test  = canonical_ids & conv_test_ids

print("Canonical evaluation images:", len(canonical_ids))

print("\nOverlap with ConvNeXt training set:")
print(len(overlap_train))

print("\nOverlap with ConvNeXt validation set:")
print(len(overlap_val))

print("\nOverlap with ConvNeXt original test set:")
print(len(overlap_test))

print("\nTotal accounted for:")
print(
    len(overlap_train)
    + len(overlap_val)
    + len(overlap_test)
)

print("\nConvNeXt original split sizes:")
print("Train:", len(conv_train_df))
print("Validation:", len(conv_val_df))
print("Test:", len(conv_test_df))

### ResNet-50 and ViT Historical Split Audit

The original ResNet-50 data preparation used a stratified 80/20
training–test split with `random_state=42`, followed by a 90/10 split of
the training portion for validation. Because the ViT training block uses
pre-existing `train_loader` and `val_loader` objects and does not define a
new split in the available ViT section, this audit first reconstructs the
historical ResNet-50 split and compares it with the revised canonical
2,003-image evaluation set. The result is used to determine whether the
canonical evaluation set corresponds to the historical held-out test
partition.

In [ ]:
# ResNet-50 / ViT Historical Split Audit

from sklearn.model_selection import train_test_split

# Load metadata in original row order
meta_rv = pd.read_csv(META_PATH)

# Reproduce original metadata-order label mapping
rv_label_dict = {
    label: idx
    for idx, label in enumerate(meta_rv["dx"].unique())
}

meta_rv["label"] = meta_rv["dx"].map(rv_label_dict).astype(int)
meta_rv["image_id"] = meta_rv["image_id"].astype(str)

print("Historical ResNet/ViT label order:")
print(rv_label_dict)

# Reconstruct original ResNet-50 split exactly
# 80% development / 20% test
# then 90/10 split of development into train/validation

rv_train_df, rv_test_df = train_test_split(
    meta_rv,
    test_size=0.20,
    stratify=meta_rv["label"],
    random_state=42
)

rv_train_df, rv_val_df = train_test_split(
    rv_train_df,
    test_size=0.10,
    stratify=rv_train_df["label"],
    random_state=42
)

rv_train_df = rv_train_df.reset_index(drop=True)
rv_val_df   = rv_val_df.reset_index(drop=True)
rv_test_df  = rv_test_df.reset_index(drop=True)

# Build image-ID sets
canonical_ids = set(
    canonical_test_df["image_id"].astype(str)
)

rv_train_ids = set(
    rv_train_df["image_id"].astype(str)
)

rv_val_ids = set(
    rv_val_df["image_id"].astype(str)
)

rv_test_ids = set(
    rv_test_df["image_id"].astype(str)
)

# Compute overlaps
overlap_rv_train = canonical_ids & rv_train_ids
overlap_rv_val   = canonical_ids & rv_val_ids
overlap_rv_test  = canonical_ids & rv_test_ids

print("\nCanonical evaluation images:")
print(len(canonical_ids))

print("\nOverlap with historical ResNet/ViT training set:")
print(len(overlap_rv_train))

print("\nOverlap with historical ResNet/ViT validation set:")
print(len(overlap_rv_val))

print("\nOverlap with historical ResNet/ViT test set:")
print(len(overlap_rv_test))

print("\nHistorical split sizes:")
print("Train:", len(rv_train_df))
print("Validation:", len(rv_val_df))
print("Test:", len(rv_test_df))

# Direct equality check
same_test_membership = canonical_ids == rv_test_ids

print("\nCanonical set identical to historical test set:")
print(same_test_membership)

if same_test_membership:
    print(
        "\nConfirmed: the canonical 2,003-image set is exactly "
        "the historical ResNet-50 test partition."
    )
else:
    print(
        "\nThe canonical set differs from the historical "
        "ResNet-50 test partition."
    )

### Preserve Historical ResNet-50/ViT Split Membership

The reconstructed historical ResNet-50 split is retained for subsequent
cross-model split auditing. The current 2,003-image evaluation set is not
identical to the historical ResNet-50 test partition; therefore, final
evaluation will be restricted to samples that are demonstrably unseen by
all archived component models.

In [ ]:
# Preserve ResNet-50 / ViT Historical Split Membership

resnet_vit_split_membership = {
    "train": rv_train_ids,
    "val": rv_val_ids,
    "test": rv_test_ids,
}

print("Historical ResNet/ViT memberships preserved.")
print("Train:", len(resnet_vit_split_membership["train"]))
print("Validation:", len(resnet_vit_split_membership["val"]))
print("Test:", len(resnet_vit_split_membership["test"]))

assert len(rv_test_ids) == 2003
assert len(rv_train_ids & rv_test_ids) == 0
assert len(rv_val_ids & rv_test_ids) == 0

### Common Untouched Test Subset: ResNet-50/ViT and ConvNeXt-Tiny

The historical test partitions of the ResNet-50/ViT pipeline and the
ConvNeXt-Tiny pipeline are intersected to identify images that were assigned
to the test partition in both historical workflows. These samples were
therefore excluded from training and validation for both pipelines and form
an initial common untouched evaluation subset.

In [ ]:
# ResNet-50 / ViT + ConvNeXt-Tiny

# Intersection of the historical test partitions
common_test_ids_rv_convnext = (
    rv_test_ids
    & conv_test_ids
)

print(
    "Common untouched test images "
    "for ResNet-50/ViT and ConvNeXt-Tiny:",
    len(common_test_ids_rv_convnext)
)

# Safety checks:
# none of these images may appear in training or validation
# for either historical pipeline

assert len(
    common_test_ids_rv_convnext & rv_train_ids
) == 0

assert len(
    common_test_ids_rv_convnext & rv_val_ids
) == 0

assert len(
    common_test_ids_rv_convnext & conv_train_ids
) == 0

assert len(
    common_test_ids_rv_convnext & conv_val_ids
) == 0

print("\nLeakage checks passed:")
print("ResNet/ViT training overlap:   0")
print("ResNet/ViT validation overlap: 0")
print("ConvNeXt training overlap:     0")
print("ConvNeXt validation overlap:   0")

# Optional percentage information
print(
    "\nPercentage of ResNet/ViT historical test set retained:",
    f"{100 * len(common_test_ids_rv_convnext) / len(rv_test_ids):.2f}%"
)

print(
    "Percentage of ConvNeXt historical test set retained:",
    f"{100 * len(common_test_ids_rv_convnext) / len(conv_test_ids):.2f}%"
)

### Historical Split Audit for the Remaining Models

The ResNet-50/ViT and ConvNeXt-Tiny pipelines used different historical
data partitions. Before defining a final common evaluation cohort, the
training, validation, and test partitioning procedures used by CNN,
DenseNet-121, EfficientNet-B3, and MobileNetV3 must therefore be
reconstructed from their original notebook sections.

The objective is to identify images that were held out from both training
and validation for every archived model. No model is retrained during
this audit.

### Historical Split Audit: DenseNet-121, EfficientNet-B3, and MobileNetV3

The original notebook used different train/validation/test partitioning
procedures across several architectures. To determine whether a common
untouched evaluation cohort can be constructed without retraining, the
historical splits for DenseNet-121, EfficientNet-B3, and MobileNetV3 are
reconstructed using the split ratios, stratification settings, and
`random_state=42` recorded in their original training sections.

The resulting image-ID memberships are preserved and compared with the
previously reconstructed ResNet-50/ViT and ConvNeXt-Tiny historical test
partitions. No model is retrained or re-fitted during this audit.

In [ ]:
# Historical Split Audit:
# DenseNet-121, EfficientNet-B3, MobileNetV3

# Load HAM10000 metadata in original row order
META_PATH = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv"
meta_hist = pd.read_csv(META_PATH)
meta_hist["image_id"] = meta_hist["image_id"].astype(str)

# These three supplied sections use the stable alphabetical
# HAM10000 label mapping.
hist_class_names = sorted(
    meta_hist["dx"].dropna().unique().tolist()
)

hist_label_map = {
    name: i
    for i, name in enumerate(hist_class_names)
}

meta_hist["label"] = (
    meta_hist["dx"]
    .map(hist_label_map)
    .astype(int)
)

print("Audit class order:")
print(hist_class_names)

# DenseNet-121 historical split
#
# First:
#   80% development / 20% test
#
# Then:
#   80% of development = train
#   20% of development = validation
#
# Effective:
#   64% train / 16% validation / 20% test

dense_train_df, dense_test_df = train_test_split(
    meta_hist,
    test_size=0.20,
    stratify=meta_hist["label"],
    random_state=42
)

dense_train_df, dense_val_df = train_test_split(
    dense_train_df,
    test_size=0.20,
    stratify=dense_train_df["label"],
    random_state=42
)

dense_train_ids = set(
    dense_train_df["image_id"].astype(str)
)

dense_val_ids = set(
    dense_val_df["image_id"].astype(str)
)

dense_test_ids = set(
    dense_test_df["image_id"].astype(str)
)

# EfficientNet-B3 historical split
#
# 70% train / 15% validation / 15% test

eff_train_df, eff_temp_df = train_test_split(
    meta_hist,
    test_size=0.30,
    stratify=meta_hist["label"],
    random_state=42
)

eff_val_df, eff_test_df = train_test_split(
    eff_temp_df,
    test_size=0.50,
    stratify=eff_temp_df["label"],
    random_state=42
)

eff_train_ids = set(
    eff_train_df["image_id"].astype(str)
)

eff_val_ids = set(
    eff_val_df["image_id"].astype(str)
)

eff_test_ids = set(
    eff_test_df["image_id"].astype(str)
)

# MobileNetV3 historical split
#
# First:
#   80% development / 20% test
#
# Then:
#   80% of development = train
#   20% of development = validation
#
# Effective:
#   64% train / 16% validation / 20% test

mobile_temp_df, mobile_test_df = train_test_split(
    meta_hist,
    test_size=0.20,
    stratify=meta_hist["label"],
    random_state=42
)

mobile_train_df, mobile_val_df = train_test_split(
    mobile_temp_df,
    test_size=0.20,
    stratify=mobile_temp_df["label"],
    random_state=42
)

mobile_train_ids = set(
    mobile_train_df["image_id"].astype(str)
)

mobile_val_ids = set(
    mobile_val_df["image_id"].astype(str)
)

mobile_test_ids = set(
    mobile_test_df["image_id"].astype(str)
)

# Verification
print("\nDenseNet-121 historical split:")
print("Train:", len(dense_train_ids))
print("Validation:", len(dense_val_ids))
print("Test:", len(dense_test_ids))

print("\nEfficientNet-B3 historical split:")
print("Train:", len(eff_train_ids))
print("Validation:", len(eff_val_ids))
print("Test:", len(eff_test_ids))

print("\nMobileNetV3 historical split:")
print("Train:", len(mobile_train_ids))
print("Validation:", len(mobile_val_ids))
print("Test:", len(mobile_test_ids))

# Ensure each reconstructed split is internally disjoint
for model_name, train_ids, val_ids, test_ids in [
    (
        "DenseNet-121",
        dense_train_ids,
        dense_val_ids,
        dense_test_ids
    ),
    (
        "EfficientNet-B3",
        eff_train_ids,
        eff_val_ids,
        eff_test_ids
    ),
    (
        "MobileNetV3",
        mobile_train_ids,
        mobile_val_ids,
        mobile_test_ids
    ),
]:

    assert len(train_ids & val_ids) == 0, \
        f"{model_name}: train/validation overlap"

    assert len(train_ids & test_ids) == 0, \
        f"{model_name}: train/test overlap"

    assert len(val_ids & test_ids) == 0, \
        f"{model_name}: validation/test overlap"

print("\nHistorical split leakage checks passed.")

### Historical Test-Set Membership Comparison

The reconstructed historical test memberships are compared to determine
whether architectures using nominally similar split ratios actually share
the same held-out samples. This comparison is important because identical
test-set sizes do not necessarily imply identical image membership.

In [ ]:
# Historical Test-Set Membership Comparison

print("DenseNet test identical to MobileNet test:")
print(dense_test_ids == mobile_test_ids)

print("\nEfficientNet test identical to ConvNeXt test:")
print(eff_test_ids == conv_test_ids)

print("\nDenseNet test identical to ResNet/ViT test:")
print(dense_test_ids == rv_test_ids)

print("\nMobileNet test identical to ResNet/ViT test:")
print(mobile_test_ids == rv_test_ids)

print("\nDenseNet / ResNet-ViT test intersection:")
print(len(dense_test_ids & rv_test_ids))

print("\nEfficientNet / ConvNeXt test intersection:")
print(len(eff_test_ids & conv_test_ids))

### Progressive Common Untouched Test Intersection

The historical test memberships of the audited model pipelines are
intersected to determine how many HAM10000 images were assigned to the
held-out test partition for every audited architecture. Samples appearing
in any corresponding training or validation partition are explicitly
excluded.

In [ ]:
# Progressive Common Untouched Test Intersection

common_test_ids_audited = (
    rv_test_ids
    & conv_test_ids
    & dense_test_ids
    & eff_test_ids
    & mobile_test_ids
)

print(
    "Common untouched test images across "
    "ResNet/ViT, ConvNeXt, DenseNet, "
    "EfficientNet and MobileNet:"
)

print(len(common_test_ids_audited))

# Leakage verification against every audited train/val set
non_test_sets = {
    "ResNet/ViT train": rv_train_ids,
    "ResNet/ViT val": rv_val_ids,

    "ConvNeXt train": conv_train_ids,
    "ConvNeXt val": conv_val_ids,

    "DenseNet train": dense_train_ids,
    "DenseNet val": dense_val_ids,

    "EfficientNet train": eff_train_ids,
    "EfficientNet val": eff_val_ids,

    "MobileNet train": mobile_train_ids,
    "MobileNet val": mobile_val_ids,
}

for name, ids in non_test_sets.items():

    overlap = common_test_ids_audited & ids

    assert len(overlap) == 0, \
        f"Leakage detected against {name}: {len(overlap)}"

print("\nCommon-set leakage verification passed.")

### Class Distribution of the Common Untouched Test Subset

The class distribution of the progressively intersected test subset is examined to verify that all seven HAM10000 diagnostic classes remain represented after restricting the analysis to images that were held out from training and validation across all audited model pipelines.

In [ ]:
# Class Distribution of Audited Common Test Set

audited_common_df = meta_hist[
    meta_hist["image_id"].isin(
        common_test_ids_audited
    )
].copy()

audited_counts = (
    audited_common_df["label"]
    .value_counts()
    .sort_index()
)

print(
    "Audited common untouched subset size:",
    len(audited_common_df)
)

print("\nClass distribution:")

for idx, cname in enumerate(hist_class_names):

    count = int(
        audited_counts.get(idx, 0)
    )

    print(
        f"{idx}  {cname:6s}: {count}"
    )

print(
    "\nRepresented classes:",
    (audited_counts > 0).sum(),
    "of",
    len(hist_class_names)
)

### CNN Historical Split Provenance

The archived CNN architecture and trained checkpoint are available; however,
the current computational notebook does not retain the original CNN-specific
construction of the training and validation partitions. The CNN training block
consumes pre-existing `train_loader` and `val_loader` objects, whereas the
available preceding data-preparation code does not establish CNN-specific
training and validation loaders. Consequently, the exact historical CNN
train/validation membership cannot be reconstructed reliably from the
available notebook state and is therefore not inferred retrospectively.

### Master Historical Protocol Audit

The preceding reproducibility audit identified multiple historical data-partitioning
and evaluation states across the archived HAM10000 model pipelines. The table
below consolidates the verified information for each architecture, including the
historical split procedure, test-set size, class-index convention, preprocessing,
and current provenance status.

This audit table is descriptive and does not constitute a new experimental result.
No model is retrained. Its purpose is to identify which aspects of the submitted
evaluation can be reproduced directly and which require correction or explicit
qualification before regenerating the main and supplementary result tables.

In [ ]:
# Master Historical Protocol Audit

protocol_audit_rows = [
    {
        "Model": "CNN",
        "Historical split": "Not recoverable from current notebook state",
        "Train N": np.nan,
        "Validation N": np.nan,
        "Test N": np.nan,
        "Historical test membership": "Unresolved",
        "Training label order": "Canonical/alphabetical assumed in available evaluation",
        "Evaluation preprocessing": "224x224; Normalize([0.5]*3, [0.5]*3)",
        "Table 1 reproduced": "Yes",
        "Audit status": "Checkpoint/architecture available; historical split unresolved",
    },

    {
        "Model": "ResNet-50",
        "Historical split": "80/20, then 90/10 of development",
        "Train N": 7210,
        "Validation N": 802,
        "Test N": 2003,
        "Historical test membership": "Verified",
        "Training label order": "Metadata order: bkl,nv,df,mel,vasc,bcc,akiec",
        "Evaluation preprocessing": "224x224; ImageNet normalization",
        "Table 1 reproduced": "Yes",
        "Audit status": "Historical split and class-index mapping reconstructed",
    },

    {
        "Model": "DenseNet-121",
        "Historical split": "80/20, then 80/20 of development",
        "Train N": 6409,
        "Validation N": 1603,
        "Test N": 2003,
        "Historical test membership": "Verified",
        "Training label order": "Canonical alphabetical",
        "Evaluation preprocessing": "224x224; Normalize([0.5]*3, [0.5]*3)",
        "Table 1 reproduced": "Yes",
        "Audit status": "Historical split reconstructed",
    },

    {
        "Model": "EfficientNet-B3",
        "Historical split": "70/15/15",
        "Train N": 7010,
        "Validation N": 1502,
        "Test N": 1503,
        "Historical test membership": "Verified",
        "Training label order": "Canonical alphabetical",
        "Evaluation preprocessing": "300x300; Normalize([0.5]*3, [0.5]*3)",
        "Table 1 reproduced": "Yes",
        "Audit status": "Historical split reconstructed",
    },

    {
        "Model": "ConvNeXt-Tiny",
        "Historical split": "70/15/15",
        "Train N": 7010,
        "Validation N": 1502,
        "Test N": 1503,
        "Historical test membership": "Verified",
        "Training label order": "Canonical alphabetical",
        "Evaluation preprocessing": "224x224; ImageNet normalization",
        "Table 1 reproduced": "Only with historical 0.5-normalized evaluation",
        "Audit status": (
            "Training used ImageNet normalization; submitted Table 1 values "
            "were reproduced using mismatched 0.5 normalization"
        ),
    },

    {
        "Model": "MobileNetV3-L",
        "Historical split": "80/20, then 80/20 of development",
        "Train N": 6409,
        "Validation N": 1603,
        "Test N": 2003,
        "Historical test membership": "Verified",
        "Training label order": "Canonical alphabetical",
        "Evaluation preprocessing": "224x224; Normalize([0.5]*3, [0.5]*3)",
        "Table 1 reproduced": "Yes",
        "Audit status": "Historical split reconstructed",
    },

    {
        "Model": "ViT-B/16",
        "Historical split": (
            "Training uses pre-existing train_loader/val_loader; "
            "ResNet-style split is strongly indicated but not independently "
            "defined in available ViT section"
        ),
        "Train N": np.nan,
        "Validation N": np.nan,
        "Test N": np.nan,
        "Historical test membership": "Not independently verified for ViT",
        "Training label order": "Metadata order: bkl,nv,df,mel,vasc,bcc,akiec",
        "Evaluation preprocessing": "ViTImageProcessor (google/vit-base-patch16-224)",
        "Table 1 reproduced": "Yes (0.9591 accuracy)",
        "Audit status": (
            "Class-index mapping verified; historical ViT split provenance "
            "depends on pre-existing loaders"
        ),
    },
]

protocol_audit_df = pd.DataFrame(protocol_audit_rows)

display(protocol_audit_df)

### Add the Verified Cross-Pipeline Membership Facts

In [ ]:
# Verified Historical Test-Membership Relationships

membership_summary = pd.DataFrame([
    {
        "Comparison": "DenseNet-121 vs MobileNetV3-L",
        "Same historical test membership": dense_test_ids == mobile_test_ids,
        "Intersection N": len(dense_test_ids & mobile_test_ids),
    },
    {
        "Comparison": "EfficientNet-B3 vs ConvNeXt-Tiny",
        "Same historical test membership": eff_test_ids == conv_test_ids,
        "Intersection N": len(eff_test_ids & conv_test_ids),
    },
    {
        "Comparison": "DenseNet-121 vs ResNet-50",
        "Same historical test membership": dense_test_ids == rv_test_ids,
        "Intersection N": len(dense_test_ids & rv_test_ids),
    },
    {
        "Comparison": "MobileNetV3-L vs ResNet-50",
        "Same historical test membership": mobile_test_ids == rv_test_ids,
        "Intersection N": len(mobile_test_ids & rv_test_ids),
    },
])

print("Historical test-membership relationships:")
display(membership_summary)

### Add the Common-Test Sensitivity Audit Summary

In [ ]:
# Common Untouched Sensitivity Cohort Summary

common_audit_summary = pd.DataFrame([
    {
        "Audited pipelines": "ResNet/ViT + ConvNeXt",
        "Common untouched N": len(common_test_ids_rv_convnext),
        "All 7 classes represented": True,
    },
    {
        "Audited pipelines": (
            "ResNet/ViT + ConvNeXt + DenseNet + "
            "EfficientNet + MobileNet"
        ),
        "Common untouched N": len(common_test_ids_audited),
        "All 7 classes represented": len(audited_counts) == 7,
    },
])

display(common_audit_summary)

print("\nFinal audited common-subset class counts:")
for idx, cname in enumerate(hist_class_names):
    print(
        f"{cname:6s}: "
        f"{int(audited_counts.get(idx, 0))}"
    )

### Save the Audit Artifacts

In [ ]:
# Save Master Reproducibility Audit

protocol_audit_path = (
    "/kaggle/working/"
    "master_ham10000_protocol_audit.csv"
)

membership_path = (
    "/kaggle/working/"
    "historical_test_membership_audit.csv"
)

protocol_audit_df.to_csv(
    protocol_audit_path,
    index=False
)

membership_summary.to_csv(
    membership_path,
    index=False
)

print("Saved:")
print(protocol_audit_path)
print(membership_path)

### Diagnostic Check of the Common Untouched Subset

Before extending the historical split audit to the remaining archived models,
the current intersection between the ResNet-50/ViT and ConvNeXt-Tiny historical
test partitions is examined for sample size and class representation.

This diagnostic does not change the reported results and does not retrain any
model. It determines whether a common strictly untouched evaluation subset is
practical as a no-retraining option for the revised analysis.

In [ ]:
# ResNet-50 / ViT + ConvNeXt-Tiny

# Start from the full HAM10000 metadata
common_subset_df = pd.read_csv(META_PATH)

# Canonical alphabetical class mapping
CANONICAL_CLASSES = [
    "akiec",
    "bcc",
    "bkl",
    "df",
    "mel",
    "nv",
    "vasc",
]

canonical_label_map = {
    name: i
    for i, name in enumerate(CANONICAL_CLASSES)
}

common_subset_df["label"] = (
    common_subset_df["dx"]
    .map(canonical_label_map)
    .astype(int)
)

common_subset_df["image_id"] = (
    common_subset_df["image_id"].astype(str)
)

# Keep only images that were test images in BOTH historical pipelines
common_subset_df = (
    common_subset_df[
        common_subset_df["image_id"].isin(
            common_test_ids_rv_convnext
        )
    ]
    .reset_index(drop=True)
)

# Verify membership
assert len(common_subset_df) == len(
    common_test_ids_rv_convnext
)

assert common_subset_df["image_id"].nunique() == len(
    common_test_ids_rv_convnext
)

# Class distribution
class_counts = (
    common_subset_df["label"]
    .value_counts()
    .sort_index()
)

print("Common untouched subset size:")
print(len(common_subset_df))

print("\nClass distribution:")
for idx, cname in enumerate(CANONICAL_CLASSES):
    count = int(class_counts.get(idx, 0))
    print(f"{idx}  {cname:6s}: {count}")

print("\nNumber of represented classes:")
print((class_counts > 0).sum(), "of", len(CANONICAL_CLASSES))

# Leakage verification
common_ids = set(
    common_subset_df["image_id"]
)

assert len(common_ids & rv_train_ids) == 0
assert len(common_ids & rv_val_ids) == 0
assert len(common_ids & conv_train_ids) == 0
assert len(common_ids & conv_val_ids) == 0

print("\nLeakage verification passed.")

# Save audit subset
audit_path = (
    "/kaggle/working/"
    "common_untouched_resnet_vit_convnext.csv"
)

common_subset_df[
    ["image_id", "dx", "label"]
].to_csv(
    audit_path,
    index=False
)

print("\nSaved audit subset to:")
print(audit_path)

### Preserve Historical ConvNeXt-Tiny Split Membership

The reconstructed training, validation, and test memberships are retained for
subsequent cross-model split auditing. Within the current 2,003-image
evaluation set, 1,005 images belong to the original ConvNeXt-Tiny test
partition and therefore constitute the strictly untouched test subset for
this archived model.

In [ ]:
# Preserve ConvNeXt Historical Split Membership

convnext_split_membership = {
    "train": conv_train_ids,
    "val": conv_val_ids,
    "test": conv_test_ids,
}

print("ConvNeXt historical split membership preserved.")
print("Train:", len(convnext_split_membership["train"]))
print("Validation:", len(convnext_split_membership["val"]))
print("Test:", len(convnext_split_membership["test"]))

# Strictly untouched ConvNeXt test images within canonical N=2,003
convnext_strict_test_ids = canonical_ids & conv_test_ids

print(
    "\nStrict ConvNeXt test images available inside canonical set:",
    len(convnext_strict_test_ids)
)

assert len(convnext_strict_test_ids) == 1005
assert len(canonical_ids & conv_train_ids) == 0

### Reload Best & Evaluate (VAL Sanity then TEST)

In [ ]:
# Reload to be extra sure we use the saved best
convnext_loaded = convnext_tiny(weights=None)
convnext_loaded.classifier[2] = nn.Linear(convnext_loaded.classifier[2].in_features, len(class_names))
convnext_loaded.load_state_dict(torch.load("/kaggle/input/datasets/nhsaykat/ham10000-convnext-tiny-weights/convnext_tiny_ham10000.pth", map_location=device))
convnext_loaded = convnext_loaded.to(device)
convnext_loaded.eval()

def evaluate(model, loader, device, name="SPLIT"):
    all_preds, all_labels, all_probs = [], [], []
    model.eval()
    with torch.no_grad():
        for x, y in tqdm(loader, desc=f"Testing ({name})"):
            x, y = x.to(device), y.to(device)
            logits = model(x)
            probs  = torch.softmax(logits, dim=1)
            preds  = probs.argmax(1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs  = np.array(all_probs)

    print(f"\n{name} — Classification Report:\n")
    print(classification_report(all_labels, all_preds, target_names=class_names))

    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average="weighted")
    try:
        auc_score = roc_auc_score(all_labels, all_probs, multi_class="ovr")
    except ValueError:
        auc_score = "N/A (needs >1 class present)"
    print(f"{name} — Acc: {acc:.4f} | F1 (weighted): {f1:.4f} | AUC (OvR): {auc_score}")

    # Confusion Matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.title(f"Confusion Matrix — ConvNeXt‑Tiny ({name})")
    plt.tight_layout()
    out_png = f"/kaggle/working/confusion_matrix_convnext_tiny_{name.lower()}.png"
    plt.savefig(out_png, dpi=300)
    plt.show()
    print(f"Saved: {out_png}")

    return acc, f1, auc_score, all_labels, all_probs

# Final numbers on test
acc, f1, auc_score, all_labels, all_probs = evaluate(convnext_loaded, test_loader, device, name="TEST")

### ROC‑AUC Curves — ConvNeXt‑Tiny

In [ ]:
plot_multiclass_roc(
    all_labels=all_labels,
    all_probs=all_probs,
    class_names=class_names,
    title="ROC Curves — ConvNeXt‑Tiny",
    outfile="/kaggle/working/roc_curves_convnext_tiny.png"
)

### Grad‑CAM Generation for Test Images — ConvNeXt‑Tiny (Plot + Save, with Fallback)

In [ ]:
# Pick objects / sanity checks
convnext_model = globals().get("convnext_loaded", None) or globals().get("convnext", None)
assert convnext_model is not None, "ConvNeXt‑Tiny model not found (convnext_loaded or convnext)."
test_loader_convnext = globals().get("test_loader_convnext", None) or globals().get("test_loader", None)
assert test_loader_convnext is not None, "Test loader not found (test_loader_convnext or test_loader)."
assert 'class_names' in globals(), "class_names not found."
assert 'device' in globals(), "device not found."

# Helpers
def inv_norm_05(img_t):
    """Invert Normalize([0.5]*3,[0.5]*3) -> RGB [0,1]. img_t: CHW tensor."""
    x = img_t.detach().cpu().clone()
    for c in range(3):
        x[c] = x[c] * 0.5 + 0.5
    return x.clamp(0, 1).permute(1, 2, 0).numpy()

def find_last_conv_module(net: nn.Module):
    last = None
    for m in net.modules():
        if isinstance(m, nn.Conv2d):
            last = m
    if last is None:
        raise RuntimeError("No Conv2d layer found in ConvNeXt‑Tiny.")
    return last

# Setup Grad‑CAM
convnext_model.eval()
target_layer = find_last_conv_module(convnext_model)  # safe default
cam = PTGradCAM(model=convnext_model, target_layers=[target_layer])

got = {c: False for c in class_names}
cam_results = {}

# Pass 1: prefer correctly predicted examples
for images, labels in test_loader_convnext:
    images, labels = images.to(device), labels.to(device)
    with torch.no_grad():
        preds = convnext_model(images).argmax(1)

    for i in range(images.size(0)):
        t_lbl = labels[i].item()
        p_lbl = preds[i].item()
        cname = class_names[t_lbl]
        if got[cname] or p_lbl != t_lbl:
            continue

        it = images[i].unsqueeze(0).requires_grad_(True)
        convnext_model.zero_grad()
        grayscale_cam = cam(input_tensor=it, targets=[ClassifierOutputTarget(p_lbl)])[0]

        rgb01 = inv_norm_05(images[i])
        cam_img = show_cam_on_image(rgb01, grayscale_cam, use_rgb=True)

        cam_results[cname] = (rgb01, cam_img)
        got[cname] = True

    if all(got.values()):
        break

# Pass 2: fallback for missing classes
if not all(got.values()):
    missing = [c for c, v in got.items() if not v]
    print(f"[Info] No correct predictions for classes: {missing}. Falling back to first available sample.")
    for images, labels in test_loader_convnext:
        images, labels = images.to(device), labels.to(device)
        for i in range(images.size(0)):
            t_lbl = labels[i].item()
            cname = class_names[t_lbl]
            if got[cname]:
                continue

            it = images[i].unsqueeze(0).requires_grad_(True)
            convnext_model.zero_grad()
            # drive CAM toward the TRUE label even if misclassified
            grayscale_cam = cam(input_tensor=it, targets=[ClassifierOutputTarget(t_lbl)])[0]

            rgb01 = inv_norm_05(images[i])
            cam_img = show_cam_on_image(rgb01, grayscale_cam, use_rgb=True)

            cam_results[cname] = (rgb01, cam_img)
            got[cname] = True

        if all(got.values()):
            break

# Plot & save
out_dir = Path("/kaggle/working/gradcam_convnext_tiny_per_class")
out_dir.mkdir(parents=True, exist_ok=True)

if not cam_results:
    print("[Warning] No Grad‑CAM images generated for ConvNeXt‑Tiny.")
else:
    for cname, (orig_rgb01, cam_img) in cam_results.items():
        plt.figure(figsize=(8, 4))
        plt.subplot(1, 2, 1); plt.imshow(orig_rgb01); plt.title(f"Original — {cname}"); plt.axis('off')
        plt.subplot(1, 2, 2); plt.imshow(cam_img);     plt.title("Grad‑CAM");          plt.axis('off')
        plt.tight_layout()
        sp = out_dir / f"gradcam_{cname}.png"
        plt.savefig(sp, dpi=300, bbox_inches="tight", pad_inches=0.05)
        plt.show()
    print(f"Saved ConvNeXt‑Tiny Grad‑CAM images to: {out_dir.resolve()}")

# Cleanup hooks
try:
    cam.activations_and_grads.release()
except Exception:
    pass
del cam

## MobileNetV3

### MobileNetV3 — Load Metadata & Resolve Image Paths

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv")

img_dir_1 = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1"
img_dir_2 = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_2"

def resolve_path(img_id):
    p1 = os.path.join(img_dir_1, f"{img_id}.jpg")
    p2 = os.path.join(img_dir_2, f"{img_id}.jpg")
    return p1 if os.path.exists(p1) else p2

df["path"] = df["image_id"].apply(resolve_path)
df["label"] = df["dx"].map(label_map).astype(int)

### MobileNetV3 — Train/Val/Test Split

In [ ]:
df_temp, test_df = train_test_split(df, test_size=0.20, stratify=df["label"], random_state=42)
train_df, val_df = train_test_split(df_temp, test_size=0.20, stratify=df_temp["label"], random_state=42)

print(len(train_df), len(val_df), len(test_df))

### MobileNetV3 — Transforms

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

### MobileNetV3 — Dataloaders

In [ ]:
train_dataset = SkinLesionDataset(train_df, transform=train_transform)
val_dataset   = SkinLesionDataset(val_df,   transform=val_transform)
test_dataset  = SkinLesionDataset(test_df,  transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

### Model Definition

In [ ]:
# Initialize MobileNetV3 large with pretrained weights
mobilenet = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1)

# Modify final classifier to match HAM10000 number of classes (7)
mobilenet.classifier[3] = nn.Linear(mobilenet.classifier[3].in_features, 7)

mobilenet = mobilenet.to(device)

# Define loss & optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(mobilenet.parameters(), lr=1e-4)

### MobileNetV3 — Train (Early Stopping, Save Best)

In [ ]:
def train_mobilenet(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=50, patience=5):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    epochs_no_improve = 0

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        model.train()
        run_loss, run_correct, run_total = 0.0, 0, 0

        for inputs, labels in tqdm(train_loader, desc="Training", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            run_loss += loss.item() * inputs.size(0)
            run_correct += (outputs.argmax(1) == labels).sum().item()
            run_total += labels.size(0)

        train_loss = run_loss / run_total
        train_acc  = run_correct / run_total

        # Validation
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc="Validating", leave=False):
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                val_correct += (outputs.argmax(1) == labels).sum().item()
                val_total   += labels.size(0)

        val_acc = val_correct / val_total
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val   Acc: {val_acc:.4f}")

        # Early stopping
        if val_acc > best_acc:
            best_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(best_model_wts, "/kaggle/working/mobilenetv3_ham10000.pth")
            print("Best MobileNetV3 model saved.")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("Early stopping triggered.")
                break

    return model, best_model_wts

mobilenet, best_model_wts = train_mobilenet(
    mobilenet, train_loader, val_loader, criterion, optimizer, device,
    num_epochs=50, patience=5
)
# Save final best
mobilenet.load_state_dict(best_model_wts)
torch.save(mobilenet.state_dict(), "/kaggle/working/mobilenetv3_ham10000.pth")
print("MobileNetV3 model saved to /kaggle/working/mobilenetv3_ham10000.pth")

### Reload the Model — MobileNetV3

In [ ]:
mobilenet = mobilenet_v3_large(weights=None)
mobilenet.classifier[3] = nn.Linear(mobilenet.classifier[3].in_features, len(class_names))
mobilenet.load_state_dict(torch.load("/kaggle/input/datasets/nhsaykat/mobilenetv3-ham10000-model-weights/mobilenetv3_ham10000.pth", map_location=device))
mobilenet = mobilenet.to(device)
mobilenet.eval()

### Evaluate on Test Set — Metrics & Confusion Matrix (MobileNetV3)

In [ ]:
all_preds_mnv3, all_labels_mnv3, all_probs_mnv3 = [], [], []

mobilenet.eval()
with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="Testing"):
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = mobilenet(inputs)
        probs = torch.softmax(outputs, dim=1)

        all_preds_mnv3.extend(probs.argmax(1).cpu().numpy())
        all_labels_mnv3.extend(labels.cpu().numpy())
        all_probs_mnv3.extend(probs.cpu().numpy())

# Convert to arrays
all_preds_mnv3  = np.array(all_preds_mnv3)
all_labels_mnv3 = np.array(all_labels_mnv3)
all_probs_mnv3  = np.array(all_probs_mnv3)

# Metrics
acc = accuracy_score(all_labels_mnv3, all_preds_mnv3)
f1  = f1_score(all_labels_mnv3, all_preds_mnv3, average="weighted")
try:
    auc_score = roc_auc_score(all_labels_mnv3, all_probs_mnv3, multi_class="ovr")
except ValueError:
    auc_score = "N/A (AUC needs >1 class in test set)"

# Report
print("\nClassification Report:\n")
print(classification_report(all_labels_mnv3, all_preds_mnv3, target_names=class_names))
print(f"Test Accuracy: {acc:.4f}")
print(f"Weighted F1 Score: {f1:.4f}")
print(f"ROC AUC Score: {auc_score}")

# Confusion Matrix
cm = confusion_matrix(all_labels_mnv3, all_preds_mnv3)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix — MobileNetV3")
plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrix_mobilenetv3.png", dpi=300)
plt.show()
print("Confusion matrix saved to 'confusion_matrix_mobilenetv3.png'.")

### ROC‑AUC Curves — MobileNetV3

In [ ]:
plot_multiclass_roc(
    all_labels=np.array(all_labels_mnv3),
    all_probs=np.array(all_probs_mnv3),
    class_names=class_names,
    title="ROC Curves — MobileNetV3",
    outfile="/kaggle/working/roc_curves_mobilenetv3.png"
)

### Grad‑CAM Generation for Test Images — MobileNetV3‑Large (Plot + Save, with Fallback)

In [ ]:
# Pick objects / sanity checks
mobilenet_model = globals().get("mobilenet", None)
assert mobilenet_model is not None, "MobileNetV3 model not found (variable 'mobilenet')."
test_loader_mnv3 = globals().get("test_loader_mnv3", None) or globals().get("test_loader", None)
assert test_loader_mnv3 is not None, "Test loader not found (test_loader_mnv3 or test_loader)."
assert 'class_names' in globals(), "class_names not found."
assert 'device' in globals(), "device not found."

# Helpers
def inv_norm_05(img_t):
    """Invert Normalize([0.5]*3,[0.5]*3) -> RGB [0,1]. img_t: CHW tensor."""
    x = img_t.detach().cpu().clone()
    for c in range(3):
        x[c] = x[c] * 0.5 + 0.5
    return x.clamp(0, 1).permute(1, 2, 0).numpy()

def find_last_conv_module(net: nn.Module):
    last = None
    for m in net.modules():
        if isinstance(m, nn.Conv2d):
            last = m
    if last is None:
        raise RuntimeError("No Conv2d layer found in MobileNetV3‑Large.")
    return last

# Setup Grad‑CAM
mobilenet_model.eval()
target_layer = find_last_conv_module(mobilenet_model)  # last conv in features
cam = PTGradCAM(model=mobilenet_model, target_layers=[target_layer])

got = {c: False for c in class_names}
cam_results = {}

# Pass 1: prefer correctly predicted examples
for images, labels in test_loader_mnv3:
    images, labels = images.to(device), labels.to(device)
    with torch.no_grad():
        preds = mobilenet_model(images).argmax(1)

    for i in range(images.size(0)):
        t_lbl = labels[i].item()
        p_lbl = preds[i].item()
        cname = class_names[t_lbl]
        if got[cname] or p_lbl != t_lbl:
            continue

        it = images[i].unsqueeze(0).requires_grad_(True)
        mobilenet_model.zero_grad()
        grayscale_cam = cam(input_tensor=it, targets=[ClassifierOutputTarget(p_lbl)])[0]

        rgb01 = inv_norm_05(images[i])
        cam_img = show_cam_on_image(rgb01, grayscale_cam, use_rgb=True)

        cam_results[cname] = (rgb01, cam_img)
        got[cname] = True

    if all(got.values()):
        break

# Pass 2: fallback for missing classes
if not all(got.values()):
    missing = [c for c, v in got.items() if not v]
    print(f"[Info] No correct predictions for classes: {missing}. Falling back to first available sample.")
    for images, labels in test_loader_mnv3:
        images, labels = images.to(device), labels.to(device)
        for i in range(images.size(0)):
            t_lbl = labels[i].item()
            cname = class_names[t_lbl]
            if got[cname]:
                continue

            it = images[i].unsqueeze(0).requires_grad_(True)
            mobilenet_model.zero_grad()
            # drive CAM toward the TRUE label even if misclassified
            grayscale_cam = cam(input_tensor=it, targets=[ClassifierOutputTarget(t_lbl)])[0]

            rgb01 = inv_norm_05(images[i])
            cam_img = show_cam_on_image(rgb01, grayscale_cam, use_rgb=True)

            cam_results[cname] = (rgb01, cam_img)
            got[cname] = True

        if all(got.values()):
            break

# Plot & save
out_dir = Path("/kaggle/working/gradcam_mobilenet_v3_per_class")
out_dir.mkdir(parents=True, exist_ok=True)

if not cam_results:
    print("[Warning] No Grad‑CAM images generated for MobileNetV3‑Large.")
else:
    for cname, (orig_rgb01, cam_img) in cam_results.items():
        plt.figure(figsize=(8, 4))
        plt.subplot(1, 2, 1); plt.imshow(orig_rgb01); plt.title(f"Original — {cname}"); plt.axis('off')
        plt.subplot(1, 2, 2); plt.imshow(cam_img);     plt.title("Grad‑CAM");          plt.axis('off')
        plt.tight_layout()
        sp = out_dir / f"gradcam_{cname}.png"
        plt.savefig(sp, dpi=300, bbox_inches="tight", pad_inches=0.05)
        plt.show()
    print(f"Saved MobileNetV3‑Large Grad‑CAM images to: {out_dir.resolve()}")

# Cleanup hooks
try:
    cam.activations_and_grads.release()
except Exception:
    pass
del cam

## 3.1 Grad-CAM Explainability

### Define ViT Wrapper for Grad-CAM Compatibility

In [ ]:
class WrappedViT(nn.Module):
    def __init__(self, vit_model):
        super().__init__()
        self.vit = vit_model.vit
        self.classifier = vit_model.classifier

    def forward(self, x):
        outputs = self.vit(pixel_values=x)
        logits = self.classifier(outputs.last_hidden_state[:, 0])
        return logits

### Load Archived Model Checkpoints

In [ ]:
# Load Archived HAM10000 Model Checkpoints

# Dependency checks
required = [
    "CNNModel",
    "canonical_test_df",
    "RESNET_VIT_TO_CANONICAL",
    "vit_model",
    "device",
    "torch",
    "nn",
    "models",
    "densenet121",
    "efficientnet_b3",
    "convnext_tiny",
    "mobilenet_v3_large",
]

missing = [
    name for name in required
    if name not in globals()
]

print("Dependency check:")
for name in required:
    print(
        f"{name:30s}: "
        f"{'OK' if name in globals() else 'MISSING'}"
    )

assert not missing, (
    "Missing required definitions before checkpoint loading: "
    + ", ".join(missing)
)

# Canonical evaluation safeguards
assert len(canonical_test_df) == 2003, \
    f"Expected 2003 canonical test images, found {len(canonical_test_df)}"

assert canonical_test_df["image_id"].nunique() == 2003, \
    "Duplicate image IDs detected in canonical test set."

assert RESNET_VIT_TO_CANONICAL == [6, 5, 0, 2, 3, 1, 4], \
    f"Unexpected ResNet/ViT mapping: {RESNET_VIT_TO_CANONICAL}"

print("\nCanonical evaluation dependencies verified.")

# Archived checkpoint paths
CNN_PATH = (
    "/kaggle/input/datasets/nhsaykat/"
    "kaggleinputham10000-model-weights/"
    "cnn_ham10000_best_model.pth"
)

RESNET_PATH = (
    "/kaggle/input/datasets/nhsaykat/"
    "resnet50-ham10000-model-weights/"
    "resnet50_ham10000.pth"
)

DENSENET_PATH = (
    "/kaggle/input/datasets/nhsaykat/"
    "densenet121-ham10000-model-weights/"
    "densenet121_ham10000.pth"
)

EFFNET_PATH = (
    "/kaggle/input/datasets/nhsaykat/"
    "efficientnetb3-ham10000-model-weights/"
    "efficientnet_b3_best.pth"
)

CONVNEXT_PATH = (
    "/kaggle/input/datasets/nhsaykat/"
    "ham10000-convnext-tiny-weights/"
    "convnext_tiny_ham10000.pth"
)

MOBILENET_PATH = (
    "/kaggle/input/datasets/nhsaykat/"
    "mobilenetv3-ham10000-model-weights/"
    "mobilenetv3_ham10000.pth"
)

checkpoint_paths = {
    "CNN": CNN_PATH,
    "ResNet-50": RESNET_PATH,
    "DenseNet-121": DENSENET_PATH,
    "EfficientNet-B3": EFFNET_PATH,
    "ConvNeXt-Tiny": CONVNEXT_PATH,
    "MobileNetV3-Large": MOBILENET_PATH,
}

print("\nCheckpoint check:")
for name, path in checkpoint_paths.items():
    exists = os.path.exists(path)
    print(f"{name:22s}: {'FOUND' if exists else 'MISSING'}")
    if not exists:
        print("  ", path)

missing_checkpoints = [
    name
    for name, path in checkpoint_paths.items()
    if not os.path.exists(path)
]

assert not missing_checkpoints, (
    "Missing checkpoint files: "
    + ", ".join(missing_checkpoints)
)

# Load CNN
cnn_model = CNNModel(num_classes=7)

cnn_model.load_state_dict(
    torch.load(
        CNN_PATH,
        map_location=device,
        weights_only=True
    )
)

cnn_model = cnn_model.to(device)
cnn_model.eval()

# Load ResNet-50
resnet_model = models.resnet50(weights=None)

resnet_model.fc = nn.Linear(
    resnet_model.fc.in_features,
    7
)

resnet_model.load_state_dict(
    torch.load(
        RESNET_PATH,
        map_location=device,
        weights_only=True
    )
)

resnet_model = resnet_model.to(device)
resnet_model.eval()

# Load DenseNet-121
densenet_model = densenet121(weights=None)

densenet_model.classifier = nn.Linear(
    densenet_model.classifier.in_features,
    7
)

densenet_model.load_state_dict(
    torch.load(
        DENSENET_PATH,
        map_location=device,
        weights_only=True
    )
)

densenet_model = densenet_model.to(device)
densenet_model.eval()

# Load EfficientNet-B3
effnet_model = efficientnet_b3(weights=None)

effnet_model.classifier[1] = nn.Linear(
    effnet_model.classifier[1].in_features,
    7
)

effnet_model.load_state_dict(
    torch.load(
        EFFNET_PATH,
        map_location=device,
        weights_only=True
    )
)

effnet_model = effnet_model.to(device)
effnet_model.eval()

# Load ConvNeXt-Tiny
convnext_model = convnext_tiny(weights=None)

convnext_model.classifier[2] = nn.Linear(
    convnext_model.classifier[2].in_features,
    7
)

convnext_model.load_state_dict(
    torch.load(
        CONVNEXT_PATH,
        map_location=device,
        weights_only=True
    )
)

convnext_model = convnext_model.to(device)
convnext_model.eval()

# Load MobileNetV3-Large
mobilenet_model = mobilenet_v3_large(weights=None)

mobilenet_model.classifier[3] = nn.Linear(
    mobilenet_model.classifier[3].in_features,
    7
)

mobilenet_model.load_state_dict(
    torch.load(
        MOBILENET_PATH,
        map_location=device,
        weights_only=True
    )
)

mobilenet_model = mobilenet_model.to(device)
mobilenet_model.eval()

# Final verification
models_loaded = {
    "CNN": cnn_model,
    "ResNet-50": resnet_model,
    "DenseNet-121": densenet_model,
    "EfficientNet-B3": effnet_model,
    "ConvNeXt-Tiny": convnext_model,
    "MobileNetV3-Large": mobilenet_model,
    "ViT": vit_model,
}

print("\nArchived models ready for canonical evaluation:")
for name, model in models_loaded.items():
    print(
        f"{name:22s}: "
        f"{'eval' if not model.training else 'TRAIN MODE'}"
    )

assert all(
    not model.training
    for model in models_loaded.values()
)

print("\nAll seven archived models are loaded and in evaluation mode.")

### Unified Canonical Inference

In [ ]:
# Unified Canonical Inference — HAM10000, N = 2,003

# Safety checks
required_preprocessing = [
    "preprocess_224_05",
    "preprocess_resnet",
    "preprocess_effnet",
    "preprocess_convnext",
    "vit_processor",
]

missing_preprocessing = [
    name for name in required_preprocessing
    if name not in globals()
]

assert not missing_preprocessing, (
    "Missing preprocessing definitions: "
    + ", ".join(missing_preprocessing)
)

assert len(canonical_test_df) == 2003
assert canonical_test_df["image_id"].nunique() == 2003
assert RESNET_VIT_TO_CANONICAL == [6, 5, 0, 2, 3, 1, 4]

model_keys = [
    "cnn",
    "resnet50",
    "densenet121",
    "efficientnet_b3",
    "convnext_tiny",
    "mobilenet_v3",
    "vit",
]

per_model_probs = {
    k: [] for k in model_keys
}

labels = []

# Per-image canonical inference
@torch.no_grad()
def predict_all_canonical(pil_img):

    outputs = {}

    # CNN
    x = preprocess_224_05(pil_img).unsqueeze(0).to(device)

    outputs["cnn"] = torch.softmax(
        cnn_model(x),
        dim=1
    ).cpu().numpy()[0]

    # ResNet-50
    x = preprocess_resnet(pil_img).unsqueeze(0).to(device)

    p = torch.softmax(
        resnet_model(x),
        dim=1
    ).cpu().numpy()[0]

    # Original ResNet training order -> canonical order
    outputs["resnet50"] = p[
        RESNET_VIT_TO_CANONICAL
    ]

    # DenseNet-121
    x = preprocess_224_05(pil_img).unsqueeze(0).to(device)

    outputs["densenet121"] = torch.softmax(
        densenet_model(x),
        dim=1
    ).cpu().numpy()[0]

    # EfficientNet-B3
    x = preprocess_effnet(pil_img).unsqueeze(0).to(device)

    outputs["efficientnet_b3"] = torch.softmax(
        effnet_model(x),
        dim=1
    ).cpu().numpy()[0]

    # ConvNeXt-Tiny
    x = preprocess_convnext(pil_img).unsqueeze(0).to(device)

    outputs["convnext_tiny"] = torch.softmax(
        convnext_model(x),
        dim=1
    ).cpu().numpy()[0]

    # MobileNetV3-Large
    x = preprocess_224_05(pil_img).unsqueeze(0).to(device)

    outputs["mobilenet_v3"] = torch.softmax(
        mobilenet_model(x),
        dim=1
    ).cpu().numpy()[0]

    # ViT
    enc = vit_processor(
        images=pil_img,
        return_tensors="pt"
    )

    pixel_values = enc["pixel_values"].to(device)

    p = torch.softmax(
        vit_model(
            pixel_values=pixel_values
        ).logits,
        dim=1
    ).cpu().numpy()[0]

    # Original ViT training order -> canonical order
    outputs["vit"] = p[
        RESNET_VIT_TO_CANONICAL
    ]

    return outputs

# Run inference over the same ordered 2,003 images
for _, row in tqdm(
    canonical_test_df.iterrows(),
    total=len(canonical_test_df),
    desc="Canonical inference"
):

    path = row["path"]

    assert isinstance(path, str) and os.path.exists(path), \
        f"Missing image path: {path}"

    img = Image.open(path).convert("RGB")

    out = predict_all_canonical(img)

    for k in model_keys:
        per_model_probs[k].append(out[k])

    labels.append(int(row["label"]))

# Convert outputs to arrays
labels = np.asarray(labels, dtype=np.int64)

for k in model_keys:
    per_model_probs[k] = np.asarray(
        per_model_probs[k],
        dtype=np.float32
    )

# Final safety checks
assert labels.shape == (2003,)

for k in model_keys:

    assert per_model_probs[k].shape == (2003, 7), \
        f"{k}: unexpected shape {per_model_probs[k].shape}"

    assert np.isfinite(per_model_probs[k]).all(), \
        f"{k}: non-finite probabilities detected"

    assert np.allclose(
        per_model_probs[k].sum(axis=1),
        1.0,
        atol=1e-5
    ), f"{k}: probabilities do not sum to one"

# Critical ViT reproducibility check
vit_check = per_model_probs["vit"]
vit_pred_check = vit_check.argmax(axis=1)

vit_check_acc = accuracy_score(
    labels,
    vit_pred_check
)

vit_check_f1 = f1_score(
    labels,
    vit_pred_check,
    average="weighted"
)

print("\nViT verification:")
print(f"Accuracy:    {vit_check_acc:.4f}")
print(f"Weighted F1: {vit_check_f1:.4f}")

assert np.isclose(vit_check_acc, 0.9591, atol=5e-5), \
    f"ViT accuracy changed unexpectedly: {vit_check_acc:.6f}"

assert np.isclose(vit_check_f1, 0.9585, atol=5e-5), \
    f"ViT weighted F1 changed unexpectedly: {vit_check_f1:.6f}"

# Preserve canonical prediction arrays
np.save(
    "/kaggle/working/canonical_ham10000_labels.npy",
    labels
)

for k in model_keys:
    np.save(
        f"/kaggle/working/canonical_{k}_probs.npy",
        per_model_probs[k]
    )

print("\nUnified canonical inference completed successfully.")
print("Canonical prediction arrays saved.")

### Canonical Model Comparison and Mean Ensemble

In [ ]:
# Canonical Model Comparison and Mean Ensemble

def canonical_metrics(y_true, probs):

    preds = probs.argmax(axis=1)

    return {
        "Accuracy": accuracy_score(
            y_true,
            preds
        ),
        "Weighted F1": f1_score(
            y_true,
            preds,
            average="weighted"
        ),
        "Macro AUC": roc_auc_score(
            y_true,
            probs,
            multi_class="ovr",
            average="macro"
        ),
        "Micro AUC": roc_auc_score(
            y_true,
            probs,
            multi_class="ovr",
            average="micro"
        ),
    }

# Equal-weight probability ensemble of all seven models
stacked_probs = np.stack(
    [per_model_probs[k] for k in model_keys],
    axis=0
)

assert stacked_probs.shape == (7, 2003, 7)

ensemble_probs = stacked_probs.mean(axis=0)

assert ensemble_probs.shape == (2003, 7)

assert np.isfinite(ensemble_probs).all()

assert np.allclose(
    ensemble_probs.sum(axis=1),
    1.0,
    atol=1e-5
)

# Preserve ensemble probabilities
np.save(
    "/kaggle/working/canonical_ensemble_probs.npy",
    ensemble_probs
)

# Individual model results
display_names = {
    "cnn": "CNN",
    "resnet50": "ResNet-50",
    "densenet121": "DenseNet-121",
    "efficientnet_b3": "EfficientNet-B3",
    "convnext_tiny": "ConvNeXt-Tiny",
    "mobilenet_v3": "MobileNetV3-L",
    "vit": "ViT",
}

rows = []

for k in model_keys:

    metrics = canonical_metrics(
        labels,
        per_model_probs[k]
    )

    rows.append({
        "Model": display_names[k],
        **metrics
    })

# Ensemble result
ensemble_metrics = canonical_metrics(
    labels,
    ensemble_probs
)

rows.append({
    "Model": "Ensemble",
    **ensemble_metrics
})

comparison_df = pd.DataFrame(rows)

# Keep the scientifically meaningful order
comparison_df["Model"] = pd.Categorical(
    comparison_df["Model"],
    categories=[
        "CNN",
        "ResNet-50",
        "DenseNet-121",
        "EfficientNet-B3",
        "ConvNeXt-Tiny",
        "MobileNetV3-L",
        "ViT",
        "Ensemble",
    ],
    ordered=True
)

comparison_df = comparison_df.sort_values(
    "Model"
).reset_index(drop=True)

print("\nCanonical HAM10000 Results — N = 2,003")
display(
    comparison_df.round(4)
)

# Save canonical results
comparison_df.to_csv(
    "/kaggle/working/"
    "canonical_ham10000_model_comparison.csv",
    index=False
)

print(
    "\nCanonical model comparison saved to:"
    "\n/kaggle/working/"
    "canonical_ham10000_model_comparison.csv"
)

# Compare canonical ensemble with submitted Table 1
print("\nCanonical ensemble:")
print(
    f"Accuracy:    {ensemble_metrics['Accuracy']:.4f}"
)
print(
    f"Weighted F1: {ensemble_metrics['Weighted F1']:.4f}"
)
print(
    f"Macro AUC:   {ensemble_metrics['Macro AUC']:.4f}"
)
print(
    f"Micro AUC:   {ensemble_metrics['Micro AUC']:.4f}"
)

print("\nSubmitted Table 1 ensemble:")
print("Accuracy:    0.9246")
print("Weighted F1: 0.9215")
print("Macro AUC:   0.9944")
print("Micro AUC:   0.9961")

## Revised Final Grad-CAM Analysis
### Standardized HAM10000 Evaluation Cohort

In [ ]:
# ============================================================
# Revised Grad-CAM Configuration and Safety Checks
# Standardized HAM10000 Evaluation Cohort

# Canonical class definitions
GRADCAM_CLASS_NAMES = [
    "akiec",
    "bcc",
    "bkl",
    "df",
    "mel",
    "nv",
    "vasc",
]

GRADCAM_CLASS_TO_IDX = {
    name: idx
    for idx, name in enumerate(GRADCAM_CLASS_NAMES)
}

# Required objects from finalized HAM10000 analysis
required_gradcam_objects = [
    "canonical_test_df",
    "labels",
    "per_model_probs",
    "ensemble_probs",
    "cnn_model",
    "resnet_model",
    "densenet_model",
    "effnet_model",
    "convnext_model",
    "mobilenet_model",
    "vit_model",
    "preprocess_224_05",
    "preprocess_resnet",
    "preprocess_effnet",
    "preprocess_convnext",
    "vit_processor",
    "RESNET_VIT_TO_CANONICAL",
    "device",
]

missing_gradcam_objects = [
    name
    for name in required_gradcam_objects
    if name not in globals()
]

assert not missing_gradcam_objects, (
    "Missing required objects before revised Grad-CAM analysis: "
    + ", ".join(missing_gradcam_objects)
)

# Cohort safeguards
assert len(canonical_test_df) == 2003, \
    f"Expected 2003 images, found {len(canonical_test_df)}"

assert canonical_test_df["image_id"].nunique() == 2003, \
    "Duplicate image IDs found in standardized cohort."

assert labels.shape == (2003,), \
    f"Unexpected label array shape: {labels.shape}"

assert ensemble_probs.shape == (2003, 7), \
    f"Unexpected ensemble probability shape: {ensemble_probs.shape}"

assert RESNET_VIT_TO_CANONICAL == [6, 5, 0, 2, 3, 1, 4], \
    f"Unexpected ResNet/ViT mapping: {RESNET_VIT_TO_CANONICAL}"

# Verify canonical labels
assert sorted(np.unique(labels).tolist()) == list(range(7)), \
    f"Unexpected labels: {np.unique(labels)}"

# Verify all model probability arrays
expected_probability_keys = [
    "cnn",
    "resnet50",
    "densenet121",
    "efficientnet_b3",
    "convnext_tiny",
    "mobilenet_v3",
    "vit",
]

for model_key in expected_probability_keys:

    assert model_key in per_model_probs, \
        f"Missing probabilities for {model_key}"

    probs = np.asarray(
        per_model_probs[model_key]
    )

    assert probs.shape == (2003, 7), \
        f"{model_key}: unexpected shape {probs.shape}"

    assert np.isfinite(probs).all(), \
        f"{model_key}: non-finite values detected."

    assert np.allclose(
        probs.sum(axis=1),
        1.0,
        atol=1e-5
    ), \
        f"{model_key}: probabilities do not sum to 1."

# Verify all models are in evaluation mode
gradcam_models = {
    "CNN": cnn_model,
    "ResNet-50": resnet_model,
    "DenseNet-121": densenet_model,
    "EfficientNet-B3": effnet_model,
    "ConvNeXt-Tiny": convnext_model,
    "MobileNetV3-L": mobilenet_model,
    "ViT": vit_model,
}

for model_name, model in gradcam_models.items():

    model.eval()

    assert not model.training, \
        f"{model_name} is not in evaluation mode."


print("PASS: Revised Grad-CAM prerequisites verified.")
print(f"Cohort size: {len(canonical_test_df)}")
print(f"Classes: {GRADCAM_CLASS_NAMES}")
print(f"Device: {device}")

### Model-Specific Grad-CAM Configuration

In [ ]:
# Model-Specific Grad-CAM Configuration

# Utility: automatically find final Conv2d layer
# Useful for custom CNN architecture

def find_last_conv2d_layer(model):

    conv_layers = [
        module
        for module in model.modules()
        if isinstance(module, nn.Conv2d)
    ]

    assert len(conv_layers) > 0, \
        "No Conv2d layer found."

    return conv_layers[-1]

# ViT wrapper
# Grad-CAM expects model output to be a tensor of logits

class ViTGradCAMWrapper(nn.Module):

    def __init__(self, vit_model):

        super().__init__()

        self.vit_model = vit_model

    def forward(self, x):

        return self.vit_model(
            pixel_values=x
        ).logits


vit_gradcam_model = ViTGradCAMWrapper(
    vit_model
).to(device)

vit_gradcam_model.eval()

# ViT reshape transform
#
# Converts:
# B x 197 x C
#
# into:
# B x C x 14 x 14
#
# while removing the CLS token.

def vit_reshape_transform(
    tensor,
    height=14,
    width=14
):

    # Remove CLS token
    result = tensor[:, 1:, :]

    expected_tokens = height * width

    assert result.shape[1] == expected_tokens, (
        f"Expected {expected_tokens} ViT patch tokens, "
        f"found {result.shape[1]}."
    )

    result = result.reshape(
        result.size(0),
        height,
        width,
        result.size(2)
    )

    result = result.transpose(
        2,
        3
    ).transpose(
        1,
        2
    )

    return result

# Correct target layers
gradcam_target_layers = {

    "CNN":
        find_last_conv2d_layer(
            cnn_model
        ),

    "ResNet-50":
        resnet_model.layer4[-1],

    "DenseNet-121":
    densenet_model.features.denseblock4.denselayer16.conv2,

    "EfficientNet-B3":
        effnet_model.features[-1],

    "ConvNeXt-Tiny":
        convnext_model.features[-1],

    "MobileNetV3-L":
        mobilenet_model.features[-1],

    "ViT":
        vit_model.vit.encoder.layer[-1].layernorm_before,
}

# Canonical target -> native model target
#
# ResNet and ViT were trained using a different output ordering.
# For those two models:
#
# canonical output j corresponds to native output
# RESNET_VIT_TO_CANONICAL[j].

def canonical_to_native_target(
    model_name,
    canonical_class_idx
):

    canonical_class_idx = int(
        canonical_class_idx
    )

    assert 0 <= canonical_class_idx < 7

    if model_name in [
        "ResNet-50",
        "ViT",
    ]:

        return int(
            RESNET_VIT_TO_CANONICAL[
                canonical_class_idx
            ]
        )

    return canonical_class_idx

# Verify target mapping
print("Canonical -> native target indices:")

for class_idx, class_name in enumerate(
    GRADCAM_CLASS_NAMES
):

    print(
        f"{class_name:6s} | "
        f"canonical={class_idx} | "
        f"ResNet native="
        f"{canonical_to_native_target('ResNet-50', class_idx)} | "
        f"ViT native="
        f"{canonical_to_native_target('ViT', class_idx)}"
    )


print("\nTarget layers:")

for model_name, layer in gradcam_target_layers.items():

    print(
        f"{model_name:18s}: "
        f"{layer.__class__.__name__}"
    )

print("\nPASS: Grad-CAM model configuration created.")

### Revised Grad-CAM Helper Functions

In [ ]:
# Correct Model-Specific Grad-CAM Input Preparation
def prepare_gradcam_input(
    model_name,
    pil_img
):
    """
    Prepare one image using exactly the preprocessing associated
    with the finalized HAM10000 evaluation pipeline.
    """

    assert isinstance(
        pil_img,
        Image.Image
    )

    # CNN
    if model_name == "CNN":

        tensor = preprocess_224_05(
            pil_img
        )

    # ResNet-50
    elif model_name == "ResNet-50":

        tensor = preprocess_resnet(
            pil_img
        )

    # DenseNet-121
    elif model_name == "DenseNet-121":

        tensor = preprocess_224_05(
            pil_img
        )

    # EfficientNet-B3
    elif model_name == "EfficientNet-B3":

        tensor = preprocess_effnet(
            pil_img
        )

    # ConvNeXt-Tiny
    elif model_name == "ConvNeXt-Tiny":

        # IMPORTANT:
        # Revised pipeline uses ImageNet normalization,
        # NOT preprocess_224_05.
        tensor = preprocess_convnext(
            pil_img
        )

    # MobileNetV3-L
    elif model_name == "MobileNetV3-L":

        tensor = preprocess_224_05(
            pil_img
        )

    # ViT
    elif model_name == "ViT":

        encoded = vit_processor(
            images=pil_img,
            return_tensors="pt"
        )

        return encoded[
            "pixel_values"
        ].to(device)

    else:

        raise ValueError(
            f"Unknown model: {model_name}"
        )


    return (
        tensor
        .unsqueeze(0)
        .to(device)
    )


print(
    "PASS: Model-specific Grad-CAM preprocessing helper ready."
)

### Single-Model Grad-CAM Function

In [ ]:
# Single-Model Revised Grad-CAM Function
def normalize_cam(cam):

    cam = np.asarray(
        cam,
        dtype=np.float32
    )

    cam = cam - cam.min()

    max_value = cam.max()

    if max_value > 0:

        cam = cam / max_value

    return np.clip(
        cam,
        0.0,
        1.0
    )


def compute_single_model_gradcam(
    model_name,
    pil_img,
    canonical_target_idx
):
    """
    Generate Grad-CAM for one component model using a target
    specified in canonical HAM10000 class order.

    Returns a normalized 2D CAM.
    """

    assert model_name in gradcam_models, \
        f"Unknown model: {model_name}"

    canonical_target_idx = int(
        canonical_target_idx
    )

    native_target_idx = (
        canonical_to_native_target(
            model_name,
            canonical_target_idx
        )
    )

    input_tensor = (
        prepare_gradcam_input(
            model_name,
            pil_img
        )
    )


    # ViT requires wrapper + reshape transform
    if model_name == "ViT":

        cam_model = vit_gradcam_model

        cam = GradCAM(
            model=cam_model,
            target_layers=[
                gradcam_target_layers[
                    "ViT"
                ]
            ],
            reshape_transform=
                vit_reshape_transform
        )

    else:

        cam_model = gradcam_models[
            model_name
        ]

        cam = GradCAM(
            model=cam_model,
            target_layers=[
                gradcam_target_layers[
                    model_name
                ]
            ]
        )


    targets = [
        ClassifierOutputTarget(
            native_target_idx
        )
    ]


    grayscale_cam = cam(
        input_tensor=input_tensor,
        targets=targets
    )


    grayscale_cam = grayscale_cam[
        0
    ]


    grayscale_cam = normalize_cam(
        grayscale_cam
    )


    return grayscale_cam


print(
    "PASS: Single-model revised Grad-CAM function ready."
)

### Deterministic Representative Sample Selection

In [ ]:
# Deterministic Representative Sample Selection
# One image per lesion class

ensemble_predictions = (
    ensemble_probs.argmax(
        axis=1
    )
)

ensemble_confidences = (
    ensemble_probs.max(
        axis=1
    )
)


selection_df = (
    canonical_test_df
    .reset_index(drop=True)
    .copy()
)

assert len(selection_df) == 2003

selection_df[
    "true_label"
] = labels

selection_df[
    "ensemble_prediction"
] = ensemble_predictions

selection_df[
    "ensemble_confidence"
] = ensemble_confidences


representative_rows = []


for class_idx, class_name in enumerate(
    GRADCAM_CLASS_NAMES
):

    candidates = selection_df[
        (
            selection_df["true_label"]
            ==
            class_idx
        )
        &
        (
            selection_df[
                "ensemble_prediction"
            ]
            ==
            class_idx
        )
    ].copy()


    assert len(candidates) > 0, \
        f"No correctly classified ensemble samples for {class_name}."


    median_confidence = (
        candidates[
            "ensemble_confidence"
        ].median()
    )


    candidates[
        "distance_from_class_median_confidence"
    ] = np.abs(
        candidates[
            "ensemble_confidence"
        ]
        -
        median_confidence
    )


    # Deterministic tie-breaking by image_id
    candidates = candidates.sort_values(
        by=[
            "distance_from_class_median_confidence",
            "image_id",
        ],
        ascending=[
            True,
            True,
        ]
    )


    chosen = candidates.iloc[
        0
    ].copy()


    representative_rows.append({

        "class_idx":
            class_idx,

        "class_name":
            class_name,

        "image_id":
            chosen["image_id"],

        "path":
            chosen["path"],

        "true_label":
            int(
                chosen["true_label"]
            ),

        "ensemble_prediction":
            int(
                chosen[
                    "ensemble_prediction"
                ]
            ),

        "ensemble_confidence":
            float(
                chosen[
                    "ensemble_confidence"
                ]
            ),

        "class_median_correct_confidence":
            float(
                median_confidence
            ),
    })


gradcam_representatives_df = pd.DataFrame(
    representative_rows
)


display(
    gradcam_representatives_df[
        [
            "class_name",
            "image_id",
            "ensemble_confidence",
            "class_median_correct_confidence",
        ]
    ]
)


# Save reproducibility manifest

GRADCAM_MANIFEST_CSV = (
    "/kaggle/working/"
    "HAM10000_gradcam_representative_samples_revised_final.csv"
)

gradcam_representatives_df.to_csv(
    GRADCAM_MANIFEST_CSV,
    index=False
)


print(
    "\nSaved representative-sample manifest:"
)

print(
    GRADCAM_MANIFEST_CSV
)

### Equal-Weight Ensemble Grad-CAM Aggregation

In [ ]:
# Equal-Weight Ensemble Grad-CAM Aggregation
GRADCAM_COMPONENT_MODELS = [
    "CNN",
    "ResNet-50",
    "DenseNet-121",
    "EfficientNet-B3",
    "ConvNeXt-Tiny",
    "MobileNetV3-L",
    "ViT",
]


def resize_cam_to_image(
    cam,
    width,
    height
):

    cam_resized = cv2.resize(
        cam.astype(np.float32),
        (width, height),
        interpolation=cv2.INTER_LINEAR
    )

    return normalize_cam(
        cam_resized
    )


def compute_ensemble_gradcam(
    pil_img,
    canonical_target_idx
):
    """
    Ensemble attribution = equal-weight mean of individually
    normalized Grad-CAM maps from all seven component models.

    This is an aggregated attribution visualization; it is not
    Grad-CAM applied directly to a single differentiable ensemble
    network.
    """

    component_cams = {}


    target_width = pil_img.width
    target_height = pil_img.height


    for model_name in GRADCAM_COMPONENT_MODELS:

        print(
            f"  Computing CAM: {model_name}"
        )


        cam = compute_single_model_gradcam(
            model_name=model_name,
            pil_img=pil_img,
            canonical_target_idx=
                canonical_target_idx
        )


        cam = resize_cam_to_image(
            cam,
            width=target_width,
            height=target_height
        )


        component_cams[
            model_name
        ] = cam


    stacked_cams = np.stack(
        [
            component_cams[
                model_name
            ]
            for model_name
            in GRADCAM_COMPONENT_MODELS
        ],
        axis=0
    )


    assert stacked_cams.shape[
        0
    ] == 7


    ensemble_cam = stacked_cams.mean(
        axis=0
    )


    ensemble_cam = normalize_cam(
        ensemble_cam
    )


    return (
        ensemble_cam,
        component_cams
    )


print(
    "PASS: Equal-weight ensemble Grad-CAM aggregation ready."
)

### Generate Revised Per-Class Ensemble Grad-CAM Figures

In [ ]:
# Generate Revised Per-Class Ensemble Grad-CAM Figures

GRADCAM_OUTPUT_DIR = (
    "/kaggle/working/"
    "gradcam_revised_final"
)

os.makedirs(
    GRADCAM_OUTPUT_DIR,
    exist_ok=True
)


generated_gradcam_files = []


for _, row in gradcam_representatives_df.iterrows():

    class_name = row[
        "class_name"
    ]

    class_idx = int(
        row[
            "class_idx"
        ]
    )

    image_id = row[
        "image_id"
    ]

    image_path = row[
        "path"
    ]


    print(
        "\n======================================"
    )

    print(
        f"Class: {class_name}"
    )

    print(
        f"Image: {image_id}"
    )


    assert os.path.exists(
        image_path
    ), \
        f"Missing image: {image_path}"


    pil_img = Image.open(
        image_path
    ).convert(
        "RGB"
    )


    ensemble_cam, component_cams = (
        compute_ensemble_gradcam(
            pil_img=pil_img,
            canonical_target_idx=
                class_idx
        )
    )


    # Convert original image to [0, 1]
    rgb = np.asarray(
        pil_img
    ).astype(
        np.float32
    ) / 255.0


    overlay = show_cam_on_image(
        rgb,
        ensemble_cam,
        use_rgb=True
    )

    # Figure
    fig, ax = plt.subplots(
        figsize=(3.2, 3.2)
    )

    ax.imshow(
        overlay
    )

    ax.axis(
        "off"
    )

    plt.tight_layout(
        pad=0
    )


    output_pdf = os.path.join(
        GRADCAM_OUTPUT_DIR,
        (
            "figures_gradcam_ensemble_per_class_"
            f"{class_name}.pdf"
        )
    )


    output_png = os.path.join(
        GRADCAM_OUTPUT_DIR,
        (
            "figures_gradcam_ensemble_per_class_"
            f"{class_name}.png"
        )
    )


    fig.savefig(
        output_pdf,
        bbox_inches="tight",
        pad_inches=0
    )

    fig.savefig(
        output_png,
        dpi=300,
        bbox_inches="tight",
        pad_inches=0
    )

    plt.close(
        fig
    )


    generated_gradcam_files.append(
        output_pdf
    )


    print(
        f"Saved: {output_pdf}"
    )


print(
    "\nPASS: Seven revised per-class ensemble attribution figures generated."
)

print(
    f"Output directory: {GRADCAM_OUTPUT_DIR}"
)

### Same-Image Melanoma Grad-CAM Model Comparison

In [ ]:
# Original Image + Grad-CAM Overlay
# Same representative melanoma sample across models

mel_row = (
    gradcam_representatives_df[
        gradcam_representatives_df["class_name"] == "mel"
    ]
    .iloc[0]
)

mel_class_idx = GRADCAM_CLASS_TO_IDX["mel"]

mel_image_id = mel_row["image_id"]
mel_image_path = mel_row["path"]

assert os.path.exists(mel_image_path), \
    f"Missing melanoma image: {mel_image_path}"

mel_pil_img = Image.open(
    mel_image_path
).convert("RGB")

# Keep image ID in console output for reproducibility,
# but do not display it inside the publication figure.
print(f"Melanoma comparison image: {mel_image_id}")
print(f"Target canonical class: mel ({mel_class_idx})")

# Generate all component CAMs and ensemble CAM
mel_ensemble_cam, mel_component_cams = (
    compute_ensemble_gradcam(
        pil_img=mel_pil_img,
        canonical_target_idx=mel_class_idx
    )
)

# Original image in [0, 1]
mel_rgb = (
    np.asarray(mel_pil_img)
    .astype(np.float32)
    / 255.0
)

# Models included in supplementary comparison
supplement_models = {
    "cnn": (
        "CNN",
        mel_component_cams["CNN"]
    ),

    "resnet": (
        "ResNet-50",
        mel_component_cams["ResNet-50"]
    ),

    "dense": (
        "DenseNet-121",
        mel_component_cams["DenseNet-121"]
    ),

    "vit": (
        "ViT",
        mel_component_cams["ViT"]
    ),

    "ensemble": (
        "Ensemble",
        mel_ensemble_cam
    ),
}

# Generate one Original | Grad-CAM figure per model
for file_key, (display_name, cam) in supplement_models.items():

    overlay = show_cam_on_image(
        mel_rgb,
        cam,
        use_rgb=True
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(6.4, 3.2)
    )

    # Original image
    axes[0].imshow(mel_rgb)
    axes[0].set_title(
        "Original",
        fontsize=11
    )
    axes[0].axis("off")

    # Grad-CAM overlay
    axes[1].imshow(overlay)
    axes[1].set_title(
        f"{display_name} Grad-CAM",
        fontsize=11
    )
    axes[1].axis("off")

    # Keep only the lesion class in the visible figure.
    # The specific ISIC image ID remains in the reproducibility manifest.
    fig.suptitle(
        "Melanoma (mel)",
        fontsize=12
    )

    plt.tight_layout(
        rect=[0, 0, 1, 0.93]
    )

    output_pdf = os.path.join(
        GRADCAM_OUTPUT_DIR,
        f"figures_gradcam_models_{file_key}_mel.pdf"
    )

    output_png = os.path.join(
        GRADCAM_OUTPUT_DIR,
        f"figures_gradcam_models_{file_key}_mel.png"
    )

    fig.savefig(
        output_pdf,
        bbox_inches="tight"
    )

    fig.savefig(
        output_png,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)

    print(
        f"Saved {display_name}: {output_pdf}"
    )


print(
    "\nPASS: Same-image melanoma Original + Grad-CAM "
    "comparison figures generated."
)

print(
    f"Representative melanoma image ID retained for reproducibility: "
    f"{mel_image_id}"
)

### Revised Grad-CAM Reproducibility Audit and Export

In [ ]:
# Revised Grad-CAM Reproducibility Audit and Export
expected_manuscript_gradcam_files = [
    (
        "figures_gradcam_ensemble_per_class_"
        f"{class_name}.pdf"
    )
    for class_name
    in GRADCAM_CLASS_NAMES
]


expected_supplement_gradcam_files = [
    "figures_gradcam_models_cnn_mel.pdf",
    "figures_gradcam_models_resnet_mel.pdf",
    "figures_gradcam_models_dense_mel.pdf",
    "figures_gradcam_models_vit_mel.pdf",
    "figures_gradcam_models_ensemble_mel.pdf",
]


all_expected_gradcam_files = (
    expected_manuscript_gradcam_files
    +
    expected_supplement_gradcam_files
)


audit_rows = []


for filename in all_expected_gradcam_files:

    full_path = os.path.join(
        GRADCAM_OUTPUT_DIR,
        filename
    )

    exists = os.path.exists(
        full_path
    )

    size_bytes = (
        os.path.getsize(
            full_path
        )
        if exists
        else 0
    )


    audit_rows.append({

        "filename":
            filename,

        "exists":
            exists,

        "size_bytes":
            size_bytes,
    })


gradcam_audit_df = pd.DataFrame(
    audit_rows
)


display(
    gradcam_audit_df
)


assert gradcam_audit_df[
    "exists"
].all(), \
    "One or more revised Grad-CAM files are missing."


assert (
    gradcam_audit_df[
        "size_bytes"
    ]
    >
    0
).all(), \
    "One or more Grad-CAM files are empty."


GRADCAM_AUDIT_CSV = os.path.join(
    GRADCAM_OUTPUT_DIR,
    "HAM10000_gradcam_output_audit_revised_final.csv"
)


gradcam_audit_df.to_csv(
    GRADCAM_AUDIT_CSV,
    index=False
)


print(
    "\nPASS: All revised Grad-CAM outputs verified."
)

print(
    f"Audit saved: {GRADCAM_AUDIT_CSV}"
)

print(
    f"Representative manifest: {GRADCAM_MANIFEST_CSV}"
)

### ConvNeXt Preprocessing Audit

In [ ]:
# Same 2,003 canonical images; checkpoint unchanged
convnext_probs_05 = []

convnext_model.eval()

with torch.no_grad():

    for _, row in tqdm(
        canonical_test_df.iterrows(),
        total=len(canonical_test_df),
        desc="ConvNeXt audit — 0.5 normalization"
    ):

        path = row["path"]
        img = Image.open(path).convert("RGB")

        # Old/shared preprocessing used in the submitted evaluation pathway
        x = preprocess_224_05(img).unsqueeze(0).to(device)

        probs = torch.softmax(
            convnext_model(x),
            dim=1
        ).cpu().numpy()[0]

        convnext_probs_05.append(probs)

convnext_probs_05 = np.asarray(
    convnext_probs_05,
    dtype=np.float32
)

assert convnext_probs_05.shape == (2003, 7)

convnext_preds_05 = convnext_probs_05.argmax(axis=1)

convnext_acc_05 = accuracy_score(
    labels,
    convnext_preds_05
)

convnext_f1_05 = f1_score(
    labels,
    convnext_preds_05,
    average="weighted"
)

convnext_macro_auc_05 = roc_auc_score(
    labels,
    convnext_probs_05,
    multi_class="ovr",
    average="macro"
)

convnext_micro_auc_05 = roc_auc_score(
    labels,
    convnext_probs_05,
    multi_class="ovr",
    average="micro"
)

print("\nConvNeXt-Tiny — 0.5 normalization")
print("----------------------------------")
print(f"Accuracy:     {convnext_acc_05:.4f}")
print(f"Weighted F1:  {convnext_f1_05:.4f}")
print(f"Macro AUC:    {convnext_macro_auc_05:.4f}")
print(f"Micro AUC:    {convnext_micro_auc_05:.4f}")

print("\nSubmitted Table 1 ConvNeXt-Tiny")
print("--------------------------------")
print("Accuracy:     0.7858")
print("Weighted F1:  0.7403")
print("Macro AUC:    0.9367")
print("Micro AUC:    0.9593")

## 3.2 ISIC External Evaluation

This section evaluates the cross-dataset performance of the seven fixed HAM10000-trained model checkpoints—CNN, ResNet-50, DenseNet-121, EfficientNet-B3, ConvNeXt-Tiny, MobileNetV3-Large, and ViT—on the external ISIC 2019 dataset.

For compatibility with the seven-class HAM10000 label space, ISIC 2019 labels `AK` and `SCC` were both mapped to the unified `akiec` category. Only samples mapping to the seven canonical classes (`akiec`, `bcc`, `bkl`, `df`, `mel`, `nv`, and `vasc`) were retained. Architecture-specific preprocessing was applied to each fixed model checkpoint, and model outputs were mapped to the same canonical seven-class ordering before evaluation.

The external evaluation includes:

- **Model-wise inference:** Each fixed model checkpoint generates class-probability predictions on the same harmonized ISIC 2019 external evaluation set.
- **Classification metrics:** Aggregate performance metrics are computed consistently for comparison across models.
- **Confusion matrices and ROC analysis:** Prediction patterns and class-wise discrimination are evaluated on the external dataset.
- **Ensemble evaluation:** The same equal-weight probability ensemble used in the harmonized HAM10000 analysis is applied, with each of the seven models assigned a weight of 1/7 and no optimization of ensemble weights.

These analyses provide a standardized cross-dataset assessment of model performance under dataset shift. ISIC 2019 is therefore described as an **external evaluation set** rather than as an external validation set.

**Overlap with HAM10000.** The ISIC 2019 challenge set is assembled from HAM10000, BCN_20000 and MSK. HAM10000 occupies the identifier block `ISIC_0024307` to `ISIC_0034317`, which contains 10,011 of the 25,331 images evaluated here (39.5%); every checkpoint saw these images during training. The aggregate results in this section therefore overstate cross-dataset generalization. The partition by source, which gives the genuinely external estimate on the 15,320 unseen images, is computed in the Revision Analyses section at the end of the notebook.


### Constants and Class Mappings

In [ ]:
ham10000_classes = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
class_to_idx = {cls: i for i, cls in enumerate(ham10000_classes)}

# ISIC to HAM10000 label mapping
label_map = {
    0: 'nv', 1: 'mel', 2: 'bkl', 3: 'bcc', 4: 'ak', 5: 'vasc', 6: 'df', 7: 'scc'
}
isic_map = {
    'nv': 'nv', 'mel': 'mel', 'bkl': 'bkl', 'bcc': 'bcc',
    'ak': 'akiec', 'scc': 'akiec',
    'vasc': 'vasc', 'df': 'df'
}

ISIC_CSV = "/kaggle/input/isic-2019-jpg-224x224-resized/train-metadata.csv"
ISIC_IMG = "/kaggle/input/isic-2019-jpg-224x224-resized/train-image/image"

### Verify ISIC 2019 Dataset Paths

In [ ]:
# Locate ISIC 2019 Dataset in Current Kaggle Session
print("Searching for ISIC 2019 metadata files...\n")

csv_candidates = glob.glob(
    "/kaggle/input/**/*.csv",
    recursive=True
)

isic_candidates = [
    p for p in csv_candidates
    if "isic" in p.lower()
]

for p in isic_candidates:
    print(p)

print("\nSearching for likely ISIC image folders...\n")

dir_candidates = glob.glob(
    "/kaggle/input/**/*",
    recursive=True
)

for p in dir_candidates:
    if os.path.isdir(p) and "isic" in p.lower():
        print(p)

### Configure and Verify ISIC 2019 Dataset Paths

In [ ]:
# Constants and Class Mappings
ham10000_classes = [
    'akiec',
    'bcc',
    'bkl',
    'df',
    'mel',
    'nv',
    'vasc'
]

class_to_idx = {
    cls: i
    for i, cls in enumerate(ham10000_classes)
}

# ISIC to HAM10000 label mapping
label_map = {
    0: 'nv',
    1: 'mel',
    2: 'bkl',
    3: 'bcc',
    4: 'ak',
    5: 'vasc',
    6: 'df',
    7: 'scc'
}

isic_map = {
    'nv': 'nv',
    'mel': 'mel',
    'bkl': 'bkl',
    'bcc': 'bcc',
    'ak': 'akiec',
    'scc': 'akiec',
    'vasc': 'vasc',
    'df': 'df'
}

# Correct current Kaggle mount paths
ISIC_CSV = (
    "/kaggle/input/datasets/nischaydnk/"
    "isic-2019-jpg-224x224-resized/"
    "train-metadata.csv"
)

ISIC_IMG = (
    "/kaggle/input/datasets/nischaydnk/"
    "isic-2019-jpg-224x224-resized/"
    "train-image/image"
)

# Safety checks
assert os.path.isfile(ISIC_CSV), \
    f"ISIC metadata file not found: {ISIC_CSV}"

assert os.path.isdir(ISIC_IMG), \
    f"ISIC image directory not found: {ISIC_IMG}"

print("ISIC metadata:")
print(ISIC_CSV)

print("\nISIC image directory:")
print(ISIC_IMG)

### Prepare ISIC Dataset DataFrame

In [ ]:
df = pd.read_csv(ISIC_CSV)

df["diagnosis"] = df["target"].map(label_map)
df["diagnosis"] = df["diagnosis"].map(isic_map)
df = df[df["diagnosis"].isin(ham10000_classes)].reset_index(drop=True)
df["label"] = df["diagnosis"].map(class_to_idx)

df["img_path"] = df["isic_id"].apply(lambda x: os.path.join(ISIC_IMG, f"{x}.jpg"))
df = df[df["img_path"].apply(os.path.exists)]

print(f"Valid ISIC Samples: {len(df)}")
print(df[['isic_id', 'diagnosis', 'label']].head())

### Locate and Verify HAM10000 Model Checkpoints

In [ ]:
# Locate HAM10000 Model Weight Files in Current Kaggle Session
patterns = [
    "cnn_ham10000_best_model.pth",
    "resnet50_ham10000.pth",
    "densenet121_ham10000.pth",
    "efficientnet_b3_best.pth",
    "convnext_tiny_ham10000.pth",
    "mobilenetv3_ham10000.pth",
    "vit_ham10000_best_model.pth",
]

for fname in patterns:
    matches = glob.glob(
        f"/kaggle/input/**/{fname}",
        recursive=True
    )

    print(f"\n{fname}")

    if matches:
        for p in matches:
            print(p)
    else:
        print("NOT FOUND")

### Load HAM10000-Trained Models for ISIC 2019 External Evaluation

In [ ]:
# Load HAM10000-Trained Models for ISIC 2019 External Evaluation

# Device and number of classes
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

num_classes = 7

print("Device:", device)

# Current permanent Kaggle checkpoint paths
CNN_PATH = (
    "/kaggle/input/datasets/nhsaykat/"
    "kaggleinputham10000-model-weights/"
    "cnn_ham10000_best_model.pth"
)

RESNET_PATH = (
    "/kaggle/input/datasets/nhsaykat/"
    "resnet50-ham10000-model-weights/"
    "resnet50_ham10000.pth"
)

DENSENET_PATH = (
    "/kaggle/input/datasets/nhsaykat/"
    "densenet121-ham10000-model-weights/"
    "densenet121_ham10000.pth"
)

EFFNET_PATH = (
    "/kaggle/input/datasets/nhsaykat/"
    "efficientnetb3-ham10000-model-weights/"
    "efficientnet_b3_best.pth"
)

CONVNEXT_PATH = (
    "/kaggle/input/datasets/nhsaykat/"
    "ham10000-convnext-tiny-weights/"
    "convnext_tiny_ham10000.pth"
)

MOBILENET_PATH = (
    "/kaggle/input/datasets/nhsaykat/"
    "mobilenetv3-ham10000-model-weights/"
    "mobilenetv3_ham10000.pth"
)

VIT_PATH = (
    "/kaggle/input/datasets/nhsaykat/"
    "vit-ham10000-model-weights/"
    "vit_ham10000_best_model.pth"
)

VIT_DIR = os.path.dirname(VIT_PATH)

# Verify all checkpoints
checkpoint_paths = {
    "CNN": CNN_PATH,
    "ResNet50": RESNET_PATH,
    "DenseNet121": DENSENET_PATH,
    "EfficientNetB3": EFFNET_PATH,
    "ConvNeXtTiny": CONVNEXT_PATH,
    "MobileNetV3": MOBILENET_PATH,
    "ViT": VIT_PATH,
}

print("\nCheckpoint verification")
print("-----------------------")

for name, path in checkpoint_paths.items():

    exists = os.path.isfile(path)

    print(
        f"{name:18s}: "
        f"{'FOUND' if exists else 'MISSING'}"
    )

    assert exists, \
        f"{name} checkpoint not found: {path}"

# Custom CNN architecture
class SimpleCNN(nn.Module):

    def __init__(self, num_classes=7):

        super(SimpleCNN, self).__init__()

        self.features = nn.Sequential(

            nn.Conv2d(
                3, 32,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                32, 64,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                64, 128,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                128, 256,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(
                256 * 14 * 14,
                512
            ),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(
                512,
                num_classes
            )
        )

    def forward(self, x):

        x = self.features(x)
        x = self.classifier(x)

        return x

# Instantiate CNN-based architectures
cnn_model = SimpleCNN(
    num_classes=num_classes
)

resnet_model = resnet50(
    weights=None
)

resnet_model.fc = nn.Linear(
    resnet_model.fc.in_features,
    num_classes
)

densenet_model = densenet121(
    weights=None
)

densenet_model.classifier = nn.Linear(
    densenet_model.classifier.in_features,
    num_classes
)

effnet_model = efficientnet_b3(
    weights=None
)

effnet_model.classifier[1] = nn.Linear(
    effnet_model.classifier[1].in_features,
    num_classes
)

convnext_model = convnext_tiny(
    weights=None
)

convnext_model.classifier[2] = nn.Linear(
    convnext_model.classifier[2].in_features,
    num_classes
)

mobilenet_model = mobilenet_v3_large(
    weights=None
)

mobilenet_model.classifier[3] = nn.Linear(
    mobilenet_model.classifier[3].in_features,
    num_classes
)

# Load CNN-based checkpoints
cnn_model.load_state_dict(
    torch.load(
        CNN_PATH,
        map_location=device,
        weights_only=True
    )
)

resnet_model.load_state_dict(
    torch.load(
        RESNET_PATH,
        map_location=device,
        weights_only=True
    )
)

densenet_model.load_state_dict(
    torch.load(
        DENSENET_PATH,
        map_location=device,
        weights_only=True
    )
)

effnet_model.load_state_dict(
    torch.load(
        EFFNET_PATH,
        map_location=device,
        weights_only=True
    )
)

convnext_model.load_state_dict(
    torch.load(
        CONVNEXT_PATH,
        map_location=device,
        weights_only=True
    )
)

mobilenet_model.load_state_dict(
    torch.load(
        MOBILENET_PATH,
        map_location=device,
        weights_only=True
    )
)

print(
    "\nCNN-based model checkpoints "
    "loaded successfully."
)

# Reconstruct ViT architecture

# First check whether the permanent dataset also contains
# the Hugging Face configuration used for training.

VIT_CONFIG_PATH = os.path.join(
    VIT_DIR,
    "config.json"
)

if os.path.isfile(VIT_CONFIG_PATH):

    print(
        "\nFound ViT config.json:"
    )
    print(VIT_CONFIG_PATH)

    vit_config = ViTConfig.from_pretrained(
        VIT_DIR,
        local_files_only=True
    )

else:

    print(
        "\nNo config.json found in ViT checkpoint directory."
    )

    print(
        "Using standard ViT-B/16 (224 x 224) "
        "configuration."
    )

    vit_config = ViTConfig(
        image_size=224,
        patch_size=16,
        num_channels=3,
        hidden_size=768,
        num_hidden_layers=12,
        num_attention_heads=12,
        intermediate_size=3072,
        hidden_dropout_prob=0.0,
        attention_probs_dropout_prob=0.0,
        num_labels=num_classes,
    )

vit_config.num_labels = num_classes

vit_model = ViTForImageClassification(
    vit_config
)

# Load ViT checkpoint
vit_state = torch.load(
    VIT_PATH,
    map_location=device,
    weights_only=True
)

vit_model.load_state_dict(
    vit_state,
    strict=True
)

print("ViT checkpoint loaded successfully.")

# Bundle all seven models
models_dict = {
    "CNN": cnn_model,
    "ResNet50": resnet_model,
    "DenseNet121": densenet_model,
    "EfficientNetB3": effnet_model,
    "ConvNeXtTiny": convnext_model,
    "MobileNetV3": mobilenet_model,
    "ViT": vit_model,
}

# Move models to device and set evaluation mode
for name, model in models_dict.items():

    model.to(device)
    model.eval()

# Final verification
assert len(models_dict) == 7

print("\nAll 7 HAM10000-trained models loaded successfully.")
print("Ready for ISIC 2019 external evaluation.")

print("\nLoaded models:")
for name in models_dict:
    print(" -", name)

### Load and Map ISIC 2019 Ground Truth to HAM10000 7-Class Labels

In [ ]:
# Load and Map ISIC 2019 Ground Truth to HAM10000 7-Class Labels

# Correct current Kaggle paths
gt_path = (
    "/kaggle/input/datasets/nhsaykat/"
    "isic-2019-groundtruth-7classes-ham10000-mapped/"
    "ISIC_2019_Training_GroundTruth.csv"
)

image_dir = (
    "/kaggle/input/datasets/nischaydnk/"
    "isic-2019-jpg-224x224-resized/"
    "train-image/image"
)

assert os.path.isfile(gt_path), \
    f"ISIC ground-truth CSV not found: {gt_path}"

assert os.path.isdir(image_dir), \
    f"ISIC image directory not found: {image_dir}"

print("Ground-truth CSV:")
print(gt_path)

print("\nImage directory:")
print(image_dir)

# Load mapped ISIC 2019 ground truth
gt_df = pd.read_csv(gt_path)

print("\nGround-truth columns:")
print(gt_df.columns.tolist())

# Diagnosis one-hot columns
onehot_cols = [
    "MEL",
    "NV",
    "BCC",
    "AK",
    "BKL",
    "DF",
    "VASC",
    "SCC",
]

missing_onehot = [
    c for c in onehot_cols
    if c not in gt_df.columns
]

assert not missing_onehot, (
    "Missing expected diagnosis columns: "
    + ", ".join(missing_onehot)
)

# Derive HAM10000-compatible diagnosis
if "diagnosis" not in gt_df.columns:

    gt_df["diagnosis"] = (
        gt_df[onehot_cols]
        .idxmax(axis=1)
        .str.lower()
    )

    gt_df["diagnosis"] = (
        gt_df["diagnosis"]
        .replace({
            "ak": "akiec",
            "scc": "akiec",
        })
    )

# Canonical HAM10000 class mapping
ham_classes = [
    "akiec",
    "bcc",
    "bkl",
    "df",
    "mel",
    "nv",
    "vasc",
]

class_to_idx = {
    cls: idx
    for idx, cls in enumerate(ham_classes)
}

gt_df["label"] = (
    gt_df["diagnosis"]
    .map(class_to_idx)
)

# Remove any unmapped rows defensively
gt_df = (
    gt_df[
        gt_df["label"].notna()
    ]
    .copy()
)

gt_df["label"] = (
    gt_df["label"]
    .astype(int)
)

# Build image paths
assert "image" in gt_df.columns, \
    "Expected 'image' column not found in ground-truth CSV."

gt_df["img_path"] = (
    gt_df["image"]
    .astype(str)
    .apply(
        lambda x: os.path.join(
            image_dir,
            f"{x}.jpg"
        )
    )
)

# Keep only available images
gt_df = (
    gt_df[
        gt_df["img_path"]
        .apply(os.path.exists)
    ]
    .reset_index(drop=True)
)

# Final verification
print("\nFinal ISIC 2019 evaluation samples:")
print(len(gt_df))

print("\nLabel distribution:")
print(
    gt_df["diagnosis"]
    .value_counts()
)

print("\nFirst rows:")
display(
    gt_df[
        [
            "image",
            "diagnosis",
            "label",
            "img_path",
        ]
    ]
    .head()
)

assert len(gt_df) > 0, \
    "No valid ISIC images found."

assert set(
    gt_df["diagnosis"].unique()
).issubset(
    set(ham_classes)
)

assert set(
    gt_df["label"].unique()
).issubset(
    set(range(7))
)

print(
    "\nISIC 2019 ground truth mapped "
    "successfully to the HAM10000 7-class label space."
)

### Model-Specific ISIC 2019 External Evaluation Preprocessing

In [ ]:
# Model-Specific ISIC 2019 External Evaluation Preprocessing

# Canonical evaluation class order
CANONICAL_CLASSES = [
    "akiec",
    "bcc",
    "bkl",
    "df",
    "mel",
    "nv",
    "vasc",
]

assert ham_classes == CANONICAL_CLASSES

# Historical ResNet/ViT training-time order
RESNET_VIT_TRAIN_CLASSES = [
    "bkl",
    "nv",
    "df",
    "mel",
    "vasc",
    "bcc",
    "akiec",
]

RESNET_VIT_TO_CANONICAL = [
    RESNET_VIT_TRAIN_CLASSES.index(c)
    for c in CANONICAL_CLASSES
]

assert RESNET_VIT_TO_CANONICAL == [
    6, 5, 0, 2, 3, 1, 4
]

# CNN / DenseNet-121 / MobileNetV3-L
preprocess_224_05 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.5, 0.5, 0.5],
        [0.5, 0.5, 0.5]
    ),
])

# ResNet-50
preprocess_resnet = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    ),
])

# EfficientNet-B3
preprocess_effnet = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.5, 0.5, 0.5],
        [0.5, 0.5, 0.5]
    ),
])

# ConvNeXt-Tiny
# Original training used ImageNet normalization

preprocess_convnext = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    ),
])

# ViT
vit_processor = ViTImageProcessor.from_pretrained(
    "google/vit-base-patch16-224"
)

print("Model-specific ISIC preprocessing pipelines defined.")
print(
    "ResNet/ViT -> canonical mapping:",
    RESNET_VIT_TO_CANONICAL
)

### Create ISIC 2019 External Evaluation Dataset

In [ ]:
# Create ISIC 2019 External Evaluation Dataset
class ISIC2019Dataset(Dataset):

    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        return (
            str(row["img_path"]),
            int(row["label"])
        )


test_dataset_isic = ISIC2019Dataset(
    gt_df
)

test_loader_isic = DataLoader(
    test_dataset_isic,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

print(
    "ISIC 2019 external evaluation samples:",
    len(test_dataset_isic)
)

assert len(test_dataset_isic) == 25331

### Run Unified Model-Specific Inference on ISIC 2019

In [ ]:
# Unified Model-Specific ISIC 2019 Inference
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

results = {}

model_keys = [
    "CNN",
    "ResNet50",
    "DenseNet121",
    "EfficientNetB3",
    "ConvNeXtTiny",
    "MobileNetV3",
    "ViT",
]


@torch.no_grad()
def predict_isic_image(
    model_name,
    pil_img
):

    model = models_dict[model_name]

    # CNN
    if model_name == "CNN":

        x = (
            preprocess_224_05(pil_img)
            .unsqueeze(0)
            .to(device)
        )

        probs = torch.softmax(
            model(x),
            dim=1
        ).cpu().numpy()[0]

    # ResNet-50
    elif model_name == "ResNet50":

        x = (
            preprocess_resnet(pil_img)
            .unsqueeze(0)
            .to(device)
        )

        probs_raw = torch.softmax(
            model(x),
            dim=1
        ).cpu().numpy()[0]

        # historical order -> canonical order
        probs = probs_raw[
            RESNET_VIT_TO_CANONICAL
        ]

    # DenseNet-121
    elif model_name == "DenseNet121":

        x = (
            preprocess_224_05(pil_img)
            .unsqueeze(0)
            .to(device)
        )

        probs = torch.softmax(
            model(x),
            dim=1
        ).cpu().numpy()[0]

    # EfficientNet-B3
    elif model_name == "EfficientNetB3":

        x = (
            preprocess_effnet(pil_img)
            .unsqueeze(0)
            .to(device)
        )

        probs = torch.softmax(
            model(x),
            dim=1
        ).cpu().numpy()[0]

    # ConvNeXt-Tiny
    elif model_name == "ConvNeXtTiny":

        x = (
            preprocess_convnext(pil_img)
            .unsqueeze(0)
            .to(device)
        )

        probs = torch.softmax(
            model(x),
            dim=1
        ).cpu().numpy()[0]

    # MobileNetV3-L
    elif model_name == "MobileNetV3":

        x = (
            preprocess_224_05(pil_img)
            .unsqueeze(0)
            .to(device)
        )

        probs = torch.softmax(
            model(x),
            dim=1
        ).cpu().numpy()[0]

    # ViT
    elif model_name == "ViT":

        encoded = vit_processor(
            images=pil_img,
            return_tensors="pt"
        )

        pixel_values = (
            encoded["pixel_values"]
            .to(device)
        )

        logits = model(
            pixel_values=pixel_values
        ).logits

        probs_raw = torch.softmax(
            logits,
            dim=1
        ).cpu().numpy()[0]

        # historical order -> canonical order
        probs = probs_raw[
            RESNET_VIT_TO_CANONICAL
        ]

    else:

        raise ValueError(
            f"Unknown model: {model_name}"
        )

    assert probs.shape == (7,)

    assert np.isfinite(
        probs
    ).all()

    assert np.isclose(
        probs.sum(),
        1.0,
        atol=1e-5
    )

    return probs

# Run all seven models on exactly the same ordered images
for model_name in model_keys:

    print(
        f"\nEvaluating {model_name}..."
    )

    model = models_dict[
        model_name
    ].to(device)

    model.eval()

    model_probs = []
    model_labels = []

    for _, row in tqdm(
        gt_df.iterrows(),
        total=len(gt_df),
        desc=f"{model_name} ISIC inference"
    ):

        path = row["img_path"]

        pil_img = Image.open(
            path
        ).convert("RGB")

        probs = predict_isic_image(
            model_name,
            pil_img
        )

        model_probs.append(
            probs
        )

        model_labels.append(
            int(row["label"])
        )

    model_probs = np.asarray(
        model_probs
    )

    model_labels = np.asarray(
        model_labels
    )

    model_preds = (
        model_probs.argmax(axis=1)
    )

    assert model_probs.shape == (
        25331,
        7
    )

    assert model_labels.shape == (
        25331,
    )

    results[model_name] = {
        "preds": model_preds,
        "labels": model_labels,
        "probs": model_probs,
    }

    print(
        f"{model_name}: "
        f"{model_probs.shape}"
    )

# Verify identical label sequence for all models
reference_labels = results[
    "CNN"
]["labels"]

for model_name in model_keys:

    assert np.array_equal(
        results[model_name]["labels"],
        reference_labels
    ), (
        f"Label ordering mismatch: "
        f"{model_name}"
    )

print(
    "\nUnified ISIC 2019 inference "
    "completed successfully."
)

print(
    "All seven models evaluated on "
    "the same ordered 25,331 images."
)

### ISIC 2019 Metrics and Classification Reports

In [ ]:
# ISIC 2019 Metrics and Classification Reports
class_names = [
    "akiec",
    "bcc",
    "bkl",
    "df",
    "mel",
    "nv",
    "vasc",
]

class_labels = np.arange(len(class_names))

# Safety checks
assert "results" in globals(), \
    "results is missing. Run the revised ISIC inference first."

expected_models = [
    "CNN",
    "ResNet50",
    "DenseNet121",
    "EfficientNetB3",
    "ConvNeXtTiny",
    "MobileNetV3",
    "ViT",
]

assert set(results.keys()) == set(expected_models), (
    "Unexpected models in results:\n"
    f"{list(results.keys())}"
)

metric_rows = []
per_class_rows = []

# Evaluate each model
for name in expected_models:

    res = results[name]

    preds = np.asarray(res["preds"])
    labels = np.asarray(res["labels"])
    probs = np.asarray(res["probs"])

    # Structural validation
    assert labels.shape == (25331,), \
        f"{name}: unexpected labels shape {labels.shape}"

    assert preds.shape == (25331,), \
        f"{name}: unexpected predictions shape {preds.shape}"

    assert probs.shape == (25331, 7), \
        f"{name}: unexpected probability shape {probs.shape}"

    assert np.isfinite(probs).all(), \
        f"{name}: non-finite probabilities detected"

    assert np.allclose(
        probs.sum(axis=1),
        1.0,
        atol=1e-5
    ), f"{name}: probability rows do not sum to 1"

    # Aggregate metrics
    acc = accuracy_score(
        labels,
        preds
    )

    f1_weighted = f1_score(
        labels,
        preds,
        labels=class_labels,
        average="weighted",
        zero_division=0,
    )

    auc_macro = roc_auc_score(
        labels,
        probs,
        labels=class_labels,
        multi_class="ovr",
        average="macro",
    )

    auc_weighted = roc_auc_score(
        labels,
        probs,
        labels=class_labels,
        multi_class="ovr",
        average="weighted",
    )

    # Per-class F1
    per_class_f1 = f1_score(
        labels,
        preds,
        labels=class_labels,
        average=None,
        zero_division=0,
    )

    # Store aggregate results
    metric_rows.append({
        "Model": name,
        "Accuracy": acc,
        "Weighted F1": f1_weighted,
        "Macro ROC-AUC (OvR)": auc_macro,
        "Weighted ROC-AUC (OvR)": auc_weighted,
    })

    # Store per-class results
    per_class_row = {
        "Model": name
    }

    for cls_name, score in zip(
        class_names,
        per_class_f1
    ):
        per_class_row[
            cls_name.upper()
        ] = score

    per_class_rows.append(
        per_class_row
    )

    # Print detailed report
    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    print(
        f"Accuracy:                {acc:.4f}"
    )

    print(
        f"Weighted F1:             {f1_weighted:.4f}"
    )

    print(
        f"Macro ROC-AUC (OvR):     {auc_macro:.4f}"
    )

    print(
        f"Weighted ROC-AUC (OvR):  {auc_weighted:.4f}"
    )

    print("\nClassification Report:\n")

    print(
        classification_report(
            labels,
            preds,
            labels=class_labels,
            target_names=class_names,
            digits=4,
            zero_division=0,
        )
    )

# Build final audit tables
isic_metrics_df = pd.DataFrame(
    metric_rows
)

isic_per_class_f1_df = pd.DataFrame(
    per_class_rows
)


print("\n\nISIC 2019 AGGREGATE METRICS")
print("===========================")

display(
    isic_metrics_df.round(4)
)


print("\nISIC 2019 PER-CLASS F1")
print("======================")

display(
    isic_per_class_f1_df.round(4)
)

# Save audit artifacts
OUTPUT_DIR = "/kaggle/working"

metrics_path = os.path.join(
    OUTPUT_DIR,
    "ISIC2019_metrics_revised_audit.csv"
)

perclass_path = os.path.join(
    OUTPUT_DIR,
    "ISIC2019_per_class_F1_revised_audit.csv"
)

isic_metrics_df.to_csv(
    metrics_path,
    index=False
)

isic_per_class_f1_df.to_csv(
    perclass_path,
    index=False
)

print("\nSaved:")
print(metrics_path)
print(perclass_path)

### Save Revised ISIC 2019 Prediction Arrays and Evaluation Results

In [ ]:
# Save Revised ISIC 2019 Prediction Arrays and Evaluation Results
OUTPUT_DIR = "/kaggle/working"

# Safety checks
assert "results" in globals(), \
    "results is missing."

expected_models = [
    "CNN",
    "ResNet50",
    "DenseNet121",
    "EfficientNetB3",
    "ConvNeXtTiny",
    "MobileNetV3",
    "ViT",
]

assert set(results.keys()) == set(expected_models)

# Save one canonical label array
reference_labels = np.asarray(
    results["CNN"]["labels"]
)

for model_name in expected_models:
    assert np.array_equal(
        results[model_name]["labels"],
        reference_labels
    ), f"Label ordering differs for {model_name}"

LABEL_PATH = os.path.join(
    OUTPUT_DIR,
    "isic2019_labels.npy"
)

np.save(
    LABEL_PATH,
    reference_labels
)

# Save model probability and prediction arrays
file_name_map = {
    "CNN": "cnn",
    "ResNet50": "resnet50",
    "DenseNet121": "densenet121",
    "EfficientNetB3": "efficientnet_b3",
    "ConvNeXtTiny": "convnext_tiny",
    "MobileNetV3": "mobilenet_v3",
    "ViT": "vit",
}

saved_files = {
    "labels": LABEL_PATH
}

for model_name in expected_models:

    short_name = file_name_map[model_name]

    probs = np.asarray(
        results[model_name]["probs"]
    )

    preds = np.asarray(
        results[model_name]["preds"]
    )

    assert probs.shape == (25331, 7)
    assert preds.shape == (25331,)

    probs_path = os.path.join(
        OUTPUT_DIR,
        f"isic2019_{short_name}_probs.npy"
    )

    preds_path = os.path.join(
        OUTPUT_DIR,
        f"isic2019_{short_name}_preds.npy"
    )

    np.save(
        probs_path,
        probs
    )

    np.save(
        preds_path,
        preds
    )

    saved_files[
        f"{short_name}_probs"
    ] = probs_path

    saved_files[
        f"{short_name}_preds"
    ] = preds_path

# Optional compact NPZ archive
NPZ_PATH = os.path.join(
    OUTPUT_DIR,
    "isic2019_external_eval_predictions.npz"
)

np.savez_compressed(
    NPZ_PATH,
    labels=reference_labels,
    **{
        f"{file_name_map[name]}_probs":
            np.asarray(results[name]["probs"])
        for name in expected_models
    },
    **{
        f"{file_name_map[name]}_preds":
            np.asarray(results[name]["preds"])
        for name in expected_models
    }
)

saved_files[
    "compressed_archive"
] = NPZ_PATH

# Save lightweight JSON manifest
MANIFEST_PATH = os.path.join(
    OUTPUT_DIR,
    "isic2019_external_eval_manifest.json"
)

manifest = {
    "evaluation_dataset": "ISIC 2019",
    "n_samples": int(len(reference_labels)),
    "n_classes": 7,
    "class_order": [
        "akiec",
        "bcc",
        "bkl",
        "df",
        "mel",
        "nv",
        "vasc",
    ],
    "models": expected_models,
    "files": {
        key: os.path.basename(path)
        for key, path in saved_files.items()
    },
}

with open(
    MANIFEST_PATH,
    "w"
) as f:
    json.dump(
        manifest,
        f,
        indent=2
    )

# Final verification
print("Saved revised ISIC 2019 evaluation artifacts:\n")

for key, path in saved_files.items():
    print(
        f"{key:25s}: {path}"
    )

print(
    f"{'manifest':25s}: "
    f"{MANIFEST_PATH}"
)

### Save Final ISIC 2019 Metrics Summary

In [ ]:
# Save Final ISIC 2019 Metrics Summary
# Source: revised audited ISIC metrics

assert "isic_metrics_df" in globals(), \
    "Run the ISIC 2019 Metrics and Classification Reports cell first."

summary_df = isic_metrics_df.copy()

name_map = {
    "CNN": "CNN",
    "ResNet50": "ResNet-50",
    "DenseNet121": "DenseNet-121",
    "EfficientNetB3": "EfficientNet-B3",
    "ConvNeXtTiny": "ConvNeXt-Tiny",
    "MobileNetV3": "MobileNetV3-L",
    "ViT": "ViT",
}

summary_df["Model"] = (
    summary_df["Model"]
    .map(name_map)
)

assert summary_df["Model"].notna().all(), \
    "Unexpected model name detected."

# Keep full precision in the saved CSV
FINAL_SUMMARY_PATH = (
    "/kaggle/working/"
    "ISIC2019_metrics_revised_final.csv"
)

summary_df.to_csv(
    FINAL_SUMMARY_PATH,
    index=False
)

print("Final revised ISIC 2019 metrics summary:")
display(summary_df.round(4))

print("\nSaved:")
print(FINAL_SUMMARY_PATH)

### Save Revised ISIC 2019 Classification Reports

In [ ]:
# Save Revised ISIC 2019 Classification Reports
assert "results" in globals(), \
    "results is missing. Run the revised ISIC inference first."

class_names = [
    "akiec",
    "bcc",
    "bkl",
    "df",
    "mel",
    "nv",
    "vasc",
]

class_labels = list(range(7))

name_map = {
    "CNN": "CNN",
    "ResNet50": "ResNet-50",
    "DenseNet121": "DenseNet-121",
    "EfficientNetB3": "EfficientNet-B3",
    "ConvNeXtTiny": "ConvNeXt-Tiny",
    "MobileNetV3": "MobileNetV3-L",
    "ViT": "ViT",
}

OUTPUT_DIR = "/kaggle/working"

saved_reports = []

for key, res in results.items():

    assert key in name_map, \
        f"Unexpected model key: {key}"

    report = classification_report(
        res["labels"],
        res["preds"],
        labels=class_labels,
        target_names=class_names,
        digits=4,
        zero_division=0
    )

    safe_name = (
        name_map[key]
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )

    out_path = os.path.join(
        OUTPUT_DIR,
        f"isic2019_{safe_name}_classification_report.txt"
    )

    with open(
        out_path,
        "w"
    ) as f:
        f.write(report)

    saved_reports.append(out_path)

print("Saved revised ISIC 2019 classification reports:\n")

for path in saved_reports:
    print(path)

### Generate Revised ISIC 2019 Confusion Matrices

In [ ]:
# Generate Revised ISIC 2019 Confusion Matrices
assert "results" in globals(), \
    "results is missing. Run the revised ISIC inference first."

class_names = [
    "akiec",
    "bcc",
    "bkl",
    "df",
    "mel",
    "nv",
    "vasc",
]

class_labels = list(range(7))

name_map = {
    "CNN": "CNN",
    "ResNet50": "ResNet-50",
    "DenseNet121": "DenseNet-121",
    "EfficientNetB3": "EfficientNet-B3",
    "ConvNeXtTiny": "ConvNeXt-Tiny",
    "MobileNetV3": "MobileNetV3-L",
    "ViT": "ViT",
}

OUTPUT_DIR = "/kaggle/working"

saved_confusion_matrices = []

for key in name_map:

    res = results[key]

    y_true = res["labels"]
    y_pred = res["preds"]

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=class_labels
    )

    display_name = name_map[key]

    plt.figure(
        figsize=(8, 6)
    )

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
    )

    plt.title(
        f"ISIC 2019 Confusion Matrix — {display_name}"
    )

    plt.xlabel(
        "Predicted Label"
    )

    plt.ylabel(
        "True Label"
    )

    plt.tight_layout()

    safe_name = (
        display_name
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )

    png_path = os.path.join(
        OUTPUT_DIR,
        f"isic2019_confusion_matrix_{safe_name}.png"
    )

    pdf_path = os.path.join(
        OUTPUT_DIR,
        f"isic2019_confusion_matrix_{safe_name}.pdf"
    )

    plt.savefig(
        png_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.savefig(
        pdf_path,
        bbox_inches="tight"
    )

    plt.show()

    saved_confusion_matrices.extend(
        [
            png_path,
            pdf_path,
        ]
    )

print(
    "Saved revised ISIC 2019 confusion matrices:\n"
)

for path in saved_confusion_matrices:
    print(path)

### Compute Revised Equal-Weight Ensemble Predictions on ISIC 2019

In [ ]:
# Compute Revised Equal-Weight Ensemble Predictions on ISIC 2019

# Safety checks
assert "results" in globals(), \
    "results is missing. Run the revised ISIC inference first."

ensemble_model_order = [
    "CNN",
    "ResNet50",
    "DenseNet121",
    "EfficientNetB3",
    "ConvNeXtTiny",
    "MobileNetV3",
    "ViT",
]

for name in ensemble_model_order:
    assert name in results, \
        f"Missing model predictions: {name}"

# Verify identical label ordering for all seven models
ensemble_labels = np.asarray(
    results["CNN"]["labels"]
)

assert ensemble_labels.shape == (25331,), \
    f"Unexpected label shape: {ensemble_labels.shape}"

for name in ensemble_model_order:

    model_labels = np.asarray(
        results[name]["labels"]
    )

    assert np.array_equal(
        model_labels,
        ensemble_labels
    ), (
        f"Label ordering differs for {name}. "
        "Ensemble averaging would not be valid."
    )

print(
    "PASS: All seven models use the same "
    "25,331-image label ordering."
)

# Validate and stack probability matrices
probability_arrays = []

for name in ensemble_model_order:

    probs = np.asarray(
        results[name]["probs"]
    )

    assert probs.shape == (25331, 7), \
        f"{name}: unexpected probability shape {probs.shape}"

    assert np.isfinite(probs).all(), \
        f"{name}: non-finite probability values detected"

    assert np.all(probs >= 0), \
        f"{name}: negative probability values detected"

    assert np.allclose(
        probs.sum(axis=1),
        1.0,
        atol=1e-5
    ), f"{name}: probability rows do not sum to 1"

    probability_arrays.append(
        probs
    )

stacked_isic_probs = np.stack(
    probability_arrays,
    axis=0
)

assert stacked_isic_probs.shape == (
    7,
    25331,
    7,
)

# Equal-weight probability ensemble
ensemble_probs = np.mean(
    stacked_isic_probs,
    axis=0
)

ensemble_preds = np.argmax(
    ensemble_probs,
    axis=1
)

assert ensemble_probs.shape == (25331, 7)
assert ensemble_preds.shape == (25331,)

assert np.isfinite(
    ensemble_probs
).all()

assert np.allclose(
    ensemble_probs.sum(axis=1),
    1.0,
    atol=1e-5
)

# Add ensemble to revised results dictionary
results["Ensemble"] = {
    "preds": ensemble_preds,
    "labels": ensemble_labels,
    "probs": ensemble_probs,
}

print(
    "\nRevised ISIC 2019 equal-weight ensemble "
    "computed successfully."
)

print(
    "Ensemble probabilities:",
    ensemble_probs.shape
)

print(
    "Ensemble predictions:",
    ensemble_preds.shape
)

print(
    "Ensemble labels:",
    ensemble_labels.shape
)

### Generate Revised Ensemble Confusion Matrix on ISIC 2019

In [ ]:
# Generate Revised Ensemble Confusion Matrix on ISIC 2019

# Safety checks
assert "ensemble_labels" in globals(), \
    "ensemble_labels is missing."

assert "ensemble_preds" in globals(), \
    "ensemble_preds is missing."

assert "ensemble_probs" in globals(), \
    "ensemble_probs is missing."

class_names = [
    "akiec",
    "bcc",
    "bkl",
    "df",
    "mel",
    "nv",
    "vasc",
]

class_labels = list(range(7))

ensemble_labels = np.asarray(
    ensemble_labels
)

ensemble_preds = np.asarray(
    ensemble_preds
)

ensemble_probs = np.asarray(
    ensemble_probs
)

assert ensemble_labels.shape == (25331,)
assert ensemble_preds.shape == (25331,)
assert ensemble_probs.shape == (25331, 7)

assert set(
    np.unique(ensemble_labels)
).issubset(set(class_labels))

assert set(
    np.unique(ensemble_preds)
).issubset(set(class_labels))

# Compute confusion matrix
cm_ensemble = confusion_matrix(
    ensemble_labels,
    ensemble_preds,
    labels=class_labels
)

assert cm_ensemble.shape == (7, 7)

assert cm_ensemble.sum() == len(
    ensemble_labels
)

print(
    "Ensemble confusion matrix sample count:",
    cm_ensemble.sum()
)

# Plot
fig, ax = plt.subplots(
    figsize=(8, 6)
)

sns.heatmap(
    cm_ensemble,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    ax=ax,
)

ax.set_title(
    "Confusion Matrix — Ensemble (ISIC 2019)"
)

ax.set_xlabel(
    "Predicted Label"
)

ax.set_ylabel(
    "True Label"
)

plt.tight_layout()

# Save publication artifacts
OUTPUT_DIR = "/kaggle/working"

PNG_PATH = os.path.join(
    OUTPUT_DIR,
    "confusion_matrix_ensemble_isic2019_revised_final.png"
)

PDF_PATH = os.path.join(
    OUTPUT_DIR,
    "confusion_matrix_ensemble_isic2019_revised_final.pdf"
)

fig.savefig(
    PNG_PATH,
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    PDF_PATH,
    bbox_inches="tight"
)

plt.show()

print("\nSaved revised ensemble confusion matrix:")
print(PNG_PATH)
print(PDF_PATH)

### Generate Revised ISIC 2019 Classwise ROC Curves

In [ ]:
# Generate Revised ISIC 2019 Classwise ROC Curves
assert "results" in globals(), \
    "results is missing. Run the revised ISIC inference first."

class_names = [
    "akiec",
    "bcc",
    "bkl",
    "df",
    "mel",
    "nv",
    "vasc",
]

class_labels = list(range(7))

# Include the revised ensemble if already computed
plot_model_order = [
    "CNN",
    "ResNet50",
    "DenseNet121",
    "EfficientNetB3",
    "ConvNeXtTiny",
    "MobileNetV3",
    "ViT",
]

if "Ensemble" in results:
    plot_model_order.append("Ensemble")

name_map = {
    "CNN": "CNN",
    "ResNet50": "ResNet-50",
    "DenseNet121": "DenseNet-121",
    "EfficientNetB3": "EfficientNet-B3",
    "ConvNeXtTiny": "ConvNeXt-Tiny",
    "MobileNetV3": "MobileNetV3-L",
    "ViT": "ViT",
    "Ensemble": "Ensemble",
}

OUTPUT_DIR = "/kaggle/working"

saved_roc_files = []

for key in plot_model_order:

    assert key in results, \
        f"Missing results for {key}"

    y_true = np.asarray(
        results[key]["labels"]
    )

    y_score = np.asarray(
        results[key]["probs"]
    )

    assert y_true.shape == (25331,), \
        f"{key}: unexpected label shape"

    assert y_score.shape == (25331, 7), \
        f"{key}: unexpected probability shape"

    assert np.isfinite(y_score).all(), \
        f"{key}: non-finite probabilities"

    assert np.allclose(
        y_score.sum(axis=1),
        1.0,
        atol=1e-5
    ), f"{key}: probabilities do not sum to 1"

    y_true_bin = label_binarize(
        y_true,
        classes=class_labels
    )

    fig, ax = plt.subplots(
        figsize=(10, 8)
    )

    for i, class_name in enumerate(class_names):

        fpr, tpr, _ = roc_curve(
            y_true_bin[:, i],
            y_score[:, i]
        )

        class_auc = sk_auc(
            fpr,
            tpr
        )

        ax.plot(
            fpr,
            tpr,
            label=(
                f"{class_name} "
                f"(AUC = {class_auc:.3f})"
            )
        )

    ax.plot(
        [0, 1],
        [0, 1],
        "k--",
        label="Chance"
    )

    display_name = name_map[key]

    ax.set_xlabel(
        "False Positive Rate"
    )

    ax.set_ylabel(
        "True Positive Rate"
    )

    ax.set_title(
        f"ISIC 2019 Classwise ROC Curves — {display_name}"
    )

    ax.legend(
        loc="lower right"
    )

    ax.grid(
        alpha=0.25
    )

    fig.tight_layout()

    safe_name = (
        display_name
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )

    png_path = os.path.join(
        OUTPUT_DIR,
        f"isic2019_roc_curves_{safe_name}_revised_final.png"
    )

    pdf_path = os.path.join(
        OUTPUT_DIR,
        f"isic2019_roc_curves_{safe_name}_revised_final.pdf"
    )

    fig.savefig(
        png_path,
        dpi=300,
        bbox_inches="tight"
    )

    fig.savefig(
        pdf_path,
        bbox_inches="tight"
    )

    plt.show()

    saved_roc_files.extend(
        [
            png_path,
            pdf_path,
        ]
    )

print("\nSaved revised ISIC 2019 ROC figures:\n")

for path in saved_roc_files:
    print(path)

### Generate and Save Revised Ensemble ROC Curve for ISIC 2019

In [ ]:
# Generate and Save Revised Ensemble ROC Curve for ISIC 2019

# Safety checks
assert "results" in globals(), \
    "results is missing."

assert "Ensemble" in results, \
    "Revised ensemble is missing. Run the revised ensemble cell first."

class_names = [
    "akiec",
    "bcc",
    "bkl",
    "df",
    "mel",
    "nv",
    "vasc",
]

class_labels = list(range(7))

ensemble_labels = np.asarray(
    results["Ensemble"]["labels"]
)

ensemble_probs = np.asarray(
    results["Ensemble"]["probs"]
)

ensemble_preds = np.asarray(
    results["Ensemble"]["preds"]
)

assert ensemble_labels.shape == (25331,)
assert ensemble_probs.shape == (25331, 7)
assert ensemble_preds.shape == (25331,)

assert np.isfinite(
    ensemble_probs
).all()

assert np.allclose(
    ensemble_probs.sum(axis=1),
    1.0,
    atol=1e-5
)

# Binarize labels
y_true_bin = label_binarize(
    ensemble_labels,
    classes=class_labels
)

# Compute and plot classwise ROC curves
fig, ax = plt.subplots(
    figsize=(10, 8)
)

ensemble_class_auc = {}

for i, cls in enumerate(class_names):

    fpr, tpr, _ = roc_curve(
        y_true_bin[:, i],
        ensemble_probs[:, i]
    )

    auc_val = sk_auc(
        fpr,
        tpr
    )

    ensemble_class_auc[cls] = auc_val

    ax.plot(
        fpr,
        tpr,
        label=f"{cls} (AUC = {auc_val:.3f})"
    )

ax.plot(
    [0, 1],
    [0, 1],
    "k--",
    label="Chance"
)

ax.set_xlabel(
    "False Positive Rate"
)

ax.set_ylabel(
    "True Positive Rate"
)

ax.set_title(
    "Classwise ROC Curves — Ensemble (ISIC 2019)"
)

ax.legend(
    loc="lower right"
)

ax.grid(
    alpha=0.25
)

fig.tight_layout()

# Save publication artifacts
OUTPUT_DIR = "/kaggle/working"

PNG_PATH = os.path.join(
    OUTPUT_DIR,
    "roc_curve_ensemble_isic2019_revised_final.png"
)

PDF_PATH = os.path.join(
    OUTPUT_DIR,
    "roc_curve_ensemble_isic2019_revised_final.pdf"
)

fig.savefig(
    PNG_PATH,
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    PDF_PATH,
    bbox_inches="tight"
)

plt.show()

# Print classwise AUC values
print("\nEnsemble class-wise ROC-AUC values:")

for cls, auc_val in ensemble_class_auc.items():
    print(f"{cls:5s}: {auc_val:.4f}")

print("\nSaved revised ensemble ROC figures:")
print(PNG_PATH)
print(PDF_PATH)

### Save Revised ISIC 2019 Evaluation Metrics to CSV

In [ ]:
# Save Revised ISIC 2019 Evaluation Metrics to CSV

# Safety checks
assert "results" in globals(), \
    "results is missing. Run the revised ISIC evaluation first."

model_order = [
    "CNN",
    "ResNet50",
    "DenseNet121",
    "EfficientNetB3",
    "ConvNeXtTiny",
    "MobileNetV3",
    "ViT",
    "Ensemble",
]

name_map = {
    "CNN": "CNN",
    "ResNet50": "ResNet-50",
    "DenseNet121": "DenseNet-121",
    "EfficientNetB3": "EfficientNet-B3",
    "ConvNeXtTiny": "ConvNeXt-Tiny",
    "MobileNetV3": "MobileNetV3-L",
    "ViT": "ViT",
    "Ensemble": "Ensemble",
}

for name in model_order:
    assert name in results, \
        f"Missing results for {name}"

# Compute revised metrics
metrics_summary = []

for name in model_order:

    y_true = np.asarray(
        results[name]["labels"]
    )

    y_pred = np.asarray(
        results[name]["preds"]
    )

    y_prob = np.asarray(
        results[name]["probs"]
    )

    assert y_true.shape == (25331,), \
        f"{name}: unexpected label shape {y_true.shape}"

    assert y_pred.shape == (25331,), \
        f"{name}: unexpected prediction shape {y_pred.shape}"

    assert y_prob.shape == (25331, 7), \
        f"{name}: unexpected probability shape {y_prob.shape}"

    assert np.isfinite(y_prob).all(), \
        f"{name}: non-finite probabilities detected"

    # Accuracy
    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    # Weighted F1
    weighted_f1 = f1_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    # Macro one-vs-rest ROC-AUC
    macro_auc = roc_auc_score(
        y_true,
        y_prob,
        labels=list(range(7)),
        multi_class="ovr",
        average="macro"
    )

    # Weighted one-vs-rest ROC-AUC
    weighted_auc = roc_auc_score(
        y_true,
        y_prob,
        labels=list(range(7)),
        multi_class="ovr",
        average="weighted"
    )

    metrics_summary.append({
        "Model": name_map[name],
        "Accuracy": accuracy,
        "Weighted F1": weighted_f1,
        "Macro ROC-AUC (OvR)": macro_auc,
        "Weighted ROC-AUC (OvR)": weighted_auc,
    })

# Create final metrics dataframe
metrics_df = pd.DataFrame(
    metrics_summary
)

# Save
OUTPUT_PATH = (
    "/kaggle/working/"
    "ISIC2019_metrics_revised_final.csv"
)

metrics_df.to_csv(
    OUTPUT_PATH,
    index=False
)

# Display and verify
print(
    "Revised ISIC 2019 evaluation metrics "
    "saved successfully:"
)

print(OUTPUT_PATH)

display(
    metrics_df.round(4)
)

### Save Revised ISIC 2019 Predictions to CSV

In [ ]:
# Save Revised ISIC 2019 Predictions to CSV

assert "gt_df" in globals(), \
    "gt_df is missing."

assert "results" in globals(), \
    "results is missing."

class_names = [
    "akiec",
    "bcc",
    "bkl",
    "df",
    "mel",
    "nv",
    "vasc",
]

model_order = [
    "CNN",
    "ResNet50",
    "DenseNet121",
    "EfficientNetB3",
    "ConvNeXtTiny",
    "MobileNetV3",
    "ViT",
    "Ensemble",
]

name_map = {
    "CNN": "CNN",
    "ResNet50": "ResNet50",
    "DenseNet121": "DenseNet121",
    "EfficientNetB3": "EfficientNetB3",
    "ConvNeXtTiny": "ConvNeXtTiny",
    "MobileNetV3": "MobileNetV3",
    "ViT": "ViT",
    "Ensemble": "Ensemble",
}

# Start from harmonized ISIC metadata/ground truth
final_df = gt_df.copy().reset_index(drop=True)

assert len(final_df) == 25331, \
    f"Unexpected number of ISIC rows: {len(final_df)}"

# Add predictions and class probabilities
for key in model_order:

    assert key in results, \
        f"Missing results for {key}"

    preds = np.asarray(
        results[key]["preds"]
    )

    probs = np.asarray(
        results[key]["probs"]
    )

    labels = np.asarray(
        results[key]["labels"]
    )

    assert preds.shape == (25331,)
    assert probs.shape == (25331, 7)
    assert labels.shape == (25331,)

    assert np.isfinite(probs).all(), \
        f"{key}: non-finite probabilities detected"

    model_name = name_map[key]

    # Predicted class index
    final_df[f"{model_name}_pred_index"] = preds

    # Predicted class name
    final_df[f"{model_name}_pred_class"] = [
        class_names[i]
        for i in preds
    ]

    # One probability column per diagnostic class
    for class_idx, class_name in enumerate(class_names):

        final_df[
            f"{model_name}_prob_{class_name}"
        ] = probs[:, class_idx]
# Save
OUTPUT_PATH = (
    "/kaggle/working/"
    "ISIC2019_all_model_predictions_revised_final.csv"
)

final_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print(
    "Revised ISIC 2019 prediction table saved successfully:"
)

print(OUTPUT_PATH)

print(
    "\nShape:",
    final_df.shape
)

display(
    final_df.head()
)

### Rank Models by Revised ISIC 2019 Evaluation Performance

In [ ]:
# Rank Models by Revised ISIC 2019 Evaluation Performance

assert "results" in globals(), \
    "results is missing."

model_order = [
    "CNN",
    "ResNet50",
    "DenseNet121",
    "EfficientNetB3",
    "ConvNeXtTiny",
    "MobileNetV3",
    "ViT",
    "Ensemble",
]

name_map = {
    "CNN": "CNN",
    "ResNet50": "ResNet-50",
    "DenseNet121": "DenseNet-121",
    "EfficientNetB3": "EfficientNet-B3",
    "ConvNeXtTiny": "ConvNeXt-Tiny",
    "MobileNetV3": "MobileNetV3-L",
    "ViT": "ViT",
    "Ensemble": "Ensemble",
}

# Compute metrics
ranking_rows = []

for key in model_order:

    assert key in results, \
        f"Missing results for {key}"

    y_true = np.asarray(
        results[key]["labels"]
    )

    y_pred = np.asarray(
        results[key]["preds"]
    )

    y_prob = np.asarray(
        results[key]["probs"]
    )

    assert y_true.shape == (25331,)
    assert y_pred.shape == (25331,)
    assert y_prob.shape == (25331, 7)

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    weighted_f1 = f1_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    macro_auc = roc_auc_score(
        y_true,
        y_prob,
        labels=list(range(7)),
        multi_class="ovr",
        average="macro"
    )

    weighted_auc = roc_auc_score(
        y_true,
        y_prob,
        labels=list(range(7)),
        multi_class="ovr",
        average="weighted"
    )

    ranking_rows.append({
        "Model": name_map[key],
        "Accuracy": accuracy,
        "Weighted F1": weighted_f1,
        "Macro ROC-AUC (OvR)": macro_auc,
        "Weighted ROC-AUC (OvR)": weighted_auc,
    })

ranking_df = pd.DataFrame(
    ranking_rows
)

# Calculate rank for each metric
ranking_df["Accuracy Rank"] = (
    ranking_df["Accuracy"]
    .rank(method="min", ascending=False)
)

ranking_df["Weighted F1 Rank"] = (
    ranking_df["Weighted F1"]
    .rank(method="min", ascending=False)
)

ranking_df["Macro AUC Rank"] = (
    ranking_df["Macro ROC-AUC (OvR)"]
    .rank(method="min", ascending=False)
)

ranking_df["Weighted AUC Rank"] = (
    ranking_df["Weighted ROC-AUC (OvR)"]
    .rank(method="min", ascending=False)
)

# Mean rank across the four reported metrics
rank_columns = [
    "Accuracy Rank",
    "Weighted F1 Rank",
    "Macro AUC Rank",
    "Weighted AUC Rank",
]

ranking_df["Mean Rank"] = (
    ranking_df[rank_columns]
    .mean(axis=1)
)

# Lower mean rank = better overall ranking
ranking_df = (
    ranking_df
    .sort_values(
        ["Mean Rank", "Accuracy"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

ranking_df.insert(
    0,
    "Overall Rank",
    np.arange(1, len(ranking_df) + 1)
)

# Save
OUTPUT_PATH = (
    "/kaggle/working/"
    "ISIC2019_model_ranking_revised_final.csv"
)

ranking_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print(
    "Revised ISIC 2019 model ranking saved:"
)

print(OUTPUT_PATH)

display(
    ranking_df.round(4)
)

### Package Revised ISIC 2019 Evaluation Artifacts

In [ ]:
# Package Revised ISIC 2019 Evaluation Artifacts
WORKING_DIR = "/kaggle/working"

ZIP_PATH = os.path.join(
    WORKING_DIR,
    "ISIC2019_revised_evaluation_artifacts.zip"
)

# File types relevant to the revised external evaluation
allowed_extensions = {
    ".csv",   # metrics, rankings, prediction tables
    ".png",   # high-resolution raster figures
    ".pdf",   # publication-ready figures
    ".npy",   # exact prediction/probability arrays
    ".npz",   # compressed numerical artifacts
    ".json",  # manifests / metadata
    ".txt",   # classification reports
}

# Collect files
files_to_archive = []

for foldername, subfolders, filenames in os.walk(WORKING_DIR):

    for filename in filenames:

        filepath = os.path.join(
            foldername,
            filename
        )

        extension = os.path.splitext(
            filename
        )[1].lower()

        # Do not include the ZIP currently being created
        if os.path.abspath(filepath) == os.path.abspath(ZIP_PATH):
            continue

        if extension in allowed_extensions:
            files_to_archive.append(
                filepath
            )

files_to_archive = sorted(
    files_to_archive
)

assert len(files_to_archive) > 0, \
    "No evaluation artifacts were found."

# Create ZIP archive
with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as zipf:

    for filepath in files_to_archive:

        arcname = os.path.relpath(
            filepath,
            WORKING_DIR
        )

        zipf.write(
            filepath,
            arcname
        )

# Verify ZIP contents
with zipfile.ZipFile(
    ZIP_PATH,
    mode="r"
) as zipf:

    archived_files = zipf.namelist()

print(
    "Revised ISIC 2019 evaluation archive created successfully."
)

print(
    "\nArchive:",
    ZIP_PATH
)

print(
    "Number of archived files:",
    len(archived_files)
)

print("\nArchived files:")

for filename in archived_files:
    print("  ", filename)

print(
    "\nArchive size:",
    f"{os.path.getsize(ZIP_PATH) / (1024 ** 2):.2f} MB"
)

### Aggregate Revised ISIC 2019 Model Metrics

In [ ]:
# Aggregate Revised ISIC 2019 Model Metrics

# Configuration
class_names = [
    "akiec",
    "bcc",
    "bkl",
    "df",
    "mel",
    "nv",
    "vasc",
]

class_labels = list(range(7))

model_order = [
    "CNN",
    "ResNet50",
    "DenseNet121",
    "EfficientNetB3",
    "ConvNeXtTiny",
    "MobileNetV3",
    "ViT",
    "Ensemble",
]

name_map = {
    "CNN": "CNN",
    "ResNet50": "ResNet-50",
    "DenseNet121": "DenseNet-121",
    "EfficientNetB3": "EfficientNet-B3",
    "ConvNeXtTiny": "ConvNeXt-Tiny",
    "MobileNetV3": "MobileNetV3-L",
    "ViT": "ViT",
    "Ensemble": "Ensemble",
}

assert "results" in globals(), \
    "results is missing."

# Aggregate metrics
metrics_list = []

for key in model_order:

    assert key in results, \
        f"Missing results for {key}"

    preds = np.asarray(
        results[key]["preds"]
    )

    labels = np.asarray(
        results[key]["labels"]
    )

    probs = np.asarray(
        results[key]["probs"]
    )

    # Safety checks
    assert labels.shape == (25331,), \
        f"{key}: unexpected label shape {labels.shape}"

    assert preds.shape == (25331,), \
        f"{key}: unexpected prediction shape {preds.shape}"

    assert probs.shape == (25331, 7), \
        f"{key}: unexpected probability shape {probs.shape}"

    assert np.isfinite(probs).all(), \
        f"{key}: non-finite probabilities detected"

    # Accuracy
    acc = accuracy_score(
        labels,
        preds
    )

    # Weighted F1
    weighted_f1 = f1_score(
        labels,
        preds,
        average="weighted",
        zero_division=0
    )

    # Macro OvR ROC-AUC
    macro_auc = roc_auc_score(
        labels,
        probs,
        labels=class_labels,
        multi_class="ovr",
        average="macro"
    )

    # Weighted OvR ROC-AUC
    weighted_auc = roc_auc_score(
        labels,
        probs,
        labels=class_labels,
        multi_class="ovr",
        average="weighted"
    )

    metrics_list.append({
        "Model": name_map[key],
        "Accuracy": acc,
        "Weighted F1": weighted_f1,
        "Macro ROC-AUC (OvR)": macro_auc,
        "Weighted ROC-AUC (OvR)": weighted_auc,
    })

# Create DataFrame
df_metrics = pd.DataFrame(
    metrics_list
)

# Sort descriptively by accuracy
df_metrics_sorted = (
    df_metrics
    .sort_values(
        by="Accuracy",
        ascending=False
    )
    .reset_index(drop=True)
)

# Display
display(
    df_metrics_sorted.round(4)
)

### Export Revised ISIC 2019 Performance Table to LaTeX

In [ ]:
# Export Revised ISIC 2019 Performance Table to LaTeX

assert "df_metrics_sorted" in globals(), \
    "df_metrics_sorted is missing. Run the aggregate metrics cell first."

expected_columns = [
    "Model",
    "Accuracy",
    "Weighted F1",
    "Macro ROC-AUC (OvR)",
    "Weighted ROC-AUC (OvR)",
]

assert list(df_metrics_sorted.columns) == expected_columns, \
    "Unexpected columns in df_metrics_sorted."

# Export LaTeX table
latex_table = df_metrics_sorted.to_latex(
    index=False,
    column_format="lcccc",
    float_format="%.4f",
    caption=(
        "External evaluation performance on the harmonized "
        "ISIC 2019 dataset."
    ),
    label="tab:isic2019-results-export",
    escape=False,
)

# Save
OUTPUT_PATH = (
    "/kaggle/working/"
    "ISIC2019_model_performance_revised_final.tex"
)

with open(
    OUTPUT_PATH,
    "w",
    encoding="utf-8"
) as f:
    f.write(latex_table)

print(
    "Revised ISIC 2019 LaTeX table saved to:"
)

print(OUTPUT_PATH)

print("\nGenerated LaTeX:\n")
print(latex_table)

### Export Revised ISIC 2019 Performance Table to Excel

In [ ]:
# Export Revised ISIC 2019 Performance Table to Excel

assert "df_metrics_sorted" in globals(), \
    "df_metrics_sorted is missing. Run the aggregate metrics cell first."

expected_columns = [
    "Model",
    "Accuracy",
    "Weighted F1",
    "Macro ROC-AUC (OvR)",
    "Weighted ROC-AUC (OvR)",
]

assert list(df_metrics_sorted.columns) == expected_columns, \
    "Unexpected columns in df_metrics_sorted."

# Export to Excel
EXCEL_PATH = (
    "/kaggle/working/"
    "ISIC2019_model_performance_revised_final.xlsx"
)

df_metrics_sorted.to_excel(
    EXCEL_PATH,
    index=False,
    sheet_name="ISIC2019 Performance"
)

# Verify export
assert os.path.exists(EXCEL_PATH), \
    "Excel file was not created."

# Read back to verify
excel_check = pd.read_excel(
    EXCEL_PATH,
    sheet_name="ISIC2019 Performance"
)

assert excel_check.shape == df_metrics_sorted.shape, \
    "Exported Excel dimensions do not match the source table."

assert list(excel_check.columns) == expected_columns, \
    "Exported Excel columns do not match."

print(
    "Revised ISIC 2019 Excel table saved successfully:"
)

print(EXCEL_PATH)

print(
    "\nShape:",
    excel_check.shape
)

display(
    excel_check.round(4)
)

## 4.1 Ensemble Evaluation

### Standardized HAM10000 Evaluation — Individual Models and Ensemble

In [ ]:
# Individual Models and Equal-Weight Ensemble

# Configuration
class_names = [
    "akiec",
    "bcc",
    "bkl",
    "df",
    "mel",
    "nv",
    "vasc",
]

class_labels = list(range(7))

model_files = {
    "CNN": "canonical_cnn_probs.npy",
    "ResNet-50": "canonical_resnet50_probs.npy",
    "DenseNet-121": "canonical_densenet121_probs.npy",
    "EfficientNet-B3": "canonical_efficientnet_b3_probs.npy",
    "ConvNeXt-Tiny": "canonical_convnext_tiny_probs.npy",
    "MobileNetV3-L": "canonical_mobilenet_v3_probs.npy",
    "ViT": "canonical_vit_probs.npy",
    "Ensemble": "canonical_ensemble_probs.npy",
}

LABEL_FILE = "canonical_ham10000_labels.npy"

# Find previously saved authoritative arrays
def find_file(filename):
    """
    Search Kaggle working and input directories for an exact filename.
    """
    search_roots = [
        "/kaggle/working",
        "/kaggle/input",
    ]

    matches = []

    for root in search_roots:
        matches.extend(
            glob.glob(
                os.path.join(root, "**", filename),
                recursive=True
            )
        )

    matches = sorted(set(matches))

    if len(matches) == 0:
        raise FileNotFoundError(
            f"Could not find {filename}. "
            "Attach the dataset containing the revised HAM10000 "
            "prediction arrays or place the file in /kaggle/working/."
        )

    if len(matches) > 1:
        print(
            f"WARNING: multiple copies of {filename} found."
        )
        for path in matches:
            print("   ", path)

        print(
            "Using:",
            matches[0]
        )

    return matches[0]


label_path = find_file(
    LABEL_FILE
)

prob_paths = {
    model_name: find_file(filename)
    for model_name, filename in model_files.items()
}

# Load labels and canonical probability arrays
ham_labels = np.load(
    label_path
)

ham_probs = {
    model_name: np.load(path)
    for model_name, path in prob_paths.items()
}

assert ham_labels.shape == (2003,), \
    f"Unexpected HAM10000 label shape: {ham_labels.shape}"

for model_name, probs in ham_probs.items():

    assert probs.shape == (2003, 7), \
        f"{model_name}: unexpected shape {probs.shape}"

    assert np.isfinite(probs).all(), \
        f"{model_name}: non-finite probabilities detected"

    assert np.all(probs >= 0), \
        f"{model_name}: negative probabilities detected"

    assert np.allclose(
        probs.sum(axis=1),
        1.0,
        atol=1e-5
    ), \
        f"{model_name}: probability rows do not sum to 1"

print(
    "PASS: Loaded standardized HAM10000 evaluation arrays."
)

print(
    "Evaluation cohort size:",
    len(ham_labels)
)

# Compute standardized HAM10000 metrics
rows = []

for model_name in model_files.keys():

    probs = ham_probs[
        model_name
    ]

    preds = np.argmax(
        probs,
        axis=1
    )

    accuracy = accuracy_score(
        ham_labels,
        preds
    )

    weighted_f1 = f1_score(
        ham_labels,
        preds,
        average="weighted",
        zero_division=0
    )

    macro_auc = roc_auc_score(
        ham_labels,
        probs,
        labels=class_labels,
        multi_class="ovr",
        average="macro"
    )

    micro_auc = roc_auc_score(
        ham_labels,
        probs,
        labels=class_labels,
        multi_class="ovr",
        average="micro"
    )

    rows.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Weighted F1": weighted_f1,
        "Macro ROC-AUC (OvR)": macro_auc,
        "Micro ROC-AUC (OvR)": micro_auc,
    })

ham_comparison_df = pd.DataFrame(
    rows
)

# Keep manuscript model order rather than ranking rows
ham_comparison_df["Model"] = pd.Categorical(
    ham_comparison_df["Model"],
    categories=list(model_files.keys()),
    ordered=True
)

ham_comparison_df = (
    ham_comparison_df
    .sort_values("Model")
    .reset_index(drop=True)
)

# Display
display(
    ham_comparison_df.round(4)
)

# Strict consistency check against finalized Table 1
expected_metrics = {
    "CNN":
        [0.7898, 0.7763, 0.9388, 0.9712],

    "ResNet-50":
        [0.6850, 0.7142, 0.9319, 0.9386],

    "DenseNet-121":
        [0.8787, 0.8742, 0.9786, 0.9892],

    "EfficientNet-B3":
        [0.8997, 0.8984, 0.9818, 0.9905],

    "ConvNeXt-Tiny":
        [0.9126, 0.9124, 0.9882, 0.9937],

    "MobileNetV3-L":
        [0.8587, 0.8551, 0.9720, 0.9857],

    "ViT":
        [0.9591, 0.9585, 0.9959, 0.9978],

    "Ensemble":
        [0.9386, 0.9374, 0.9951, 0.9971],
}

metric_columns = [
    "Accuracy",
    "Weighted F1",
    "Macro ROC-AUC (OvR)",
    "Micro ROC-AUC (OvR)",
]

for _, row in ham_comparison_df.iterrows():

    model_name = str(
        row["Model"]
    )

    observed = np.array(
        [
            row[col]
            for col in metric_columns
        ]
    )

    expected = np.array(
        expected_metrics[model_name]
    )

    assert np.allclose(
        observed,
        expected,
        atol=5e-5
    ), (
        f"{model_name}: standardized metrics do not match "
        f"the finalized Table 1 values.\n"
        f"Observed: {observed}\n"
        f"Expected: {expected}"
    )

print(
    "\nPASS: All 32 standardized HAM10000 "
    "Table 1 metric values match."
)

# Save revised comparison table
HAM_METRICS_PATH = (
    "/kaggle/working/"
    "HAM10000_model_comparison_revised_final.csv"
)

ham_comparison_df.to_csv(
    HAM_METRICS_PATH,
    index=False
)

print(
    "\nSaved:",
    HAM_METRICS_PATH
)

# Ensemble classification report
ham_ensemble_probs = ham_probs[
    "Ensemble"
]

ham_ensemble_preds = np.argmax(
    ham_ensemble_probs,
    axis=1
)

print(
    "\nEnsemble classification report "
    "— standardized HAM10000 evaluation cohort:\n"
)

print(
    classification_report(
        ham_labels,
        ham_ensemble_preds,
        labels=class_labels,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

# Revised HAM10000 ensemble confusion matrix
ham_ensemble_cm = confusion_matrix(
    ham_labels,
    ham_ensemble_preds,
    labels=class_labels
)

assert ham_ensemble_cm.sum() == 2003

fig, ax = plt.subplots(
    figsize=(8, 6)
)

sns.heatmap(
    ham_ensemble_cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    ax=ax,
)

ax.set_xlabel(
    "Predicted Label"
)

ax.set_ylabel(
    "True Label"
)

ax.set_title(
    "Confusion Matrix — Ensemble "
    "(HAM10000 Standardized Evaluation)"
)

fig.tight_layout()

HAM_CM_PNG = (
    "/kaggle/working/"
    "confusion_matrix_ensemble_ham10000_revised_final.png"
)

HAM_CM_PDF = (
    "/kaggle/working/"
    "confusion_matrix_ensemble_ham10000_revised_final.pdf"
)

fig.savefig(
    HAM_CM_PNG,
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    HAM_CM_PDF,
    bbox_inches="tight"
)

plt.show()

print(
    "\nSaved revised HAM10000 ensemble confusion matrix:"
)

print(HAM_CM_PNG)
print(HAM_CM_PDF)

### Ensemble — Classwise ROC-AUC Curves on HAM10000

In [ ]:
# Standardized HAM10000 Evaluation Cohort

# Safety checks
assert "ham_labels" in globals(), \
    "ham_labels is missing. Run the standardized HAM10000 evaluation cell first."

assert "ham_ensemble_probs" in globals(), \
    "ham_ensemble_probs is missing. Run the standardized HAM10000 evaluation cell first."

assert "class_names" in globals(), \
    "class_names is missing."

assert ham_labels.shape == (2003,), \
    f"Unexpected label shape: {ham_labels.shape}"

assert ham_ensemble_probs.shape == (2003, 7), \
    f"Unexpected ensemble probability shape: {ham_ensemble_probs.shape}"

assert np.isfinite(ham_ensemble_probs).all(), \
    "Non-finite ensemble probabilities detected."

assert np.allclose(
    ham_ensemble_probs.sum(axis=1),
    1.0,
    atol=1e-5
), "Ensemble probability rows do not sum to 1."

# Binarize labels
class_labels = np.arange(7)

y_bin = label_binarize(
    ham_labels,
    classes=class_labels
)

assert y_bin.shape == (2003, 7)

# Compute classwise ROC curves and AUC
fpr = {}
tpr = {}
class_auc = {}

for i, class_name in enumerate(class_names):

    fpr[i], tpr[i], _ = roc_curve(
        y_bin[:, i],
        ham_ensemble_probs[:, i]
    )

    class_auc[i] = auc(
        fpr[i],
        tpr[i]
    )

# Compute aggregate ROC-AUC values
macro_auc = roc_auc_score(
    ham_labels,
    ham_ensemble_probs,
    labels=class_labels,
    multi_class="ovr",
    average="macro"
)

micro_auc = roc_auc_score(
    ham_labels,
    ham_ensemble_probs,
    labels=class_labels,
    multi_class="ovr",
    average="micro"
)

print("Classwise ROC-AUC values:\n")

for i, class_name in enumerate(class_names):
    print(
        f"{class_name:>5s}: "
        f"{class_auc[i]:.4f}"
    )

print(
    f"\nMacro OvR ROC-AUC: {macro_auc:.4f}"
)

print(
    f"Micro OvR ROC-AUC: {micro_auc:.4f}"
)

# Verify against finalized HAM10000 Table 1
assert np.isclose(
    macro_auc,
    0.9951,
    atol=5e-5
), (
    f"Macro AUC mismatch. "
    f"Observed={macro_auc:.6f}, expected=0.9951"
)

assert np.isclose(
    micro_auc,
    0.9971,
    atol=5e-5
), (
    f"Micro AUC mismatch. "
    f"Observed={micro_auc:.6f}, expected=0.9971"
)

print(
    "\nPASS: Aggregate ensemble ROC-AUC values "
    "match the finalized HAM10000 results."
)

# Plot classwise ROC curves
fig, ax = plt.subplots(
    figsize=(8.5, 7)
)

for i, class_name in enumerate(class_names):

    ax.plot(
        fpr[i],
        tpr[i],
        linewidth=2,
        label=(
            f"{class_name} "
            f"(AUC = {class_auc[i]:.3f})"
        )
    )


# Chance diagonal
ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1.5,
    label="Chance"
)

ax.set_xlim(
    0.0,
    1.0
)

ax.set_ylim(
    0.0,
    1.02
)

ax.set_xlabel(
    "False Positive Rate"
)

ax.set_ylabel(
    "True Positive Rate"
)

ax.set_title(
    "Classwise ROC Curves — Equal-Weight Ensemble\n"
    "HAM10000 Standardized Evaluation"
)

ax.legend(
    loc="lower right",
    fontsize=9
)

ax.grid(
    alpha=0.25
)

fig.tight_layout()

# Save PNG and PDF
ROC_PNG = (
    "/kaggle/working/"
    "roc_curves_ensemble_ham10000_revised_final.png"
)

ROC_PDF = (
    "/kaggle/working/"
    "roc_curves_ensemble_ham10000_revised_final.pdf"
)

fig.savefig(
    ROC_PNG,
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    ROC_PDF,
    bbox_inches="tight"
)

plt.show()

print(
    "\nSaved revised HAM10000 ensemble ROC figures:"
)

print(ROC_PNG)
print(ROC_PDF)

### Ensemble Weighting Strategy

In [ ]:
# Ensemble Weighting Strategy

# The finalized ensemble uses equal-weight probability averaging
# across all seven individual models.
#
# No post-hoc weight optimization is performed in the revised
# evaluation because the archived models were developed using
# different historical train/validation split constructions.
#
# Using one historical validation split to optimize weights across
# all models could therefore introduce an inconsistent model-selection
# procedure and complicate interpretation of the external evaluation.

ensemble_members = [
    "CNN",
    "ResNet-50",
    "DenseNet-121",
    "EfficientNet-B3",
    "ConvNeXt-Tiny",
    "MobileNetV3-L",
    "ViT",
]

equal_weights = np.full(
    len(ensemble_members),
    1.0 / len(ensemble_members)
)

print("Final ensemble strategy: equal-weight probability averaging")
print(f"Number of component models: {len(ensemble_members)}")

print("\nEnsemble weights:")

for model_name, weight in zip(
    ensemble_members,
    equal_weights
):
    print(
        f"{model_name:<18s}: {weight:.6f}"
    )

assert np.isclose(
    equal_weights.sum(),
    1.0
)

print(
    f"\nWeight sum: {equal_weights.sum():.6f}"
)

print(
    "\nNo validation-based weight optimization is used "
    "in the finalized analysis."
)

## 4.2 Statistical Analysis

### Prepare HAM10000 Per-Model Predictions and Metric Helper

In [ ]:
# Prepare HAM10000 Per-Model Predictions and Metric Helper

# Safety checks
assert "ham_labels" in globals(), \
    "ham_labels is missing. Run the standardized HAM10000 evaluation cell first."

assert "ham_probs" in globals(), \
    "ham_probs is missing. Run the standardized HAM10000 evaluation cell first."

assert isinstance(ham_labels, np.ndarray), \
    "ham_labels must be a NumPy array."

assert ham_labels.ndim == 1, \
    "ham_labels must be one-dimensional."

assert ham_labels.shape == (2003,), \
    f"Unexpected HAM10000 label shape: {ham_labels.shape}"

# Fixed finalized model order
ham_model_list = [
    "CNN",
    "ResNet-50",
    "DenseNet-121",
    "EfficientNet-B3",
    "ConvNeXt-Tiny",
    "MobileNetV3-L",
    "ViT",
]

# Verify all individual models and ensemble are present
for model_name in ham_model_list + ["Ensemble"]:

    assert model_name in ham_probs, \
        f"{model_name} is missing from ham_probs."

    probs = np.asarray(
        ham_probs[model_name]
    )

    assert probs.shape == (2003, 7), \
        f"{model_name}: unexpected probability shape {probs.shape}"

    assert np.isfinite(probs).all(), \
        f"{model_name}: non-finite probabilities detected."

    assert np.allclose(
        probs.sum(axis=1),
        1.0,
        atol=1e-5
    ), \
        f"{model_name}: probability rows do not sum to 1."

# Construct finalized probability dictionary
ham_all_probs = {
    model_name: ham_probs[model_name]
    for model_name in ham_model_list
}

ham_all_probs["Ensemble"] = ham_probs["Ensemble"]

# Predicted label
ham_all_preds = {
    model_name: probs.argmax(axis=1)
    for model_name, probs in ham_all_probs.items()
}

# Metric helper
def compute_ham_metrics(y_true, probs):
    """
    Compute the four finalized HAM10000 evaluation metrics:

    1. Accuracy
    2. Weighted F1
    3. Macro one-vs-rest ROC-AUC
    4. Micro one-vs-rest ROC-AUC
    """

    y_true = np.asarray(y_true)
    probs = np.asarray(probs)

    assert y_true.ndim == 1, \
        "y_true must be one-dimensional."

    assert probs.ndim == 2, \
        "probs must be two-dimensional."

    assert probs.shape[0] == y_true.shape[0], \
        "Number of predictions and labels must match."

    assert probs.shape[1] == 7, \
        "Expected seven probability columns."

    assert np.isfinite(probs).all(), \
        "Non-finite probabilities detected."

    y_pred = probs.argmax(
        axis=1
    )

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    weighted_f1 = f1_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    macro_auc = roc_auc_score(
        y_true,
        probs,
        labels=np.arange(7),
        multi_class="ovr",
        average="macro"
    )

    micro_auc = roc_auc_score(
        y_true,
        probs,
        labels=np.arange(7),
        multi_class="ovr",
        average="micro"
    )

    return (
        accuracy,
        weighted_f1,
        macro_auc,
        micro_auc,
    )

# Sanity check using finalized ensemble
ensemble_metrics_check = compute_ham_metrics(
    ham_labels,
    ham_all_probs["Ensemble"]
)

print(
    "Finalized HAM10000 ensemble metrics:"
)

print(
    f"Accuracy:      {ensemble_metrics_check[0]:.4f}"
)

print(
    f"Weighted F1:   {ensemble_metrics_check[1]:.4f}"
)

print(
    f"Macro OvR AUC: {ensemble_metrics_check[2]:.4f}"
)

print(
    f"Micro OvR AUC: {ensemble_metrics_check[3]:.4f}"
)

assert np.allclose(
    ensemble_metrics_check,
    [0.9386, 0.9374, 0.9951, 0.9971],
    atol=5e-5
), (
    "Ensemble metrics do not match the finalized "
    "HAM10000 results."
)

print(
    "\nPASS: Prediction objects and metric helper are "
    "consistent with the finalized HAM10000 evaluation."
)

### Corrected Bootstrap Confidence Intervals for Supplementary Table S1

For consistency between the main manuscript and supplementary material,
point estimates are calculated directly on the complete evaluation set.
Bootstrap resampling is used exclusively to estimate the 95% confidence
limits. This avoids reporting the mean of bootstrap replicates as the
point estimate.

Each metric is bootstrapped separately with 1,000 resamples, using `numpy.random.default_rng` seeded from a base seed of 42: accuracy 42, weighted F1 43, macro ROC-AUC 44 and micro ROC-AUC 45. A resample on which a metric is undefined is excluded. These are the intervals reported in Supplementary Table S1.


In [ ]:
# Direct point estimate + bootstrap 95% CI

def bootstrap_metric_ci_corrected(
    y,
    probs,
    metric_fn,
    B=1000,
    alpha=0.05,
    seed=42
):
    """
    Return:
        point estimate calculated on the complete evaluation set,
        lower bootstrap percentile,
        upper bootstrap percentile.
    """

    y = np.asarray(y)
    probs = np.asarray(probs)

    assert len(y) == len(probs)

    # Direct point estimate on the complete evaluation set
    point_estimate = metric_fn(y, probs)

    # Bootstrap only for confidence limits
    rng = np.random.default_rng(seed)

    n = len(y)
    bootstrap_stats = []

    for _ in range(B):

        idx = rng.integers(
            0,
            n,
            size=n
        )

        try:
            value = metric_fn(
                y[idx],
                probs[idx]
            )
        except Exception:
            value = np.nan

        bootstrap_stats.append(value)

    bootstrap_stats = np.asarray(
        bootstrap_stats,
        dtype=float
    )

    lo = np.nanpercentile(
        bootstrap_stats,
        100 * alpha / 2
    )

    hi = np.nanpercentile(
        bootstrap_stats,
        100 * (1 - alpha / 2)
    )

    return (
        float(point_estimate),
        float(lo),
        float(hi)
    )


def metric_bundle_corrected(
    y,
    probs,
    B=1000,
    seed=42
):

    def m_acc(yy, pp):
        return accuracy_score(
            yy,
            pp.argmax(axis=1)
        )

    def m_f1w(yy, pp):
        return f1_score(
            yy,
            pp.argmax(axis=1),
            average="weighted"
        )

    def m_auc_macro(yy, pp):
        return roc_auc_score(
            yy,
            pp,
            multi_class="ovr",
            average="macro"
        )

    def m_auc_micro(yy, pp):
        return roc_auc_score(
            yy,
            pp,
            multi_class="ovr",
            average="micro"
        )

    acc = bootstrap_metric_ci_corrected(
        y, probs, m_acc,
        B=B,
        seed=seed
    )

    f1w = bootstrap_metric_ci_corrected(
        y, probs, m_f1w,
        B=B,
        seed=seed + 1
    )

    aucM = bootstrap_metric_ci_corrected(
        y, probs, m_auc_macro,
        B=B,
        seed=seed + 2
    )

    aucm = bootstrap_metric_ci_corrected(
        y, probs, m_auc_micro,
        B=B,
        seed=seed + 3
    )

    return {
        "Accuracy": acc,
        "F1 (weighted)": f1w,
        "AUC (macro, OvR)": aucM,
        "AUC (micro, OvR)": aucm,
    }

print("Corrected bootstrap functions defined.")

### Restore Canonical Prediction Arrays from Permanent Kaggle Dataset

In [ ]:
# Restore Canonical Prediction Arrays from Permanent Kaggle Dataset

# Find the permanent Kaggle dataset path automatically
candidates = glob.glob(
    "/kaggle/input/**/canonical_ham10000_labels.npy",
    recursive=True
)

assert candidates, \
    "Could not find canonical_ham10000_labels.npy under /kaggle/input."

LABEL_PATH = candidates[0]
BASE = os.path.dirname(LABEL_PATH)

print("Prediction archive found at:")
print(BASE)

# Load labels
labels = np.load(
    os.path.join(
        BASE,
        "canonical_ham10000_labels.npy"
    )
)

# Canonical model order
model_list = [
    "cnn",
    "resnet50",
    "densenet121",
    "efficientnet_b3",
    "convnext_tiny",
    "mobilenet_v3",
    "vit",
]

# Load model probability arrays
all_probs = {
    "cnn": np.load(
        os.path.join(BASE, "canonical_cnn_probs.npy")
    ),

    "resnet50": np.load(
        os.path.join(BASE, "canonical_resnet50_probs.npy")
    ),

    "densenet121": np.load(
        os.path.join(BASE, "canonical_densenet121_probs.npy")
    ),

    "efficientnet_b3": np.load(
        os.path.join(BASE, "canonical_efficientnet_b3_probs.npy")
    ),

    "convnext_tiny": np.load(
        os.path.join(BASE, "canonical_convnext_tiny_probs.npy")
    ),

    "mobilenet_v3": np.load(
        os.path.join(BASE, "canonical_mobilenet_v3_probs.npy")
    ),

    "vit": np.load(
        os.path.join(BASE, "canonical_vit_probs.npy")
    ),

    "ensemble_mean": np.load(
        os.path.join(BASE, "canonical_ensemble_probs.npy")
    ),
}

# Safety checks
assert labels.shape == (2003,), \
    f"Unexpected label shape: {labels.shape}"

for key in model_list + ["ensemble_mean"]:

    assert all_probs[key].shape == (2003, 7), \
        f"{key}: unexpected shape {all_probs[key].shape}"

    assert np.isfinite(all_probs[key]).all(), \
        f"{key}: non-finite probabilities detected"

    assert np.allclose(
        all_probs[key].sum(axis=1),
        1.0,
        atol=1e-5
    ), f"{key}: probabilities do not sum to 1"

print("\nCanonical prediction arrays restored successfully.")
print("Labels:", labels.shape)

for key in model_list + ["ensemble_mean"]:
    print(
        f"{key:20s}: "
        f"{all_probs[key].shape}"
    )

### Restore Historical/Canonical Prediction Arrays After Session Restart

The current Kaggle session no longer contains the in-memory prediction objects
used by the bootstrap analysis. Before recomputing any confidence intervals,
the previously saved prediction arrays are restored from `/kaggle/working`
when available.

In [ ]:
# Verify Permanent Canonical Prediction Files

saved_prediction_files = {
    "labels": "/kaggle/input/datasets/nhsaykat/ham10000-canonical-prediction-arrays/canonical_ham10000_labels.npy",
    "cnn": "/kaggle/input/datasets/nhsaykat/ham10000-canonical-prediction-arrays/canonical_cnn_probs.npy",
    "resnet50": "/kaggle/input/datasets/nhsaykat/ham10000-canonical-prediction-arrays/canonical_resnet50_probs.npy",
    "densenet121": "/kaggle/input/datasets/nhsaykat/ham10000-canonical-prediction-arrays/canonical_densenet121_probs.npy",
    "efficientnet_b3": "/kaggle/input/datasets/nhsaykat/ham10000-canonical-prediction-arrays/canonical_efficientnet_b3_probs.npy",
    "convnext_tiny": "/kaggle/input/datasets/nhsaykat/ham10000-canonical-prediction-arrays/canonical_convnext_tiny_probs.npy",
    "mobilenet_v3": "/kaggle/input/datasets/nhsaykat/ham10000-canonical-prediction-arrays/canonical_mobilenet_v3_probs.npy",
    "vit": "/kaggle/input/datasets/nhsaykat/ham10000-canonical-prediction-arrays/canonical_vit_probs.npy",
    "ensemble_mean": "/kaggle/input/datasets/nhsaykat/ham10000-canonical-prediction-arrays/canonical_ensemble_probs.npy",
}

print("Permanent canonical prediction files:")

for name, path in saved_prediction_files.items():
    print(
        f"{name:20s}: "
        f"{'FOUND' if os.path.isfile(path) else 'MISSING'}"
    )

### Audit Prediction Sources Used for the Revised Bootstrap Analysis

In [ ]:
# Audit Prediction Sources Used for the Revised Bootstrap Analysis

# Required-object checks
required_objects = [
    "labels",
    "all_probs",
    "model_list",
]

print("Revised bootstrap input objects:")

for name in required_objects:
    print(
        f"{name:20s}: "
        f"{'OK' if name in globals() else 'MISSING'}"
    )

missing = [
    name
    for name in required_objects
    if name not in globals()
]

assert not missing, (
    "Missing revised bootstrap inputs: "
    + ", ".join(missing)
)

# Convert labels to NumPy and inspect structure
labels_arr = np.asarray(labels)

print("\nLabels")
print("------")
print("Shape:", labels_arr.shape)
print("N:", len(labels_arr))
print("Unique classes:", np.unique(labels_arr))

assert labels_arr.shape == (2003,), \
    f"Unexpected label shape: {labels_arr.shape}"

assert set(np.unique(labels_arr)) == set(range(7)), \
    f"Unexpected label values: {np.unique(labels_arr)}"

# Inspect model keys
print("\nmodel_list")
print("----------")
print(model_list)

expected_model_list = [
    "cnn",
    "resnet50",
    "densenet121",
    "efficientnet_b3",
    "convnext_tiny",
    "mobilenet_v3",
    "vit",
]

assert list(model_list) == expected_model_list, (
    "Unexpected model_list order:\n"
    f"{model_list}"
)

print("\nall_probs keys")
print("--------------")
print(list(all_probs.keys()))

# Validate each probability matrix
all_keys = list(model_list) + ["ensemble_mean"]

print("\nProbability-array validation")
print("----------------------------")

for key in all_keys:

    assert key in all_probs, \
        f"Missing probability array: {key}"

    p = np.asarray(all_probs[key])

    assert p.shape == (2003, 7), \
        f"{key}: unexpected shape {p.shape}"

    assert np.isfinite(p).all(), \
        f"{key}: non-finite values detected"

    assert np.all(p >= 0), \
        f"{key}: negative probability values detected"

    assert np.allclose(
        p.sum(axis=1),
        1.0,
        atol=1e-5
    ), f"{key}: probability rows do not sum to 1"

    print(
        f"{key:20s} "
        f"shape={p.shape}  "
        f"finite=True  "
        f"row_sums_ok=True"
    )

# Direct point estimates BEFORE bootstrap
print("\nDirect point estimates from restored canonical predictions")
print("----------------------------------------------------------")

audit_rows = []

for key in all_keys:

    p = np.asarray(all_probs[key])
    pred = p.argmax(axis=1)

    acc = accuracy_score(
        labels_arr,
        pred
    )

    f1w = f1_score(
        labels_arr,
        pred,
        average="weighted"
    )

    macro_auc = roc_auc_score(
        labels_arr,
        p,
        multi_class="ovr",
        average="macro"
    )

    micro_auc = roc_auc_score(
        labels_arr,
        p,
        multi_class="ovr",
        average="micro"
    )

    audit_rows.append({
        "Model": key,
        "Accuracy": acc,
        "Weighted F1": f1w,
        "Macro AUC": macro_auc,
        "Micro AUC": micro_auc,
    })

    print(
        f"{key:20s} "
        f"Accuracy={acc:.4f}  "
        f"Weighted F1={f1w:.4f}  "
        f"Macro AUC={macro_auc:.4f}  "
        f"Micro AUC={micro_auc:.4f}"
    )

# Build audit table
revised_prediction_audit_df = pd.DataFrame(
    audit_rows
)

print("\nRevised prediction-source audit table:")
display(
    revised_prediction_audit_df.round(4)
)

# Critical reproducibility checks
vit_row = revised_prediction_audit_df[
    revised_prediction_audit_df["Model"] == "vit"
].iloc[0]

ensemble_row = revised_prediction_audit_df[
    revised_prediction_audit_df["Model"] == "ensemble_mean"
].iloc[0]

assert np.isclose(
    vit_row["Accuracy"],
    0.9591,
    atol=5e-5
), (
    "ViT accuracy does not match the previously verified "
    f"canonical result: {vit_row['Accuracy']:.6f}"
)

assert np.isclose(
    vit_row["Weighted F1"],
    0.9585,
    atol=5e-5
), (
    "ViT weighted F1 does not match the previously verified "
    f"canonical result: {vit_row['Weighted F1']:.6f}"
)

print("\nCritical checks")
print("---------------")
print(
    f"ViT accuracy verified:        "
    f"{vit_row['Accuracy']:.4f}"
)
print(
    f"ViT weighted F1 verified:     "
    f"{vit_row['Weighted F1']:.4f}"
)
print(
    f"Ensemble accuracy restored:   "
    f"{ensemble_row['Accuracy']:.4f}"
)
print(
    f"Ensemble weighted F1 restored:"
    f" {ensemble_row['Weighted F1']:.4f}"
)

# ------------------------------------------------------------
# 8. Save audit result
# ------------------------------------------------------------
audit_output_path = (
    "/kaggle/working/"
    "revised_prediction_source_audit.csv"
)

revised_prediction_audit_df.to_csv(
    audit_output_path,
    index=False
)

print("\nSaved revised prediction-source audit to:")
print(audit_output_path)

### Generate Corrected 95% Confidence Intervals

The verified canonical prediction arrays are used to compute the revised
confidence-interval table. For each metric, the point estimate is calculated
directly on the complete 2,003-image evaluation set, while nonparametric
bootstrap resampling is used only to estimate the 95% confidence limits.

This ensures that the point estimates reported in the confidence-interval
table are identical to the corresponding direct results reported in the
main performance table.

In [ ]:
# Generate Corrected 95% Confidence Intervals

# Safety checks
assert "metric_bundle_corrected" in globals(), \
    "Run the corrected bootstrap-function cell first."

assert len(labels) == 2003
assert len(model_list) == 7

display_names = {
    "cnn": "CNN",
    "resnet50": "ResNet-50",
    "densenet121": "DenseNet-121",
    "efficientnet_b3": "EfficientNet-B3",
    "convnext_tiny": "ConvNeXt-Tiny",
    "mobilenet_v3": "MobileNetV3-L",
    "vit": "ViT",
    "ensemble_mean": "Ensemble",
}

corrected_ci_rows = []

for key in model_list + ["ensemble_mean"]:

    print(f"Bootstrapping {display_names[key]}...")

    bundle = metric_bundle_corrected(
        labels,
        all_probs[key],
        B=1000,
        seed=42
    )

    corrected_ci_rows.append({
        "Model": display_names[key],

        "Accuracy": (
            f"{bundle['Accuracy'][0]:.4f} "
            f"[{bundle['Accuracy'][1]:.4f}, "
            f"{bundle['Accuracy'][2]:.4f}]"
        ),

        "F1 (weighted)": (
            f"{bundle['F1 (weighted)'][0]:.4f} "
            f"[{bundle['F1 (weighted)'][1]:.4f}, "
            f"{bundle['F1 (weighted)'][2]:.4f}]"
        ),

        "AUC (macro, OvR)": (
            f"{bundle['AUC (macro, OvR)'][0]:.4f} "
            f"[{bundle['AUC (macro, OvR)'][1]:.4f}, "
            f"{bundle['AUC (macro, OvR)'][2]:.4f}]"
        ),

        "AUC (micro, OvR)": (
            f"{bundle['AUC (micro, OvR)'][0]:.4f} "
            f"[{bundle['AUC (micro, OvR)'][1]:.4f}, "
            f"{bundle['AUC (micro, OvR)'][2]:.4f}]"
        ),
    })

corrected_ci_table = pd.DataFrame(corrected_ci_rows)

print("\nCorrected 95% CI table:")
display(corrected_ci_table)

# Consistency checks against direct point estimates
direct_lookup = (
    revised_prediction_audit_df
    .set_index("Model")
)

for key in model_list + ["ensemble_mean"]:

    direct_acc = float(
        direct_lookup.loc[key, "Accuracy"]
    )

    ci_acc_point = float(
        corrected_ci_table.loc[
            corrected_ci_table["Model"] == display_names[key],
            "Accuracy"
        ]
        .iloc[0]
        .split()[0]
    )

    assert np.isclose(
        direct_acc,
        ci_acc_point,
        atol=5e-5
    ), (
        f"{key}: direct accuracy and CI-table point estimate differ "
        f"({direct_acc:.6f} vs {ci_acc_point:.6f})"
    )

print(
    "\nConsistency check passed: "
    "all CI-table accuracy point estimates match "
    "the direct canonical estimates."
)

# Save
corrected_ci_path = (
    "/kaggle/working/"
    "metrics_with_95CI_corrected.csv"
)

corrected_ci_table.to_csv(
    corrected_ci_path,
    index=False
)

print("\nSaved corrected CI table to:")
print(corrected_ci_path)

### Lock Final Revised HAM10000 Performance Tables

The revised main and supplementary performance tables are generated from the
same permanently archived prediction arrays and the same 2,003-image
harmonized HAM10000 evaluation cohort. Table 1 contains the direct point
estimates, while Supplementary Table S1 reports those identical point estimates
together with bootstrap 95% confidence intervals.

This ensures that no separate prediction state, evaluation cohort, or bootstrap
mean is used to generate the supplementary point estimates.

In [ ]:
# Lock Final Revised Table 1 and Supplementary Table S1

# Safety checks
assert "revised_prediction_audit_df" in globals(), \
    "revised_prediction_audit_df is missing."

assert "corrected_ci_table" in globals(), \
    "corrected_ci_table is missing."

# Revised Table 1: direct point estimates
table1_name_map = {
    "cnn": "CNN",
    "resnet50": "ResNet-50",
    "densenet121": "DenseNet-121",
    "efficientnet_b3": "EfficientNet-B3",
    "convnext_tiny": "ConvNeXt-Tiny",
    "mobilenet_v3": "MobileNetV3-L",
    "vit": "ViT",
    "ensemble_mean": "Ensemble",
}

table1_revised = revised_prediction_audit_df.copy()

table1_revised["Model"] = (
    table1_revised["Model"]
    .map(table1_name_map)
)

assert table1_revised["Model"].notna().all(), \
    "Unmapped model name detected in Table 1."

table1_revised = table1_revised.rename(
    columns={
        "Weighted F1": "F1 (weighted)",
        "Macro AUC": "AUC (macro, OvR)",
        "Micro AUC": "AUC (micro, OvR)",
    }
)

table1_revised = table1_revised[
    [
        "Model",
        "Accuracy",
        "F1 (weighted)",
        "AUC (macro, OvR)",
        "AUC (micro, OvR)",
    ]
].copy()

# Lock intended order
final_model_order = [
    "CNN",
    "ResNet-50",
    "DenseNet-121",
    "EfficientNet-B3",
    "ConvNeXt-Tiny",
    "MobileNetV3-L",
    "ViT",
    "Ensemble",
]

table1_revised["Model"] = pd.Categorical(
    table1_revised["Model"],
    categories=final_model_order,
    ordered=True
)

table1_revised = (
    table1_revised
    .sort_values("Model")
    .reset_index(drop=True)
)

table1_revised["Model"] = (
    table1_revised["Model"]
    .astype(str)
)

# Supplementary Table S1
# Same prediction source, with bootstrap 95% CIs
tableS1_revised = corrected_ci_table.copy()

# Lock same model order
tableS1_revised["Model"] = pd.Categorical(
    tableS1_revised["Model"],
    categories=final_model_order,
    ordered=True
)

tableS1_revised = (
    tableS1_revised
    .sort_values("Model")
    .reset_index(drop=True)
)

tableS1_revised["Model"] = (
    tableS1_revised["Model"]
    .astype(str)
)

# Verify exact model correspondence
assert (
    table1_revised["Model"].tolist()
    == tableS1_revised["Model"].tolist()
), "Table 1 and Table S1 model orders differ."

# Verify EVERY S1 point estimate equals Table 1
metric_pairs = [
    ("Accuracy", "Accuracy"),
    ("F1 (weighted)", "F1 (weighted)"),
    ("AUC (macro, OvR)", "AUC (macro, OvR)"),
    ("AUC (micro, OvR)", "AUC (micro, OvR)"),
]

consistency_rows = []

for col_t1, col_s1 in metric_pairs:

    for i in range(len(table1_revised)):

        model_name = table1_revised.loc[i, "Model"]

        direct_value = float(
            table1_revised.loc[i, col_t1]
        )

        s1_string = str(
            tableS1_revised.loc[i, col_s1]
        )

        s1_point = float(
            s1_string.split()[0]
        )

        matches = np.isclose(
            direct_value,
            s1_point,
            atol=5e-4
        )

        consistency_rows.append({
            "Model": model_name,
            "Metric": col_t1,
            "Table1_point_estimate": direct_value,
            "TableS1_point_estimate": s1_point,
            "Match": bool(matches),
        })

        assert matches, (
            f"Inconsistency for {model_name} / {col_t1}: "
            f"Table 1={direct_value:.6f}, "
            f"S1={s1_point:.6f}"
        )

consistency_df = pd.DataFrame(
    consistency_rows
)

print(
    "PASS: Every Table S1 point estimate "
    "matches Table 1."
)

# Critical editor-request checks
vit_t1 = float(
    table1_revised.loc[
        table1_revised["Model"] == "ViT",
        "Accuracy"
    ].iloc[0]
)

ensemble_t1 = float(
    table1_revised.loc[
        table1_revised["Model"] == "Ensemble",
        "Accuracy"
    ].iloc[0]
)

vit_s1 = float(
    tableS1_revised.loc[
        tableS1_revised["Model"] == "ViT",
        "Accuracy"
    ].iloc[0].split()[0]
)

ensemble_s1 = float(
    tableS1_revised.loc[
        tableS1_revised["Model"] == "Ensemble",
        "Accuracy"
    ].iloc[0].split()[0]
)

assert np.isclose(vit_t1, 0.9591, atol=5e-5)
assert np.isclose(vit_s1, 0.9591, atol=5e-4)

assert np.isclose(
    ensemble_t1,
    0.9386,
    atol=5e-5
)

assert np.isclose(
    ensemble_s1,
    0.9386,
    atol=5e-4
)

print("\nCritical final values verified:")
print(f"ViT accuracy:      {vit_t1:.4f}")
print(f"Ensemble accuracy: {ensemble_t1:.4f}")

# Display final tables
print("\nFINAL REVISED TABLE 1")
print("---------------------")

display(
    table1_revised.round(4)
)

print("\nFINAL REVISED SUPPLEMENTARY TABLE S1")
print("------------------------------------")

display(
    tableS1_revised
)

# Save final revision artifacts
OUTPUT_DIR = "/kaggle/working"

TABLE1_PATH = os.path.join(
    OUTPUT_DIR,
    "Table1_HAM10000_revised_final.csv"
)

S1_PATH = os.path.join(
    OUTPUT_DIR,
    "TableS1_HAM10000_revised_final.csv"
)

CONSISTENCY_PATH = os.path.join(
    OUTPUT_DIR,
    "Table1_TableS1_consistency_audit.csv"
)

table1_revised.to_csv(
    TABLE1_PATH,
    index=False
)

tableS1_revised.to_csv(
    S1_PATH,
    index=False
)

consistency_df.to_csv(
    CONSISTENCY_PATH,
    index=False
)

print("\nSaved final revision artifacts:")
print(TABLE1_PATH)
print(S1_PATH)
print(CONSISTENCY_PATH)

### Regenerate HAM10000 Model-Comparison Figures from Final Revised Table 1

In [ ]:
# Regenerate Final HAM10000 Model-Comparison Figures
# Source: locked revised Table 1

# Use the FINAL locked Table 1 as the only figure source
assert "table1_revised" in globals(), \
    "Run the 'Lock Final Revised Table 1 and Supplementary Table S1' cell first."

df = table1_revised.copy()

required_cols = [
    "Model",
    "Accuracy",
    "F1 (weighted)",
    "AUC (macro, OvR)",
    "AUC (micro, OvR)",
]

assert all(c in df.columns for c in required_cols), \
    f"Missing required columns: {df.columns.tolist()}"

final_order = [
    "CNN",
    "ResNet-50",
    "DenseNet-121",
    "EfficientNet-B3",
    "ConvNeXt-Tiny",
    "MobileNetV3-L",
    "ViT",
    "Ensemble",
]

assert set(df["Model"]) == set(final_order), \
    f"Unexpected model names: {df['Model'].tolist()}"

df["Model"] = pd.Categorical(
    df["Model"],
    categories=final_order,
    ordered=True
)

df = (
    df
    .sort_values("Model")
    .reset_index(drop=True)
)

df["Model"] = df["Model"].astype(str)

# Critical consistency checks
def get_value(model, metric):
    return float(
        df.loc[df["Model"] == model, metric].iloc[0]
    )

assert np.isclose(
    get_value("ViT", "Accuracy"),
    0.9591,
    atol=5e-5
)

assert np.isclose(
    get_value("Ensemble", "Accuracy"),
    0.9386,
    atol=5e-5
)

assert np.isclose(
    get_value("ConvNeXt-Tiny", "Accuracy"),
    0.9126,
    atol=5e-5
)

assert np.isclose(
    get_value("ConvNeXt-Tiny", "F1 (weighted)"),
    0.9124,
    atol=5e-5
)

print("Final figure-source metrics verified.")

# Plot settings
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 16,
    "axes.labelsize": 12,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "axes.facecolor": "white",
})

def _pct_fmt(x, _):
    return f"{x:.0%}"

def _annotate(ax, bars, decimals=3):
    for b in bars:
        h = b.get_height()
        ax.annotate(
            f"{h:.{decimals}f}",
            xy=(
                b.get_x() + b.get_width() / 2,
                h
            ),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=8.5
        )

def _finish(ax, title, ylabel, ymin, ymax):
    ax.set_title(
        title,
        pad=10,
        weight="semibold"
    )
    ax.set_ylabel(ylabel)
    ax.yaxis.set_major_formatter(
        FuncFormatter(_pct_fmt)
    )
    ax.set_ylim(ymin, ymax)
    ax.grid(
        axis="y",
        linestyle="--",
        linewidth=0.6,
        alpha=0.35
    )
    ax.set_axisbelow(True)

    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

def _mark_ensemble(bars, labels):
    """
    Distinguish the proposed ensemble visually without implying
    that it is the best-performing method.
    """
    if "Ensemble" in labels:
        i = labels.index("Ensemble")
        bars[i].set_linewidth(1.5)
        bars[i].set_edgecolor("black")
        bars[i].set_hatch("//")

# Figure 1: Accuracy and weighted F1
x = np.arange(len(df))
width = 0.34

fig1, ax1 = plt.subplots(
    figsize=(11.2, 4.6),
    constrained_layout=True
)

b1 = ax1.bar(
    x - width / 2,
    df["Accuracy"].values,
    width,
    label="Accuracy"
)

b2 = ax1.bar(
    x + width / 2,
    df["F1 (weighted)"].values,
    width,
    label="F1 (weighted)"
)

_mark_ensemble(
    b1,
    df["Model"].tolist()
)

_mark_ensemble(
    b2,
    df["Model"].tolist()
)

ax1.set_xticks(
    x,
    df["Model"].values,
    rotation=14,
    ha="right"
)

_finish(
    ax1,
    "HAM10000 Model Comparison - Accuracy and Weighted F1",
    "Score",
    ymin=0.60,
    ymax=1.005
)

_annotate(ax1, b1, decimals=3)
_annotate(ax1, b2, decimals=3)

ax1.legend(
    loc="upper left",
    frameon=False,
    ncol=2
)

FIG1_PNG = (
    "/kaggle/working/"
    "model_comp_accuracy_f1_revised_final.png"
)

FIG1_SVG = (
    "/kaggle/working/"
    "model_comp_accuracy_f1_revised_final.svg"
)

FIG1_PDF = (
    "/kaggle/working/"
    "model_comp_accuracy_f1_revised_final.pdf"
)

fig1.savefig(
    FIG1_PNG,
    bbox_inches="tight"
)

fig1.savefig(
    FIG1_SVG,
    bbox_inches="tight"
)

fig1.savefig(
    FIG1_PDF,
    bbox_inches="tight"
)

plt.show()

# Figure 2: Macro and micro ROC-AUC
fig2, ax2 = plt.subplots(
    figsize=(11.2, 4.6),
    constrained_layout=True
)

b3 = ax2.bar(
    x - width / 2,
    df["AUC (macro, OvR)"].values,
    width,
    label="AUC (macro, OvR)"
)

b4 = ax2.bar(
    x + width / 2,
    df["AUC (micro, OvR)"].values,
    width,
    label="AUC (micro, OvR)"
)

_mark_ensemble(
    b3,
    df["Model"].tolist()
)

_mark_ensemble(
    b4,
    df["Model"].tolist()
)

ax2.set_xticks(
    x,
    df["Model"].values,
    rotation=14,
    ha="right"
)

_finish(
    ax2,
    "HAM10000 Model Comparison - Macro and Micro ROC-AUC",
    "ROC-AUC",
    ymin=0.90,
    ymax=1.005
)

_annotate(ax2, b3, decimals=3)
_annotate(ax2, b4, decimals=3)

ax2.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, 1.17),
    ncol=2,
    frameon=False
)

FIG2_PNG = (
    "/kaggle/working/"
    "model_comp_auc_revised_final.png"
)

FIG2_SVG = (
    "/kaggle/working/"
    "model_comp_auc_revised_final.svg"
)

FIG2_PDF = (
    "/kaggle/working/"
    "model_comp_auc_revised_final.pdf"
)

fig2.savefig(
    FIG2_PNG,
    bbox_inches="tight"
)

fig2.savefig(
    FIG2_SVG,
    bbox_inches="tight"
)

fig2.savefig(
    FIG2_PDF,
    bbox_inches="tight"
)

plt.show()

# Save exact figure-source data
FIGURE_DATA_PATH = (
    "/kaggle/working/"
    "HAM10000_final_figure_source_metrics.csv"
)

df.to_csv(
    FIGURE_DATA_PATH,
    index=False
)

print("\nSaved revised final figures:")
print(FIG1_PNG)
print(FIG1_SVG)
print(FIG1_PDF)

print(FIG2_PNG)
print(FIG2_SVG)
print(FIG2_PDF)

print("\nSaved exact figure-source metrics:")
print(FIGURE_DATA_PATH)

### McNemar’s Test

In [ ]:
# Revised McNemar's Test
# Ensemble vs Each Individual Model
# Using Final Canonical HAM10000 Predictions

# Safety checks
assert "labels" in globals(), \
    "labels is missing. Restore the canonical prediction arrays first."

assert "all_probs" in globals(), \
    "all_probs is missing. Restore the canonical prediction arrays first."

assert "model_list" in globals(), \
    "model_list is missing. Restore the canonical model list first."

labels_arr = np.asarray(labels)

assert labels_arr.shape == (2003,), \
    f"Unexpected labels shape: {labels_arr.shape}"

expected_models = [
    "cnn",
    "resnet50",
    "densenet121",
    "efficientnet_b3",
    "convnext_tiny",
    "mobilenet_v3",
    "vit",
]

assert list(model_list) == expected_models, \
    f"Unexpected model_list: {model_list}"

assert "ensemble_mean" in all_probs, \
    "ensemble_mean probabilities are missing."

# Reconstruct predictions DIRECTLY from canonical
# probability arrays

all_preds_revised = {}

for key in model_list + ["ensemble_mean"]:

    probs = np.asarray(all_probs[key])

    assert probs.shape == (2003, 7), \
        f"{key}: unexpected probability shape {probs.shape}"

    assert np.isfinite(probs).all(), \
        f"{key}: non-finite probability values detected"

    all_preds_revised[key] = probs.argmax(axis=1)

    assert all_preds_revised[key].shape == (2003,), \
        f"{key}: unexpected prediction shape"

print("Canonical class predictions reconstructed successfully.")

# Verify predictions reproduce FINAL Table 1 accuracy
expected_accuracy = {
    "cnn": 0.7898,
    "resnet50": 0.6850,
    "densenet121": 0.8787,
    "efficientnet_b3": 0.8997,
    "convnext_tiny": 0.9126,
    "mobilenet_v3": 0.8587,
    "vit": 0.9591,
    "ensemble_mean": 0.9386,
}

print("\nAccuracy verification")
print("---------------------")

for key in model_list + ["ensemble_mean"]:

    acc = np.mean(
        all_preds_revised[key] == labels_arr
    )

    print(
        f"{key:20s}: "
        f"{acc:.4f}"
    )

    assert np.isclose(
        acc,
        expected_accuracy[key],
        atol=5e-5
    ), (
        f"{key}: reconstructed prediction accuracy "
        f"{acc:.6f} does not match final Table 1 "
        f"value {expected_accuracy[key]:.4f}"
    )

print(
    "\nPASS: Predictions reproduce all final "
    "Table 1 accuracies."
)

# McNemar tests
ensemble_pred = all_preds_revised["ensemble_mean"]
ensemble_correct = (
    ensemble_pred == labels_arr
)

display_names = {
    "cnn": "CNN",
    "resnet50": "ResNet-50",
    "densenet121": "DenseNet-121",
    "efficientnet_b3": "EfficientNet-B3",
    "convnext_tiny": "ConvNeXt-Tiny",
    "mobilenet_v3": "MobileNetV3-L",
    "vit": "ViT",
}

mcnemar_rows = []

for key in model_list:

    model_pred = all_preds_revised[key]

    model_correct = (
        model_pred == labels_arr
    )

    # Discordant pairs
    #
    # b = ensemble correct, individual model wrong
    # c = ensemble wrong, individual model correct

    b = int(
        np.sum(
            ensemble_correct
            & ~model_correct
        )
    )

    c = int(
        np.sum(
            ~ensemble_correct
            & model_correct
        )
    )

    discordant = b + c

    # Full 2x2 correctness contingency table
    both_correct = int(
        np.sum(
            ensemble_correct
            & model_correct
        )
    )

    both_wrong = int(
        np.sum(
            ~ensemble_correct
            & ~model_correct
        )
    )

    table = [
        [both_correct, b],
        [c, both_wrong],
    ]

    # Exact McNemar for small number of discordant pairs;
    # otherwise chi-square approximation with continuity correction.
    use_exact = discordant < 25

    result = mcnemar(
        table,
        exact=use_exact,
        correction=not use_exact
    )

    mcnemar_rows.append({
        "Comparison": (
            f"Ensemble vs {display_names[key]}"
        ),
        "b (Ens correct, Model wrong)": b,
        "c (Ens wrong, Model correct)": c,
        "Discordant pairs (b+c)": discordant,
        "Method": (
            "Exact"
            if use_exact
            else "Chi-square with continuity correction"
        ),
        "McNemar p-value": float(result.pvalue),
    })

# Build revised McNemar table
mcnemar_revised_df = pd.DataFrame(
    mcnemar_rows
)

print("\nFINAL REVISED McNEMAR RESULTS")
print("-----------------------------")

display(
    mcnemar_revised_df
)

# Additional critical check:
# Ensemble vs ViT direction

vit_pred = all_preds_revised["vit"]

ensemble_accuracy = np.mean(
    ensemble_pred == labels_arr
)

vit_accuracy = np.mean(
    vit_pred == labels_arr
)

print("\nCritical Ensemble-vs-ViT check")
print("------------------------------")

print(
    f"Ensemble accuracy: {ensemble_accuracy:.4f}"
)

print(
    f"ViT accuracy:      {vit_accuracy:.4f}"
)

print(
    f"Difference "
    f"(Ensemble - ViT): "
    f"{ensemble_accuracy - vit_accuracy:.4f}"
)

assert ensemble_accuracy < vit_accuracy, (
    "Unexpected result: final Table 1 requires "
    "ViT accuracy to exceed ensemble accuracy."
)

# Save revised results
OUTPUT_PATH = (
    "/kaggle/working/"
    "mcnemar_ensemble_vs_models_revised_final.csv"
)

mcnemar_revised_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\nSaved revised McNemar results to:")
print(OUTPUT_PATH)

### Δ-Metrics (Ensemble − Model)

In [ ]:
# Revised Paired Bootstrap Differences
# Ensemble - Individual Model
# Final Canonical HAM10000 Evaluation

# Safety checks
assert "labels" in globals(), \
    "labels is missing. Restore canonical prediction arrays first."

assert "all_probs" in globals(), \
    "all_probs is missing. Restore canonical prediction arrays first."

assert "model_list" in globals(), \
    "model_list is missing."

labels_arr = np.asarray(labels)

assert labels_arr.shape == (2003,), \
    f"Unexpected labels shape: {labels_arr.shape}"

expected_models = [
    "cnn",
    "resnet50",
    "densenet121",
    "efficientnet_b3",
    "convnext_tiny",
    "mobilenet_v3",
    "vit",
]

assert list(model_list) == expected_models, \
    f"Unexpected model_list: {model_list}"

all_keys = model_list + ["ensemble_mean"]

for key in all_keys:

    assert key in all_probs, \
        f"Missing probability array: {key}"

    p = np.asarray(all_probs[key])

    assert p.shape == (2003, 7), \
        f"{key}: unexpected shape {p.shape}"

    assert np.isfinite(p).all(), \
        f"{key}: non-finite probabilities detected"

print(
    "Canonical prediction arrays verified "
    "for paired-bootstrap analysis."
)

# Metric functions
def metric_accuracy(y, probs):

    return accuracy_score(
        y,
        probs.argmax(axis=1)
    )


def metric_f1_weighted(y, probs):

    return f1_score(
        y,
        probs.argmax(axis=1),
        average="weighted"
    )


def metric_auc_macro(y, probs):

    return roc_auc_score(
        y,
        probs,
        multi_class="ovr",
        average="macro"
    )

# Correct paired-bootstrap function
def paired_bootstrap_diff_corrected(
    y,
    probs_ensemble,
    probs_model,
    metric_fn,
    B=1000,
    alpha=0.05,
    seed=42
):
    """
    Difference is defined as:

        Ensemble - Individual Model

    Point estimate:
        Direct metric difference on the complete evaluation set.

    Confidence interval:
        Percentile 95% CI from paired bootstrap resampling.

    The SAME bootstrap indices are applied to the ensemble
    and individual model, preserving the paired design.
    """

    y = np.asarray(y)
    probs_ensemble = np.asarray(probs_ensemble)
    probs_model = np.asarray(probs_model)

    assert len(y) == len(probs_ensemble)
    assert len(y) == len(probs_model)

    # Direct full-sample point estimate
    ensemble_metric = metric_fn(
        y,
        probs_ensemble
    )

    model_metric = metric_fn(
        y,
        probs_model
    )

    point_estimate = (
        ensemble_metric - model_metric
    )

    # Paired bootstrap for CI only
    rng = np.random.default_rng(seed)

    n = len(y)

    bootstrap_diffs = []

    for _ in range(B):

        idx = rng.integers(
            0,
            n,
            size=n
        )

        try:

            ensemble_boot = metric_fn(
                y[idx],
                probs_ensemble[idx]
            )

            model_boot = metric_fn(
                y[idx],
                probs_model[idx]
            )

            diff = (
                ensemble_boot
                - model_boot
            )

        except Exception:

            diff = np.nan

        bootstrap_diffs.append(
            diff
        )

    bootstrap_diffs = np.asarray(
        bootstrap_diffs,
        dtype=float
    )

    lo = np.nanpercentile(
        bootstrap_diffs,
        100 * alpha / 2
    )

    hi = np.nanpercentile(
        bootstrap_diffs,
        100 * (1 - alpha / 2)
    )

    return (
        float(point_estimate),
        float(lo),
        float(hi)
    )

# Display names
display_names = {
    "cnn": "CNN",
    "resnet50": "ResNet-50",
    "densenet121": "DenseNet-121",
    "efficientnet_b3": "EfficientNet-B3",
    "convnext_tiny": "ConvNeXt-Tiny",
    "mobilenet_v3": "MobileNetV3-L",
    "vit": "ViT",
}

# Metrics to compare
metric_functions = [
    (
        r"ΔAccuracy",
        metric_accuracy
    ),
    (
        r"ΔF1 (weighted)",
        metric_f1_weighted
    ),
    (
        r"ΔAUC (macro, OvR)",
        metric_auc_macro
    ),
]

# Run paired bootstrap
paired_rows = []

for model_index, key in enumerate(model_list):

    print(
        f"Bootstrapping Ensemble vs "
        f"{display_names[key]}..."
    )

    for metric_index, (metric_name, metric_fn) \
            in enumerate(metric_functions):

        # Deterministic but distinct seed for each
        # model/metric combination
        seed = (
            42
            + model_index * 10
            + metric_index
        )

        estimate, lo, hi = (
            paired_bootstrap_diff_corrected(
                labels_arr,
                all_probs["ensemble_mean"],
                all_probs[key],
                metric_fn,
                B=1000,
                alpha=0.05,
                seed=seed
            )
        )

        paired_rows.append({
            "Comparison": (
                f"Ensemble - {display_names[key]}"
            ),
            "Metric Δ": metric_name,
            "Estimate": estimate,
            "CI lower": lo,
            "CI upper": hi,
        })

# Create numerical audit DataFrame
paired_bootstrap_audit_df = pd.DataFrame(
    paired_rows
)

print(
    "\nREVISED PAIRED-BOOTSTRAP RESULTS "
    "(numerical audit)"
)
print("------------------------------------")

display(
    paired_bootstrap_audit_df.round(4)
)

# Create publication-formatted table
paired_bootstrap_final_df = (
    paired_bootstrap_audit_df[
        [
            "Comparison",
            "Metric Δ",
            "Estimate",
            "CI lower",
            "CI upper",
        ]
    ]
    .copy()
)

paired_bootstrap_final_df[
    "Estimate"
] = paired_bootstrap_final_df[
    "Estimate"
].map(
    lambda x: f"{x:.4f}"
)

paired_bootstrap_final_df[
    "95% CI"
] = paired_bootstrap_audit_df.apply(
    lambda row:
        f"[{row['CI lower']:.4f}, "
        f"{row['CI upper']:.4f}]",
    axis=1
)

paired_bootstrap_final_df = (
    paired_bootstrap_final_df[
        [
            "Comparison",
            "Metric Δ",
            "Estimate",
            "95% CI",
        ]
    ]
)

print(
    "\nFINAL REVISED PAIRED-BOOTSTRAP TABLE"
)
print("------------------------------------")

display(
    paired_bootstrap_final_df
)

# Critical consistency checks
# Direct Table 1 differences that MUST be reproduced

expected_direct_differences = {

    # Ensemble - ViT
    ("vit", "Accuracy"):
        0.9386 - 0.9591,

    ("vit", "F1"):
        0.9374 - 0.9585,

    ("vit", "AUC"):
        0.9951 - 0.9959,

    # Ensemble - ConvNeXt
    ("convnext_tiny", "Accuracy"):
        0.9386 - 0.9126,
}

# Get unrounded calculated estimates

def get_estimate(model_name, metric_name):

    comparison = (
        f"Ensemble - "
        f"{display_names[model_name]}"
    )

    row = paired_bootstrap_audit_df[
        (
            paired_bootstrap_audit_df[
                "Comparison"
            ] == comparison
        )
        &
        (
            paired_bootstrap_audit_df[
                "Metric Δ"
            ] == metric_name
        )
    ]

    assert len(row) == 1

    return float(
        row["Estimate"].iloc[0]
    )


vit_acc_diff = get_estimate(
    "vit",
    r"ΔAccuracy"
)

vit_f1_diff = get_estimate(
    "vit",
    r"ΔF1 (weighted)"
)

vit_auc_diff = get_estimate(
    "vit",
    r"ΔAUC (macro, OvR)"
)

convnext_acc_diff = get_estimate(
    "convnext_tiny",
    r"ΔAccuracy"
)

assert np.isclose(
    vit_acc_diff,
    expected_direct_differences[
        ("vit", "Accuracy")
    ],
    atol=1e-4
)

assert np.isclose(
    vit_f1_diff,
    expected_direct_differences[
        ("vit", "F1")
    ],
    atol=1e-4
)

assert np.isclose(
    vit_auc_diff,
    expected_direct_differences[
        ("vit", "AUC")
    ],
    atol=1e-4
)

assert np.isclose(
    convnext_acc_diff,
    expected_direct_differences[
        ("convnext_tiny", "Accuracy")
    ],
    atol=1e-4
)

print(
    "\nPASS: Paired-bootstrap point estimates "
    "are consistent with final Table 1."
)

print("\nCritical differences:")
print(
    f"Ensemble - ViT Accuracy: "
    f"{vit_acc_diff:.4f}"
)
print(
    f"Ensemble - ViT Weighted F1: "
    f"{vit_f1_diff:.4f}"
)
print(
    f"Ensemble - ViT Macro AUC: "
    f"{vit_auc_diff:.4f}"
)
print(
    f"Ensemble - ConvNeXt Accuracy: "
    f"{convnext_acc_diff:.4f}"
)

# Save final revision artifacts
OUTPUT_DIR = "/kaggle/working"

AUDIT_PATH = os.path.join(
    OUTPUT_DIR,
    "paired_bootstrap_differences_revised_audit.csv"
)

FINAL_PATH = os.path.join(
    OUTPUT_DIR,
    "paired_bootstrap_differences_revised_final.csv"
)

paired_bootstrap_audit_df.to_csv(
    AUDIT_PATH,
    index=False
)

paired_bootstrap_final_df.to_csv(
    FINAL_PATH,
    index=False
)

print("\nSaved:")
print(AUDIT_PATH)
print(FINAL_PATH)

### Optional Calibration Analysis — Expected Calibration Error

In [ ]:
# Expected Calibration Error (ECE)
# Standardized HAM10000 Evaluation Cohort

# Safety checks
assert "ham_labels" in globals(), \
    "ham_labels is missing."

assert "ham_all_probs" in globals(), \
    "ham_all_probs is missing. Run the HAM10000 prediction-preparation cell first."

assert "ham_model_list" in globals(), \
    "ham_model_list is missing."

assert ham_labels.shape == (2003,), \
    f"Unexpected HAM10000 label shape: {ham_labels.shape}"

# Multiclass Expected Calibration Error
def multiclass_ece(
    y_true,
    probs,
    n_bins=15
):
    """
    Multiclass Expected Calibration Error using
    maximum predicted probability (confidence) binning.

    Lower ECE indicates better calibration.
    """

    y_true = np.asarray(y_true)
    probs = np.asarray(probs)

    assert y_true.ndim == 1
    assert probs.ndim == 2
    assert probs.shape[0] == y_true.shape[0]
    assert probs.shape[1] == 7
    assert np.isfinite(probs).all()

    confidence = probs.max(axis=1)

    predictions = probs.argmax(
        axis=1
    )

    correct = (
        predictions == y_true
    ).astype(float)

    bin_edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1
    )

    ece = 0.0

    for bin_idx in range(n_bins):

        lower = bin_edges[
            bin_idx
        ]

        upper = bin_edges[
            bin_idx + 1
        ]

        # Include confidence == 0 in first bin
        if bin_idx == 0:

            mask = (
                (confidence >= lower)
                &
                (confidence <= upper)
            )

        else:

            mask = (
                (confidence > lower)
                &
                (confidence <= upper)
            )

        if not np.any(mask):
            continue

        bin_accuracy = correct[
            mask
        ].mean()

        bin_confidence = confidence[
            mask
        ].mean()

        bin_weight = mask.mean()

        ece += (
            bin_weight
            *
            abs(
                bin_accuracy
                -
                bin_confidence
            )
        )

    return float(ece)

# Compute ECE for all individual models + ensemble
calibration_rows = []

evaluation_models = (
    ham_model_list
    +
    ["Ensemble"]
)

for model_name in evaluation_models:

    probs = ham_all_probs[
        model_name
    ]

    ece = multiclass_ece(
        ham_labels,
        probs,
        n_bins=15
    )

    calibration_rows.append({
        "Model": model_name,
        "ECE": ece,
    })

# Build calibration table
ham_calibration_df = pd.DataFrame(
    calibration_rows
)

ham_calibration_df = (
    ham_calibration_df
    .sort_values(
        "ECE",
        ascending=True
    )
    .reset_index(drop=True)
)

display(
    ham_calibration_df.round(4)
)

# Save revised calibration table
CALIBRATION_CSV = (
    "/kaggle/working/"
    "HAM10000_calibration_ece_revised_final.csv"
)

ham_calibration_df.to_csv(
    CALIBRATION_CSV,
    index=False
)

print(
    "Saved revised HAM10000 ECE table:"
)

print(
    CALIBRATION_CSV
)

print(
    "\nNote: lower ECE indicates better calibration."
)

### Master HAM10000 Results Table with 95% Bootstrap Confidence Intervals

The master results table is Supplementary Table S1, computed above with the
per-metric bootstrap procedure. An earlier version of this cell repeated the
bootstrap with a single shared random stream, which gives F1 and ROC-AUC
intervals that differ from Table S1 in the third or fourth decimal place; it has
been replaced so that only one set of intervals exists.


In [ ]:
# Master HAM10000 results table = Supplementary Table S1

assert "tableS1_revised" in globals(), \
    "Run the 'Lock Final Revised Table 1 and Supplementary Table S1' cell first."

ham_master_df = tableS1_revised.rename(
    columns={
        "F1 (weighted)": "Weighted F1",
        "AUC (macro, OvR)": "Macro ROC-AUC (OvR)",
        "AUC (micro, OvR)": "Micro ROC-AUC (OvR)",
    }
)[["Model", "Accuracy", "Weighted F1", "Macro ROC-AUC (OvR)", "Micro ROC-AUC (OvR)"]]

print(ham_master_df.to_string(index=False))

MASTER_PATH = "/kaggle/working/HAM10000_master_results_with_95CI_revised_final.csv"
ham_master_df.to_csv(MASTER_PATH, index=False)
print("\nSaved revised HAM10000 master results table (identical to Supplementary Table S1):")
print(MASTER_PATH)


### Export Revised HAM10000 Master Results to LaTeX

In [ ]:
# Export Revised HAM10000 Master Results to LaTeX

assert "ham_master_df" in globals(), \
    "ham_master_df is missing. Run the revised master-results cell first."

# Rename columns for LaTeX presentation
ham_master_latex_df = ham_master_df.rename(
    columns={
        "Model": "Model",
        "Accuracy": "Accuracy (95\\% CI)",
        "Weighted F1": "Weighted F1 (95\\% CI)",
        "Macro ROC-AUC (OvR)": "Macro ROC--AUC (OvR, 95\\% CI)",
        "Micro ROC-AUC (OvR)": "Micro ROC--AUC (OvR, 95\\% CI)",
    }
)

# Export LaTeX table
latex_table = ham_master_latex_df.to_latex(
    index=False,
    escape=False,
    column_format="lcccc",
    caption=(
        "Performance of the seven individual models and the equal-weight "
        "ensemble on the standardized HAM10000 evaluation cohort. "
        "Values are direct point estimates with 95\\% percentile bootstrap "
        "confidence intervals based on 1,000 resamples."
    ),
    label="tab:ham10000-master-results-export",
)

# Save reproducibility artifact
LATEX_PATH = (
    "/kaggle/working/"
    "HAM10000_master_results_with_95CI_revised_final.tex"
)

with open(
    LATEX_PATH,
    "w",
    encoding="utf-8"
) as f:
    f.write(
        latex_table
    )

# Confirmation
print(
    "Saved revised HAM10000 LaTeX results table:"
)

print(
    LATEX_PATH
)

print(
    "\nPreview:\n"
)

print(
    latex_table
)

### Statistical Significance and Performance Differences — Ensemble vs Individual Models

In [ ]:
# Ensemble vs Individual Models
# Standardized HAM10000 Evaluation Cohort

# Safety checks
assert "ham_labels" in globals(), \
    "ham_labels is missing."

assert "ham_all_probs" in globals(), \
    "ham_all_probs is missing."

assert "ham_model_list" in globals(), \
    "ham_model_list is missing."

assert isinstance(ham_labels, np.ndarray), \
    "ham_labels must be a NumPy array."

assert ham_labels.shape == (2003,), \
    f"Unexpected HAM10000 label shape: {ham_labels.shape}"

evaluation_models = list(ham_model_list)

assert "Ensemble" in ham_all_probs, \
    "Ensemble probabilities are missing."


for model_name in evaluation_models + ["Ensemble"]:

    probs = np.asarray(
        ham_all_probs[model_name]
    )

    assert probs.shape == (2003, 7), \
        f"{model_name}: unexpected probability shape {probs.shape}"

    assert np.isfinite(probs).all(), \
        f"{model_name}: non-finite probabilities detected."

    assert np.allclose(
        probs.sum(axis=1),
        1.0,
        atol=1e-5
    ), \
        f"{model_name}: probability rows do not sum to 1."

# Hard predictions
ham_individual_preds = {
    model_name: ham_all_probs[
        model_name
    ].argmax(axis=1)

    for model_name in evaluation_models
}

ham_ensemble_pred = (
    ham_all_probs["Ensemble"]
    .argmax(axis=1)
)

# McNemar test
# Chi-square approximation with continuity correction

def mcnemar_continuity_corrected(
    y_true,
    model_pred,
    ensemble_pred
):
    """
    McNemar test using the chi-square approximation
    with continuity correction.

    b = Model wrong, Ensemble correct
    c = Model correct, Ensemble wrong

    Statistic:
        chi2 = (|b - c| - 1)^2 / (b + c)

    with 1 degree of freedom.

    Interpretation:
        b > c -> paired accuracy favors Ensemble
        c > b -> paired accuracy favors individual model
    """

    y_true = np.asarray(
        y_true
    )

    model_pred = np.asarray(
        model_pred
    )

    ensemble_pred = np.asarray(
        ensemble_pred
    )

    assert (
        y_true.shape
        ==
        model_pred.shape
        ==
        ensemble_pred.shape
    ), \
        "Label and prediction shapes must match."

    model_correct = (
        model_pred == y_true
    )

    ensemble_correct = (
        ensemble_pred == y_true
    )

    # Model wrong, Ensemble correct
    b = int(
        np.sum(
            (~model_correct)
            &
            ensemble_correct
        )
    )

    # Model correct, Ensemble wrong
    c = int(
        np.sum(
            model_correct
            &
            (~ensemble_correct)
        )
    )

    disagreements = (
        b + c
    )

    if disagreements == 0:

        chi2_stat = 0.0
        p_value = 1.0

    else:

        chi2_stat = (
            (abs(b - c) - 1.0) ** 2
            /
            disagreements
        )

        p_value = chi2.sf(
            chi2_stat,
            df=1
        )

    return (
        b,
        c,
        disagreements,
        float(chi2_stat),
        float(p_value),
    )

# Compute McNemar results
mcnemar_rows = []

for model_name in evaluation_models:

    (
        b,
        c,
        disagreements,
        chi2_stat,
        p_value,
    ) = mcnemar_continuity_corrected(
        ham_labels,
        ham_individual_preds[
            model_name
        ],
        ham_ensemble_pred
    )

    if p_value < 0.001:
        significance = "***"

    elif p_value < 0.01:
        significance = "**"

    elif p_value < 0.05:
        significance = "*"

    else:
        significance = "ns"

    if b > c:
        direction = "Ensemble"

    elif c > b:
        direction = model_name

    else:
        direction = "Neither"

    mcnemar_rows.append({
        "Model":
            model_name,

        "b (Model wrong, Ensemble correct)":
            b,

        "c (Model correct, Ensemble wrong)":
            c,

        "Disagreements (b+c)":
            disagreements,

        "Chi-square":
            chi2_stat,

        "p-value":
            p_value,

        "Significance":
            significance,

        "Direction":
            direction,
    })


ham_mcnemar_df = pd.DataFrame(
    mcnemar_rows
)

display(
    ham_mcnemar_df
)

# Verify against finalized authoritative results
expected_mcnemar = {

    "CNN": {
        "b": 322,
        "c": 24,
        "p": 2.176043362031653e-57,
    },

    "ResNet-50": {
        "b": 540,
        "c": 32,
        "p": 9.809797288984139e-100,
    },

    "DenseNet-121": {
        "b": 137,
        "c": 17,
        "p": 8.86904175115787e-22,
    },

    "EfficientNet-B3": {
        "b": 100,
        "c": 22,
        "p": 3.1413169866547386e-12,
    },

    "ConvNeXt-Tiny": {
        "b": 84,
        "c": 32,
        "p": 2.1880560706933413e-06,
    },

    "MobileNetV3-L": {
        "b": 175,
        "c": 15,
        "p": 8.781328499225416e-31,
    },

    "ViT": {
        "b": 47,
        "c": 88,
        "p": 5.760403386062825e-04,
    },
}


for _, row in ham_mcnemar_df.iterrows():

    model_name = row[
        "Model"
    ]

    expected = expected_mcnemar[
        model_name
    ]

    observed_b = int(
        row[
            "b (Model wrong, Ensemble correct)"
        ]
    )

    observed_c = int(
        row[
            "c (Model correct, Ensemble wrong)"
        ]
    )

    observed_p = float(
        row[
            "p-value"
        ]
    )

    assert observed_b == expected["b"], (
        f"{model_name}: McNemar b mismatch. "
        f"Expected {expected['b']}, observed {observed_b}."
    )

    assert observed_c == expected["c"], (
        f"{model_name}: McNemar c mismatch. "
        f"Expected {expected['c']}, observed {observed_c}."
    )

    assert np.isclose(
        observed_p,
        expected["p"],
        rtol=1e-10,
        atol=0.0
    ), (
        f"{model_name}: McNemar p-value mismatch. "
        f"Expected {expected['p']:.12e}, "
        f"observed {observed_p:.12e}."
    )


print(
    "\nPASS: McNemar discordant counts and "
    "continuity-corrected p-values reproduce the "
    "finalized HAM10000 analysis."
)

# Save McNemar CSV
MCNEMAR_CSV = (
    "/kaggle/working/"
    "HAM10000_mcnemar_ensemble_vs_models_revised_final.csv"
)

ham_mcnemar_df.to_csv(
    MCNEMAR_CSV,
    index=False
)

print(
    "\nSaved:"
)

print(
    MCNEMAR_CSV
)

# Export McNemar LaTeX
mcnemar_latex_df = (
    ham_mcnemar_df[
        [
            "Model",
            "b (Model wrong, Ensemble correct)",
            "c (Model correct, Ensemble wrong)",
            "p-value",
        ]
    ]
    .copy()
)

mcnemar_latex_df[
    "p-value"
] = mcnemar_latex_df[
    "p-value"
].map(
    lambda x: f"{x:.3e}"
)

mcnemar_latex_df = (
    mcnemar_latex_df.rename(
        columns={
            "Model": "Comparison",
            "b (Model wrong, Ensemble correct)": "$b$",
            "c (Model correct, Ensemble wrong)": "$c$",
            "p-value": "McNemar $p$-value",
        }
    )
)

mcnemar_latex_df[
    "Comparison"
] = (
    "Ensemble vs "
    +
    mcnemar_latex_df[
        "Comparison"
    ]
)


mcnemar_latex = (
    mcnemar_latex_df.to_latex(
        index=False,
        escape=False,
        column_format="lrrc",
        caption=(
            "Results of McNemar's test comparing the "
            "equal-weight ensemble with each individual model "
            "on the standardized HAM10000 evaluation cohort. "
            "The values $b$ and $c$ denote discordant prediction "
            "pairs, where $b$ is the number of samples correctly "
            "classified by the ensemble but misclassified by the "
            "individual model, and $c$ is the number misclassified "
            "by the ensemble but correctly classified by the "
            "individual model. McNemar's test was performed using "
            "the chi-square approximation with continuity correction."
        ),
        label="tab:ham10000-mcnemar-export"
    )
)


MCNEMAR_TEX = (
    "/kaggle/working/"
    "HAM10000_mcnemar_ensemble_vs_models_revised_final.tex"
)

with open(
    MCNEMAR_TEX,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        mcnemar_latex
    )


print(
    "Saved:"
)

print(
    MCNEMAR_TEX
)

# Performance differences
# Ensemble minus individual model

def get_ham_point_metrics(
    y_true,
    probs
):

    y_true = np.asarray(
        y_true
    )

    probs = np.asarray(
        probs
    )

    y_pred = probs.argmax(
        axis=1
    )

    return {

        "Accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "Weighted F1":
            f1_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0
            ),

        "Macro ROC-AUC (OvR)":
            roc_auc_score(
                y_true,
                probs,
                labels=np.arange(7),
                multi_class="ovr",
                average="macro"
            ),

        "Micro ROC-AUC (OvR)":
            roc_auc_score(
                y_true,
                probs,
                labels=np.arange(7),
                multi_class="ovr",
                average="micro"
            ),
    }


ensemble_metrics = (
    get_ham_point_metrics(
        ham_labels,
        ham_all_probs["Ensemble"]
    )
)


delta_rows = []

for model_name in evaluation_models:

    model_metrics = (
        get_ham_point_metrics(
            ham_labels,
            ham_all_probs[
                model_name
            ]
        )
    )

    delta_rows.append({

        "Model":
            model_name,

        "Delta Accuracy":
            ensemble_metrics[
                "Accuracy"
            ]
            -
            model_metrics[
                "Accuracy"
            ],

        "Delta Weighted F1":
            ensemble_metrics[
                "Weighted F1"
            ]
            -
            model_metrics[
                "Weighted F1"
            ],

        "Delta Macro ROC-AUC (OvR)":
            ensemble_metrics[
                "Macro ROC-AUC (OvR)"
            ]
            -
            model_metrics[
                "Macro ROC-AUC (OvR)"
            ],

        "Delta Micro ROC-AUC (OvR)":
            ensemble_metrics[
                "Micro ROC-AUC (OvR)"
            ]
            -
            model_metrics[
                "Micro ROC-AUC (OvR)"
            ],
    })


ham_delta_df = pd.DataFrame(
    delta_rows
)


print(
    "\nPerformance differences "
    "(Ensemble minus individual model):"
)

display(
    ham_delta_df.round(4)
)

# Save delta CSV
DELTA_CSV = (
    "/kaggle/working/"
    "HAM10000_point_deltas_ensemble_vs_models_revised_final.csv"
)

ham_delta_df.to_csv(
    DELTA_CSV,
    index=False
)

print(
    "Saved:"
)

print(
    DELTA_CSV
)

# Export delta LaTeX
ham_delta_latex_df = (
    ham_delta_df.copy()
)

for column in [

    "Delta Accuracy",

    "Delta Weighted F1",

    "Delta Macro ROC-AUC (OvR)",

    "Delta Micro ROC-AUC (OvR)",

]:

    ham_delta_latex_df[
        column
    ] = ham_delta_latex_df[
        column
    ].map(
        lambda x: f"{x:+.4f}"
    )


delta_latex = (
    ham_delta_latex_df.to_latex(
        index=False,
        escape=True,
        column_format="lcccc",
        caption=(
            "Point-estimate performance differences between the "
            "equal-weight ensemble and each individual model on "
            "the standardized HAM10000 evaluation cohort. "
            "Values are calculated as Ensemble minus Model; "
            "positive values favor the ensemble and negative "
            "values favor the individual model."
        ),
        label="tab:ham10000-point-deltas-export"
    )
)


DELTA_TEX = (
    "/kaggle/working/"
    "HAM10000_point_deltas_ensemble_vs_models_revised_final.tex"
)

with open(
    DELTA_TEX,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        delta_latex
    )


print(
    "Saved:"
)

print(
    DELTA_TEX
)

# Revision Analyses

The cells below produce the analyses added in the revised manuscript. They use
the fixed checkpoints and the prediction arrays saved earlier in this notebook;
nothing is retrained.

1. Checkpoint pre-flight check for the computational-cost benchmark.
2. Computational cost of each model and the ensemble on an NVIDIA Tesla T4
   (manuscript Table 6).
3. Partition of ISIC 2019 by source (manuscript Table 3).
4. Per-class F1 on the full set and on the unseen remainder (Supplementary
   Tables S4 and S5).
5. Post-hoc temperature scaling and the ensemble reliability table (manuscript
   Table 5, Supplementary Table S7).

The outputs of cells 1 to 3 are those of the run reported in the manuscript.
Benchmark latencies depend on the GPU and its load, so a re-run reproduces them
only approximately; parameter counts, MACs and checkpoint sizes are exact.
Cells 4 and 5 run on the CPU in seconds.


In [ ]:
"""
Run this ONE CELL FIRST in the Kaggle notebook, before benchmark_overhead.py.

It prints what is actually mounted and whether each checkpoint can be found,
so a wrong dataset path is caught in two seconds instead of mid-benchmark.
Nothing is loaded or modified.
"""
import os
from glob import glob

print("torch:", __import__("torch").__version__,
      "| cuda:", __import__("torch").cuda.is_available(),
      "|", __import__("torch").cuda.get_device_name(0)
      if __import__("torch").cuda.is_available() else "NO GPU - enable the T4 accelerator")

try:
    from torch.utils.flop_counter import FlopCounterMode
    print("MACs backend: torch.utils.flop_counter (built in, no install needed)")
except ImportError:
    print("MACs backend: FlopCounterMode unavailable, will try thop")

print("\nMounted datasets under /kaggle/input:")
for d in sorted(glob("/kaggle/input/*")):
    print("  ", d)

WANTED = [
    "cnn_ham10000_best_model.pth",
    "resnet50_ham10000.pth",
    "densenet121_ham10000.pth",
    "efficientnet_b3_best.pth",
    "convnext_tiny_ham10000.pth",
    "mobilenetv3_ham10000.pth",
    "vit_ham10000_best_model.pth",
]
print("\nCheckpoint discovery:")
missing = []
for w in WANTED:
    hits = glob(f"/kaggle/input/**/{w}", recursive=True)
    if hits:
        p = sorted(hits, key=len)[0]
        print(f"  OK      {w:<34} {os.path.getsize(p)/1024**2:7.1f} MB  {p}")
    else:
        print(f"  MISSING {w}")
        missing.append(w)

print("\nViT config files next to the ViT checkpoint:")
vit = glob("/kaggle/input/**/vit_ham10000_best_model.pth", recursive=True)
if vit:
    d = os.path.dirname(sorted(vit, key=len)[0])
    found = sorted(os.listdir(d))
    print("  ", d)
    print("   contents:", ", ".join(found) if found else "(empty)")
    print("   config.json present:", "config.json" in found,
          "(not required; the benchmark falls back to ViTConfig() defaults)")
else:
    print("   ViT checkpoint not found")

print("\n" + ("All checkpoints located." if not missing
              else f"{len(missing)} checkpoint(s) missing: attach those datasets, "
                   "or accept n/a in the Checkpoint (MB) column."))

In [ ]:
"""
Computational cost of each model and of the seven-model ensemble (manuscript Table 6).

Run this ON KAGGLE with the T4 accelerator enabled, in a notebook that has the seven
checkpoints attached as datasets. It measures, per model and for the full ensemble:

    trainable parameters, MACs at the model's own evaluation resolution,
    checkpoint size on disk, peak GPU memory during inference,
    and per-image latency at batch size 32.

It prints a LaTeX table body for the computational-cost table of the manuscript
and writes computational_overhead.json.

Nothing here retrains or modifies any checkpoint; it is inference-only.

Notes on robustness:

  * Checkpoint paths are auto-discovered. If a configured path does not exist the
    script searches /kaggle/input recursively for a file of the same name, so the
    exact dataset mount form (/kaggle/input/<slug>/ versus
    /kaggle/input/datasets/<user>/<slug>/) no longer matters.

  * The ViT config no longer depends on a local config.json. It tries the checkpoint's
    own directory, then the Hub id, then falls back to ViTConfig() defaults, which are
    exactly ViT-B/16 at 224 with patch 16. The fallback needs no files and no network.

  * MACs are counted with torch.utils.flop_counter.FlopCounterMode, built into
    PyTorch 2.x, so thop does not need to be installed. thop is used only if
    FlopCounterMode is unavailable.

  * A failure on one model no longer aborts the run; that row is reported as n/a and
    the remaining models still measure.
"""

import json
import os
import time
from glob import glob

import torch
import torch.nn as nn
from torchvision.models import (convnext_tiny, densenet121, efficientnet_b3,
                                mobilenet_v3_large, resnet50)

# --- MACs backend: prefer the built-in counter, fall back to thop -------------
FLOP_BACKEND = None
try:
    from torch.utils.flop_counter import FlopCounterMode
    FLOP_BACKEND = "torch"
except ImportError:
    try:
        from thop import profile as thop_profile
        FLOP_BACKEND = "thop"
    except ImportError:
        print("No MACs backend available; MACs will be reported as n/a.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH = 32
N_WARMUP = 50
N_MEASURE = 200
NUM_CLASSES = 7
KAGGLE_INPUT = "/kaggle/input"

# Configured paths. If one does not exist, the basename is searched for under
# /kaggle/input, so these act as hints rather than hard requirements.
CKPT = {
    "CNN": f"{KAGGLE_INPUT}/kaggleinputham10000-model-weights/cnn_ham10000_best_model.pth",
    "ResNet-50": f"{KAGGLE_INPUT}/resnet50-ham10000-model-weights/resnet50_ham10000.pth",
    "DenseNet-121": f"{KAGGLE_INPUT}/densenet121-ham10000-model-weights/densenet121_ham10000.pth",
    "EfficientNet-B3": f"{KAGGLE_INPUT}/efficientnetb3-ham10000-model-weights/efficientnet_b3_best.pth",
    "ConvNeXt-Tiny": f"{KAGGLE_INPUT}/ham10000-convnext-tiny-weights/convnext_tiny_ham10000.pth",
    "MobileNetV3-L": f"{KAGGLE_INPUT}/mobilenetv3-ham10000-model-weights/mobilenetv3_ham10000.pth",
    "ViT-B/16": f"{KAGGLE_INPUT}/vit-ham10000-model-weights/vit_ham10000_best_model.pth",
}

# Evaluation resolution per model - must match Supplementary Table S6.
RESOLUTION = {
    "CNN": 224, "ResNet-50": 224, "DenseNet-121": 224,
    "EfficientNet-B3": 300, "ConvNeXt-Tiny": 224,
    "MobileNetV3-L": 224, "ViT-B/16": 224,
}

_resolved = {}


def resolve_ckpt(name):
    """Return the real path of a checkpoint, searching /kaggle/input if needed."""
    if name in _resolved:
        return _resolved[name]
    hint = CKPT[name]
    path = hint if os.path.exists(hint) else None
    if path is None:
        hits = glob(f"{KAGGLE_INPUT}/**/{os.path.basename(hint)}", recursive=True)
        if hits:
            path = sorted(hits, key=len)[0]
            print(f"  [{name}] found at {path}")
        else:
            print(f"  [{name}] checkpoint not found; size reported as n/a")
    _resolved[name] = path
    return path


class SimpleCNN(nn.Module):
    """The custom 4-block baseline, as defined in the evaluation notebook."""

    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(256 * 14 * 14, 512), nn.ReLU(),
            nn.Dropout(0.5), nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def build_vit():
    """ViT-B/16 with a 7-class head, without depending on a local config.json."""
    from transformers import ViTConfig, ViTForImageClassification

    ckpt = resolve_ckpt("ViT-B/16")
    candidates = []
    if ckpt:
        candidates.append(os.path.dirname(ckpt))
    candidates.append("google/vit-base-patch16-224")

    for src in candidates:
        try:
            if src.startswith("/") and not os.path.exists(os.path.join(src, "config.json")):
                continue  # no local config: do not let transformers treat it as a Hub id
            cfg = ViTConfig.from_pretrained(src)
            cfg.num_labels = NUM_CLASSES
            print(f"  [ViT-B/16] config from {src}")
            return ViTForImageClassification(cfg)
        except Exception as exc:
            print(f"  [ViT-B/16] config source {src} unusable ({type(exc).__name__})")

    # ViTConfig() defaults are exactly ViT-B/16 at 224 with patch size 16:
    # hidden 768, 12 layers, 12 heads, intermediate 3072. No files, no network.
    print("  [ViT-B/16] config from ViTConfig() defaults (ViT-B/16 224, patch 16)")
    cfg = ViTConfig(num_labels=NUM_CLASSES)
    return ViTForImageClassification(cfg)


def build(name):
    """Return an architecture with a 7-class head (weights loaded separately)."""
    if name == "CNN":
        return SimpleCNN()
    if name == "ResNet-50":
        m = resnet50(weights=None)
        m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
        return m
    if name == "DenseNet-121":
        m = densenet121(weights=None)
        m.classifier = nn.Linear(m.classifier.in_features, NUM_CLASSES)
        return m
    if name == "EfficientNet-B3":
        m = efficientnet_b3(weights=None)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, NUM_CLASSES)
        return m
    if name == "ConvNeXt-Tiny":
        m = convnext_tiny(weights=None)
        m.classifier[2] = nn.Linear(m.classifier[2].in_features, NUM_CLASSES)
        return m
    if name == "MobileNetV3-L":
        m = mobilenet_v3_large(weights=None)
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, NUM_CLASSES)
        return m
    if name == "ViT-B/16":
        return build_vit()
    raise ValueError(name)


class LogitsOnly(nn.Module):
    """Unwrap HuggingFace outputs so the counters and timing loop see a plain tensor."""

    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, x):
        out = self.m(x)
        return out.logits if hasattr(out, "logits") else out


def count_macs(model, res):
    """MACs in G at the given square resolution, or nan if no backend works."""
    x = torch.randn(1, 3, res, res, device=DEVICE)
    try:
        if FLOP_BACKEND == "torch":
            counter = FlopCounterMode(display=False)
            with counter, torch.no_grad():
                model(x)
            # FlopCounterMode reports 2 flops per multiply-accumulate.
            return counter.get_total_flops() / 2 / 1e9
        if FLOP_BACKEND == "thop":
            with torch.no_grad():
                return thop_profile(model, inputs=(x,), verbose=False)[0] / 1e9
    except Exception as exc:
        print(f"  MACs counting failed: {type(exc).__name__}: {exc}")
    return float("nan")


def measure(name):
    res = RESOLUTION[name]
    nan = float("nan")
    try:
        model = build(name)
    except Exception as exc:
        print(f"  [{name}] BUILD FAILED: {type(exc).__name__}: {exc}")
        return dict(model=name, params_M=nan, input=res, macs_G=nan,
                    ckpt_MB=nan, peak_mem_MB=nan, ms_per_image=nan)

    ckpt_mb = nan
    path = resolve_ckpt(name)
    if path:
        ckpt_mb = os.path.getsize(path) / 1024 ** 2
        try:
            sd = torch.load(path, map_location="cpu", weights_only=False)
            if isinstance(sd, dict) and "state_dict" in sd:
                sd = sd["state_dict"]
            model.load_state_dict(sd)
        except Exception as exc:
            print(f"  [{name}] weights not loaded ({type(exc).__name__}); cost metrics "
                  f"are architecture-determined and remain valid")

    model = LogitsOnly(model).to(DEVICE).eval()
    params = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
    macs = count_macs(model, res)

    x = torch.randn(BATCH, 3, res, res, device=DEVICE)
    with torch.no_grad():
        for _ in range(N_WARMUP):
            model(x)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
        t0 = time.perf_counter()
        for _ in range(N_MEASURE):
            model(x)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - t0

    peak_mb = (torch.cuda.max_memory_allocated() / 1024 ** 2
               if DEVICE.type == "cuda" else nan)
    ms_per_image = elapsed / (N_MEASURE * BATCH) * 1000

    del model, x
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return dict(model=name, params_M=params, input=res, macs_G=macs,
                ckpt_MB=ckpt_mb, peak_mem_MB=peak_mb, ms_per_image=ms_per_image)


def fmt(v, nd=2):
    return "n/a" if v != v else f"{v:.{nd}f}"


def nansum(vals):
    good = [v for v in vals if v == v]
    return sum(good) if good else float("nan")


def main():
    print(f"device: {DEVICE}"
          f"{' - ' + torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else ''}")
    print(f"MACs backend: {FLOP_BACKEND or 'none'}")
    print(f"batch {BATCH}, {N_WARMUP} warm-up + {N_MEASURE} measured iterations\n")

    rows = []
    for name in CKPT:
        print(f"measuring {name} ...")
        rows.append(measure(name))

    total = dict(model="Ensemble (all 7)",
                 params_M=nansum([r["params_M"] for r in rows]),
                 input="-",
                 macs_G=nansum([r["macs_G"] for r in rows]),
                 ckpt_MB=nansum([r["ckpt_MB"] for r in rows]),
                 peak_mem_MB=max([r["peak_mem_MB"] for r in rows if r["peak_mem_MB"] == r["peak_mem_MB"]] or [float("nan")]),
                 ms_per_image=nansum([r["ms_per_image"] for r in rows]))

    with open("computational_overhead.json", "w") as fh:
        json.dump(rows + [total], fh, indent=2)

    print("\n" + "=" * 78)
    print("LaTeX table body - paste over the red [measure] cells in the manuscript")
    print("=" * 78)
    for r in rows:
        print(f"{r['model']:<16} & {fmt(r['params_M'], 3):>7} & ${r['input']}$ & "
              f"{fmt(r['macs_G']):>6} & {fmt(r['ckpt_MB'], 0):>5} & "
              f"{fmt(r['ms_per_image']):>6} \\\\")
    print("\\midrule")
    print(f"{'Ensemble (all 7)':<16} & {fmt(total['params_M'], 3):>7} & - & "
          f"{fmt(total['macs_G']):>6} & {fmt(total['ckpt_MB'], 0):>5} & "
          f"{fmt(total['ms_per_image']):>6} \\\\")
    print("=" * 78)
    print(f"\nPeak GPU memory, largest single model: {fmt(total['peak_mem_MB'], 0)} MB "
          f"at batch {BATCH}.")
    print("Components run sequentially, so ensemble peak memory equals the largest single")
    print("component; quote that in the caption rather than a sum.")
    print("Saved: computational_overhead.json")


if __name__ == "__main__":
    main()

In [ ]:
"""
Partition of the ISIC 2019 evaluation set by source (manuscript Table 3,
Supplementary Table S5).

The ISIC 2019 challenge training set is assembled from HAM10000, BCN_20000 and MSK.
Every HAM10000 image therefore appears inside the 25,331-image ISIC 2019 set that this
study used for external evaluation, so about 40% of that "external" set is the dataset
the seven checkpoints were trained on.

This script partitions ISIC 2019 by source and recomputes every external metric on the
non-HAM10000 remainder, which is genuinely unseen by every model. It needs no retraining
and no new data: it reuses the released per-model probability arrays.

HOW TO RUN

  In a Kaggle or Jupyter notebook, just run the cell. Inputs are discovered automatically
  under /kaggle/input and the working directory; no arguments are needed. (An earlier
  version read sys.argv, which breaks in a notebook because the kernel passes its own
  "-f <connection file>" flag.)

  From a terminal, optionally pass paths:
      python external_decontamination.py ground_truth.csv predictions.npz

INPUTS   ISIC_2019_Training_GroundTruth.csv
         isic2019_external_eval_predictions.npz, or the individual
         isic2019_<model>_probs.npy files plus isic2019_labels.npy
OUTPUTS  ISIC2019_decontaminated_metrics.csv, ISIC2019_contamination_summary.csv,
         and a LaTeX table body for the manuscript
"""

import os
import sys
from glob import glob

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

CLASSES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
ONEHOT = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]
MODELS = ["cnn", "resnet50", "densenet121", "efficientnet_b3",
          "convnext_tiny", "mobilenet_v3", "vit"]
LABEL = {"cnn": "CNN", "resnet50": "ResNet-50", "densenet121": "DenseNet-121",
         "efficientnet_b3": "EfficientNet-B3", "convnext_tiny": "ConvNeXt-Tiny",
         "mobilenet_v3": "MobileNetV3-L", "vit": "ViT"}

# HAM10000 occupies a contiguous ISIC id block. The bounds come from the released
# HAM10000 split files; the resulting count is checked against the documented 10,015.
HAM_LO, HAM_HI = 24307, 34317

SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working", ".", ".."]


def find(pattern, needed=True):
    """Locate a file by glob pattern under the usual roots. Shortest path wins."""
    hits = []
    for root in SEARCH_ROOTS:
        if os.path.isdir(root):
            hits += glob(os.path.join(root, "**", pattern), recursive=True)
    hits = sorted(set(hits), key=len)
    if hits:
        return hits[0]
    if needed:
        raise FileNotFoundError(
            f"Could not find '{pattern}' under {', '.join(SEARCH_ROOTS)}. "
            f"Attach the dataset containing it, or pass the path explicitly.")
    return None


def cli_arg(suffix):
    """Return a command-line path with this suffix, ignoring notebook kernel flags."""
    for a in sys.argv[1:]:
        if a.lower().endswith(suffix) and os.path.exists(a):
            return a
    return None


def load_predictions():
    """Return (labels, {model: probs}) from the npz, or from individual npy files."""
    npz = cli_arg(".npz") or find("isic2019_external_eval_predictions.npz", needed=False)
    if npz:
        z = np.load(npz)
        missing = [m for m in MODELS if f"{m}_probs" not in z]
        if not missing:
            print(f"predictions: {npz}")
            return z["labels"], {m: z[f"{m}_probs"] for m in MODELS}
        print(f"  {npz} is missing {missing}; falling back to individual .npy files")

    lab = find("isic2019_labels.npy")
    probs = {}
    for m in MODELS:
        probs[m] = np.load(find(f"isic2019_{m}_probs.npy"))
    print(f"predictions: individual .npy files next to {lab}")
    return np.load(lab), probs


def main():
    gt_path = cli_arg(".csv") or find("ISIC_2019_Training_GroundTruth.csv")
    print(f"ground truth: {gt_path}")

    gt = pd.read_csv(gt_path)
    gt["diagnosis"] = (gt[ONEHOT].idxmax(axis=1).str.lower()
                       .replace({"ak": "akiec", "scc": "akiec"}))
    gt["label"] = gt["diagnosis"].map({c: i for i, c in enumerate(CLASSES)})
    gt["num"] = gt["image"].str.extract(r"ISIC_(\d+)").astype(int)
    is_ham = ((gt["num"] >= HAM_LO) & (gt["num"] <= HAM_HI)).values

    y, P = load_predictions()

    if len(y) != len(gt):
        raise SystemExit(
            f"Length mismatch: {len(y)} predictions vs {len(gt)} ground-truth rows. "
            f"These must come from the same evaluation run.")
    agree = (gt["label"].values == y).mean()
    if agree < 1.0:
        raise SystemExit(
            f"Prediction row order does not match the ground-truth file "
            f"(label agreement {agree:.6f}). Re-export the predictions in CSV order.")
    print(f"row order verified against the ground truth (agreement {agree:.6f})")

    P["ensemble"] = np.mean([P[m] for m in MODELS], axis=0)

    print(f"\nISIC 2019 total            : {len(y):>6}")
    print(f"  HAM10000 source (seen)   : {is_ham.sum():>6}  ({100*is_ham.mean():.1f}%)")
    print(f"  non-HAM10000 (unseen)    : {(~is_ham).sum():>6}  ({100*(~is_ham).mean():.1f}%)")

    rows = []
    for key in MODELS + ["ensemble"]:
        p = P[key]
        r = {"Model": LABEL.get(key, "Ensemble")}
        for tag, mask in [("full", np.ones(len(y), bool)),
                          ("ham", is_ham), ("clean", ~is_ham)]:
            yy, pp = y[mask], p[mask]
            pred = pp.argmax(1)
            present = np.unique(yy)
            r[f"acc_{tag}"] = accuracy_score(yy, pred)
            r[f"f1_{tag}"] = f1_score(yy, pred, average="weighted", zero_division=0)
            # Restrict to classes present in the subset so the OvR AUC stays defined,
            # and renormalise so the retained columns sum to one.
            sub = pp[:, present]
            r[f"auc_{tag}"] = roc_auc_score(
                yy, sub / sub.sum(1, keepdims=True),
                multi_class="ovr", average="macro", labels=present)
            r[f"wauc_{tag}"] = roc_auc_score(
                yy, sub / sub.sum(1, keepdims=True),
                multi_class="ovr", average="weighted", labels=present)
        r["acc_drop"] = r["acc_full"] - r["acc_clean"]
        rows.append(r)

    df = pd.DataFrame(rows)
    df.to_csv("ISIC2019_decontaminated_metrics.csv", index=False)

    pd.DataFrame({
        "Subset": ["ISIC 2019 as evaluated", "HAM10000 source images",
                   "non-HAM10000 remainder"],
        "N": [len(y), int(is_ham.sum()), int((~is_ham).sum())],
        "Share": ["100.0%", f"{100*is_ham.mean():.1f}%", f"{100*(~is_ham).mean():.1f}%"],
        "Seen during training": ["partly", "yes", "no"],
    }).to_csv("ISIC2019_contamination_summary.csv", index=False)

    print(f"\n{'Model':<16}{'acc full':>9}{'acc seen':>9}{'acc unseen':>11}{'drop':>8}"
          f"{'F1 unseen':>10}{'AUC unseen':>11}")
    for _, r in df.iterrows():
        print(f"{r['Model']:<16}{r['acc_full']:>9.4f}{r['acc_ham']:>9.4f}"
              f"{r['acc_clean']:>11.4f}{r['acc_drop']:>+8.4f}"
              f"{r['f1_clean']:>10.4f}{r['auc_clean']:>11.4f}")

    print("\n" + "=" * 78)
    print("LaTeX body for the partitioned external table")
    print("Model & Acc full & Acc seen & Acc unseen & wF1 unseen & macroAUC & wAUC")
    print("=" * 78)
    for _, r in df.iterrows():
        print(f"{r['Model']:<16} & {r['acc_full']:.4f} & {r['acc_ham']:.4f} & "
              f"{r['acc_clean']:.4f} & {r['f1_clean']:.4f} & "
              f"{r['auc_clean']:.4f} & {r['wauc_clean']:.4f} \\\\")
    print("=" * 78)
    print("Saved: ISIC2019_decontaminated_metrics.csv, ISIC2019_contamination_summary.csv")


main()


## Per-Class F1 on the Unseen ISIC 2019 Remainder

Expected: full-set F1 is higher than unseen-set F1 in 54 of the 56 model-class
cells; on the unseen images ConvNeXt-Tiny leads for AKIEC, BKL, DF and NV,
DenseNet-121 for BCC and MEL (0.3777 against 0.3757), and the ensemble for VASC.


In [ ]:
"""
Per-class F1 on the full ISIC 2019 set and on the genuinely unseen remainder
(Supplementary Tables S4 and S5).

The unseen remainder is the 15,320 BCN_20000 and MSK images, after removing the
10,011 HAM10000 images (identifier block ISIC_0024307 to ISIC_0034317) that every
checkpoint saw during training. Inputs are located automatically under
/kaggle/working, /kaggle/input and the current folder.
"""

import os
from glob import glob

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

SEARCH_ROOTS = ["/kaggle/working", "/kaggle/input", "."]
CLASSES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
MODELS = ["cnn", "resnet50", "densenet121", "efficientnet_b3", "convnext_tiny", "mobilenet_v3", "vit"]
LABEL = dict(zip(MODELS, ["CNN", "ResNet-50", "DenseNet-121", "EfficientNet-B3",
                          "ConvNeXt-Tiny", "MobileNetV3-L", "ViT"]))
HAM_LO, HAM_HI = 24307, 34317


def find(name):
    hits = []
    for root in SEARCH_ROOTS:
        if os.path.isdir(root):
            hits += glob(os.path.join(root, "**", name), recursive=True)
    if not hits:
        raise FileNotFoundError(f"{name} not found under {SEARCH_ROOTS}")
    return sorted(set(hits), key=len)[0]


gt = pd.read_csv(find("ISIC_2019_Training_GroundTruth.csv"))
z = np.load(find("isic2019_external_eval_predictions.npz"))
y = z["labels"]
P = {m: z[f"{m}_probs"] for m in MODELS}
P["ensemble"] = np.mean([P[m] for m in MODELS], axis=0)

num = gt["image"].str.extract(r"ISIC_(\d+)")[0].astype(int).values
is_ham = (num >= HAM_LO) & (num <= HAM_HI)
assert len(y) == len(gt) and is_ham.sum() == 10011 and (~is_ham).sum() == 15320

tables = {}
for title, mask in [("S4: full ISIC 2019 set (n = 25,331)", np.ones(len(y), bool)),
                    ("S5: unseen remainder (n = 15,320)", ~is_ham)]:
    rows = {LABEL.get(k, "Ensemble"): f1_score(y[mask], p[mask].argmax(1), average=None,
                                              labels=range(7), zero_division=0)
            for k, p in P.items()}
    df = pd.DataFrame(rows, index=[c.upper() for c in CLASSES]).T
    tables[title] = df
    print(f"\nPer-class F1, {title}")
    print(df.round(2).to_string())
    print("Highest per class:", ", ".join(f"{c}: {df[c].idxmax()} ({df[c].max():.4f})" for c in df.columns))

full, unseen = tables.values()
higher = (full.values > unseen.values).sum()
print(f"\nFull-set F1 higher than unseen F1 in {higher} of {full.size} model-class cells")
unseen.round(4).to_csv("ISIC2019_unseen_per_class_F1.csv")
print("Saved: ISIC2019_unseen_per_class_F1.csv")


## Post-hoc Temperature Scaling

Expected: ensemble ECE 11.54% before and 1.26% after scaling with T = 0.482,
accuracy unchanged at 0.9386; per-model scaling followed by averaging gives
13.17%. Eleven of the fifteen confidence bins are populated, and observed
accuracy exceeds mean confidence in all eleven.


In [ ]:
"""
Post-hoc temperature scaling of the equal-weight ensemble and of each model
(manuscript Table 5 and Supplementary Table S7).

Inputs are the canonical prediction arrays written earlier in this notebook
(canonical_ham10000_labels.npy, canonical_<model>_probs.npy,
canonical_ensemble_probs.npy) and isic2019_external_eval_predictions.npz.
They are located automatically under /kaggle/working, /kaggle/input and the
current folder. Nothing is retrained and no GPU is needed.

Temperature scaling divides recovered logits L = log(p) by one scalar T fitted
by minimizing multiclass negative log-likelihood. T is estimated out of fold
(5-fold stratified, seed 42): fitted on four folds and applied to the fifth.
A positive temperature cannot change the predicted class, so accuracy and
weighted F1 are unchanged. ECE uses 15 equal-width right-closed bins (lo, hi].
"""

import os
from glob import glob

import numpy as np
from scipy.optimize import minimize_scalar
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold

SEARCH_ROOTS = ["/kaggle/working", "/kaggle/input", "."]
NAMES = ["cnn", "resnet50", "densenet121", "efficientnet_b3", "convnext_tiny", "mobilenet_v3", "vit"]
LABEL = dict(zip(NAMES, ["CNN", "ResNet-50", "DenseNet-121", "EfficientNet-B3",
                         "ConvNeXt-Tiny", "MobileNetV3-L", "ViT"]))
NBINS, EPS = 15, 1e-12


def find(name):
    hits = []
    for root in SEARCH_ROOTS:
        if os.path.isdir(root):
            hits += glob(os.path.join(root, "**", name), recursive=True)
    if not hits:
        raise FileNotFoundError(f"{name} not found under {SEARCH_ROOTS}")
    return sorted(set(hits), key=len)[0]


def ece(p, y, n_bins=NBINS):
    conf, correct = p.max(1), (p.argmax(1) == y).astype(float)
    edges, total = np.linspace(0, 1, n_bins + 1), 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            total += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return total


def softmax(z):
    e = np.exp(z - z.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)


def fit_T(p, y):
    L = np.log(np.clip(p, EPS, None))

    def nll(logT):
        q = softmax(L / np.exp(logT))
        return -np.log(np.clip(q[np.arange(len(y)), y], EPS, None)).mean()

    r = minimize_scalar(nll, bounds=(np.log(0.05), np.log(20.0)), method="bounded")
    return float(np.exp(r.x))


def crossfit(p, y, seed=42):
    out, Ts = np.zeros_like(p), []
    for tr, te in StratifiedKFold(5, shuffle=True, random_state=seed).split(p, y):
        T = fit_T(p[tr], y[tr])
        Ts.append(T)
        out[te] = softmax(np.log(np.clip(p[te], EPS, None)) / T)
    return out, float(np.mean(Ts))


y = np.load(find("canonical_ham10000_labels.npy"))
P = {n: np.load(find(f"canonical_{n}_probs.npy")) for n in NAMES}
ens = np.load(find("canonical_ensemble_probs.npy"))
assert np.allclose(np.mean([P[n] for n in NAMES], axis=0), ens, atol=1e-6)

print(f"HAM10000 cohort n={len(y)}")
print(f"{'Model':<17}{'acc':>8}{'conf':>8}{'dir':>7}{'ECE':>8}{'T':>8}{'ECE TS':>9}")
for n, p in list(P.items()) + [("ensemble", ens)]:
    acc, conf = accuracy_score(y, p.argmax(1)), p.max(1).mean()
    cal, T = crossfit(p, y)
    print(f"{LABEL.get(n, 'Ensemble'):<17}{acc:>8.4f}{conf:>8.4f}"
          f"{'under' if acc > conf else 'over':>7}{100 * ece(p, y):>7.2f}%{T:>8.3f}{100 * ece(cal, y):>8.2f}%")

cal, T = crossfit(ens, y)
print(f"\nEnsemble after TS: acc {accuracy_score(y, cal.argmax(1)):.4f}  "
      f"wF1 {f1_score(y, cal.argmax(1), average='weighted'):.4f}  "
      f"macro AUC {roc_auc_score(y, cal, multi_class='ovr'):.4f}")
pm = np.mean([crossfit(P[n], y)[0] for n in NAMES], axis=0)
print(f"Per-model TS, then average: ECE {100 * ece(pm, y):.2f}%")

print("\nReliability table, ensemble (populated bins only)")
conf, correct = ens.max(1), (ens.argmax(1) == y).astype(float)
edges = np.linspace(0, 1, NBINS + 1)
for lo, hi in zip(edges[:-1], edges[1:]):
    m = (conf > lo) & (conf <= hi)
    if m.any():
        print(f"  ({lo:.2f}, {hi:.2f}]  n={m.sum():4d}  conf {conf[m].mean():.4f}  "
              f"acc {correct[m].mean():.4f}  gap {correct[m].mean() - conf[m].mean():+.4f}")

z = np.load(find("isic2019_external_eval_predictions.npz"))
yi = z["labels"]
Pi = {n: z[f"{n}_probs"] for n in NAMES}
Pi["ensemble"] = np.mean([Pi[n] for n in NAMES], axis=0)
print(f"\nISIC 2019 n={len(yi)}")
for n, p in Pi.items():
    cal, T = crossfit(p, yi)
    print(f"{LABEL.get(n, 'Ensemble'):<17}ECE {100 * ece(p, yi):6.2f}%  T {T:6.3f}  ECE TS {100 * ece(cal, yi):6.2f}%")
